# Weather-Aware Ship Routing Algorithm
**Supply Chain and Coastal Chain Intelligence Platform — module `ml/ship_routing`**

## 1. Project introduction and objectives

This notebook builds the **routing engine** used by the *Shipping Company* module. Given a source port, a destination port, vessel details and a departure time, it finds a sea route that balances **distance** against **weather risk**.

**What the notebook does**

| Step | What happens |
|---|---|
| Maritime graph | Ports and offshore waypoints are **nodes**; navigable sea lanes are **edges** with distance, heading and an (illustrative) depth. |
| Weather | For points along every lane it downloads **wave height/direction/period, swell, wind speed/direction and gusts** from the free **Open-Meteo** APIs (no API key). |
| Risk scoring | Every lane gets a risk level: **Low, Moderate, High or Severe**, using configurable thresholds and the wave direction relative to the ship. |
| Routing | **Dijkstra's algorithm** finds (a) the shortest-distance route and (b) the weather-aware route with cost = distance + weather penalty. Lanes that break vessel limits are removed. A* is also available. |
| Output | A comparison table, maps, and a structured **JSON** response that a FastAPI backend can return later. |

> **This is not a machine-learning model.** Route finding is a graph optimisation problem, so no model is trained.

### Real data vs. synthetic data (please read)
| Item | Real or synthetic? |
|---|---|
| Weather in Sections 9, 10, 14, 15 | **Real forecast** from Open-Meteo (when the API is reachable). Weather data by [Open-Meteo.com](https://open-meteo.com) (CC-BY 4.0). |
| Weather in the test scenarios (Section 16) | **Synthetic**, reproducible test weather. Every record is tagged `synthetic:<scenario>`. |
| Waypoint network (Section 6) | **Synthetic demonstration network.** Hand-placed waypoints, illustrative depths. **Not a navigational chart.** |
| Risk thresholds / vessel limits | **Prototype values**, not certified maritime safety limits. They must be tuned with vessel data and professional maritime guidance. |

Open-Meteo itself states that its marine data has limited accuracy near coasts and is **not suitable for coastal navigation**.

**Run the cells in order, top to bottom** (Runtime → Run all works too).

## 2. Install required libraries
Colab already includes NumPy, Pandas, Matplotlib, Requests and NetworkX. This cell makes sure everything (including Folium for interactive maps) is present.

In [ ]:
%pip install -q "networkx>=3.0" "folium>=0.14" "requests>=2.31" "pandas>=2.0" "matplotlib>=3.7"

## 3. Mount Google Drive
Everything important is saved to Google Drive so nothing is lost when the Colab runtime disconnects. A pop-up will ask you to allow access.

If you run this notebook outside Colab, it automatically uses a local folder instead.

In [ ]:
import os, sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab - a local folder will be used instead of Google Drive.")

print("Running in Colab:", IN_COLAB)

## 4. Set up project directories
The module lives in its **own folder** `ml/ship_routing/`. The notebook never reads, writes or deletes anything in `ml/port_congestion/`, so your congestion model is safe.

```
coastal_platform/ml/ship_routing/
    notebooks/   <- this notebook (copy saved in Section 19)
    data/        <- waypoint network CSVs + configuration JSON
    results/     <- json/, maps/, figures/, tables/, cache/
    routing.py, weather_service.py, visualization.py, sample_network.py
    requirements.txt, README.md, fastapi_app_example.py
```

In [ ]:
if IN_COLAB:
    PLATFORM_DIR = "/content/drive/MyDrive/coastal_platform"
else:
    PLATFORM_DIR = os.path.abspath("coastal_platform")

PROJECT_DIR = os.path.join(PLATFORM_DIR, "ml", "ship_routing")
EXPORT_DIR = os.path.join(PLATFORM_DIR, "exports")          # ZIP packages go here

DIRS = {
    "project":   PROJECT_DIR,
    "notebooks": os.path.join(PROJECT_DIR, "notebooks"),
    "data":      os.path.join(PROJECT_DIR, "data"),
    "results":   os.path.join(PROJECT_DIR, "results"),
    "json":      os.path.join(PROJECT_DIR, "results", "json"),
    "maps":      os.path.join(PROJECT_DIR, "results", "maps"),
    "figures":   os.path.join(PROJECT_DIR, "results", "figures"),
    "tables":    os.path.join(PROJECT_DIR, "results", "tables"),
    "cache":     os.path.join(PROJECT_DIR, "results", "cache"),
    "exports":   EXPORT_DIR,
}
for path in DIRS.values():
    os.makedirs(path, exist_ok=True)      # exist_ok=True -> reuse, never wipe

# Let Python import the module files we save in Section 18
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

congestion_dir = os.path.join(PLATFORM_DIR, "ml", "port_congestion")
print("Project directory:", PROJECT_DIR)
for name, path in DIRS.items():
    print(f"  {name:<10} {path}")
print("\nport_congestion folder present:", os.path.isdir(congestion_dir), "(left untouched)")

## 5. Import libraries and central configuration
All imports are in one place. Then we define the **configuration**: risk thresholds, penalty weights and vessel profiles. Change values here (or pass your own `RoutingConfig(...)`) to experiment.

**How a lane's cost is calculated**

`edge cost = distance_weight × distance_km + weather_weight × weather_penalty`

`weather_penalty = distance_km × penalty_multiplier[risk level]`  with default multipliers Low 0, Moderate 0.3, High 1.5, Severe 4.0.

A lane is **removed** from the search (unavailable) when:
* the significant wave height or wind is above the **vessel's limit**, or
* the risk is **Severe** and `block_severe_segments=True`, or
* the vessel's draft + under-keel clearance is deeper than the lane's depth.

⚠️ All numbers below are **prototype values**, not certified safety limits.

In [ ]:
import copy, heapq, json, logging, math, shutil, time, zipfile
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime, timedelta, timezone
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import networkx as nx
import requests
import matplotlib.pyplot as plt
from IPython.display import display, HTML

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
logger = logging.getLogger("ship_routing")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("networkx", nx.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Configuration dataclasses and vessel presets

# ---------------------------------------------------------------------------
# Configuration: risk thresholds, penalties, weights and vessel profiles
# (ALL VALUES ARE PROTOTYPE VALUES - NOT CERTIFIED SAFETY LIMITS)
# ---------------------------------------------------------------------------

RISK_LABELS = ["Low", "Moderate", "High", "Severe"]   # index = risk level 0..3


@dataclass
class RoutingConfig:
    # Thresholds are the lower bounds of Moderate, High and Severe.
    # Wave height (m): Low < 2.0 <= Moderate < 3.5 <= High < 5.0 <= Severe
    wave_thresholds_m: Tuple[float, float, float] = (2.0, 3.5, 5.0)
    # Sustained wind (knots), roughly Beaufort 5 / 7 / 9 boundaries
    wind_thresholds_kn: Tuple[float, float, float] = (17.0, 28.0, 41.0)
    # Gusts (knots)
    gust_thresholds_kn: Tuple[float, float, float] = (25.0, 40.0, 55.0)

    # Wave direction relative to the ship's heading (0 deg = waves from dead
    # ahead, 90 = on the beam, 180 = from astern). The wave height is
    # multiplied by this factor BEFORE classification (prototype heuristic:
    # beam seas cause heavy rolling, head seas cause slamming).
    head_sea_max_angle: float = 45.0
    following_sea_min_angle: float = 135.0
    head_sea_factor: float = 1.10
    beam_sea_factor: float = 1.20
    following_sea_factor: float = 1.00

    # Edge cost = distance_weight * km + weather_weight * weather_penalty
    # weather_penalty = km * penalty_multiplier[risk level]
    distance_weight: float = 1.0
    weather_weight: float = 1.0
    penalty_multipliers: Tuple[float, float, float, float] = (0.0, 0.3, 1.5, 4.0)

    # Hard constraints
    block_severe_segments: bool = True     # Severe segments become unavailable
    under_keel_clearance_m: float = 1.5    # draft + clearance must be <= depth

    # Missing weather data: "conservative" (assume at least Moderate + extra
    # penalty), "block" (segment unavailable) or "ignore" (use what exists)
    missing_data_policy: str = "conservative"
    missing_data_min_level: int = 1
    missing_data_penalty_multiplier: float = 0.2

    # Speed loss in bad weather (fraction of service speed) per risk level
    speed_loss_fraction: Tuple[float, float, float, float] = (0.0, 0.05, 0.15, 0.30)

    # Weather sampling along each segment
    sample_spacing_km: float = 100.0
    # Weather planning window (hours from departure). None = automatic
    weather_window_hours: Optional[int] = None
    max_weather_window_hours: int = 168

    algorithm: str = "dijkstra"            # "dijkstra" or "astar"

    def to_dict(self) -> dict:
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(self).items()}

    @classmethod
    def from_dict(cls, d: dict) -> "RoutingConfig":
        fields_ = {f for f in cls.__dataclass_fields__}
        clean = {k: (tuple(v) if isinstance(v, list) else v) for k, v in d.items() if k in fields_}
        return cls(**clean)


@dataclass
class VesselProfile:
    vessel_type: str
    service_speed_kn: float
    draft_m: float
    max_wave_height_m: float        # hard limit on significant wave height
    max_wind_speed_kn: float        # hard limit on sustained wind
    wave_thresholds_m: Optional[Tuple[float, float, float]] = None  # overrides config
    notes: str = "prototype values - replace with real vessel data"


# Illustrative presets only. Replace with data from the Shipping Company module.
VESSEL_PRESETS: Dict[str, VesselProfile] = {
    "container_ship": VesselProfile("container_ship", 16.0, 12.5, 6.0, 45.0),
    "bulk_carrier":   VesselProfile("bulk_carrier",   13.0, 11.5, 5.5, 40.0),
    "tanker":         VesselProfile("tanker",         14.0, 13.0, 6.0, 45.0),
    "general_cargo":  VesselProfile("general_cargo",  12.0,  8.0, 4.5, 38.0),
    "small_coastal_vessel": VesselProfile("small_coastal_vessel", 10.0, 2.5, 3.5, 34.0,
                                          wave_thresholds_m=(1.25, 2.0, 3.0)),
}


def get_vessel_profile(vessel_type: str = "container_ship",
                       vessel_speed: Optional[float] = None,
                       vessel_specs: Optional[dict] = None) -> VesselProfile:
    """Start from a preset and apply speed/spec overrides from the request."""
    key = (vessel_type or "container_ship").strip().lower()
    if key not in VESSEL_PRESETS:
        raise ValueError(f"Unknown vessel_type '{vessel_type}'. "
                         f"Known types: {sorted(VESSEL_PRESETS)}")
    vessel = replace(VESSEL_PRESETS[key])
    if vessel_speed is not None:
        vessel.service_speed_kn = float(vessel_speed)
    allowed = {"draft_m", "max_wave_height_m", "max_wind_speed_kn"}
    for k, v in (vessel_specs or {}).items():
        if k in allowed and v is not None:
            setattr(vessel, k, float(v))
        elif k == "wave_thresholds_m" and v:
            vessel.wave_thresholds_m = tuple(float(x) for x in v)
    if not (0 < vessel.service_speed_kn <= 40):
        raise ValueError("vessel_speed must be between 0 and 40 knots")
    if vessel.draft_m <= 0:
        raise ValueError("draft_m must be positive")
    return vessel

In [ ]:
ROUTING_CONFIG = RoutingConfig()          # default prototype settings
print("Risk thresholds (Moderate / High / Severe lower bounds):")
print("  wave height (m):", ROUTING_CONFIG.wave_thresholds_m)
print("  wind (kn):      ", ROUTING_CONFIG.wind_thresholds_kn)
print("  gusts (kn):     ", ROUTING_CONFIG.gust_thresholds_kn)
print("Penalty multipliers:", dict(zip(RISK_LABELS, ROUTING_CONFIG.penalty_multipliers)))
display(pd.DataFrame([asdict(v) for v in VESSEL_PRESETS.values()]).set_index("vessel_type"))

## 6. Load or create maritime waypoint data
We use a **small synthetic network** around India and Sri Lanka because a free, verified maritime lane graph is not readily available for this project.

* Ports use UN/LOCODE-style IDs (e.g. `INNSA` = Nhava Sheva/JNPT, `INMAA` = Chennai).
* Offshore waypoints were placed by hand in open water so that lanes do not cross land (including the Lakshadweep islands). They are **not** official traffic lanes.
* The route from the west coast to the east coast goes **around Sri Lanka**, because the Pamban / Palk Strait passage is too shallow for large ships. That passage is included with a very shallow illustrative depth, so only a small vessel can use it.

**To use real data later:** put your own `waypoints.csv` and `edges.csv` (same column names) in the `data/` folder and set `USE_CUSTOM_NETWORK = True`.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Synthetic demonstration network (NOT a navigational chart)

DATA_NOTICE = ("SYNTHETIC DEMONSTRATION NETWORK - hand-placed waypoints and "
               "illustrative depths. Not a navigational chart and not verified "
               "real-world route data. Do not use for actual navigation.")

# node_id, name, type, lat, lon, country, notes
_WAYPOINTS = [
    # --- Ports (approximate locations; IDs follow UN/LOCODE style) ---------
    ("INNSA", "Mumbai (JNPT / Nhava Sheva)", "port", 18.95, 72.95, "IN", "port"),
    ("INMRM", "Mormugao (Goa)",              "port", 15.41, 73.80, "IN", "port"),
    ("INNML", "New Mangalore",               "port", 12.92, 74.80, "IN", "port"),
    ("INCOK", "Kochi (Cochin)",              "port",  9.97, 76.24, "IN", "port"),
    ("INTUT", "Tuticorin (V.O. Chidambaranar)", "port", 8.76, 78.20, "IN", "port"),
    ("INMAA", "Chennai",                     "port", 13.10, 80.30, "IN", "port"),
    ("INVTZ", "Visakhapatnam",               "port", 17.69, 83.30, "IN", "port"),
    ("LKCMB", "Colombo",                     "port",  6.95, 79.84, "LK", "port"),
    # --- Offshore approach waypoints (synthetic) ---------------------------
    ("WP_MUM_OFF", "Off Mumbai",        "waypoint", 18.90, 72.40, "", "synthetic"),
    ("WP_GOA_OFF", "Off Goa",           "waypoint", 15.40, 73.30, "", "synthetic"),
    ("WP_MNG_OFF", "Off Mangalore",     "waypoint", 12.90, 74.30, "", "synthetic"),
    ("WP_KOC_OFF", "Off Kochi",         "waypoint",  9.95, 75.80, "", "synthetic"),
    ("WP_TUT_OFF", "Off Tuticorin",     "waypoint",  8.70, 78.60, "", "synthetic"),
    ("WP_CHE_OFF", "Off Chennai",       "waypoint", 13.10, 80.70, "", "synthetic"),
    ("WP_VIZ_OFF", "Off Visakhapatnam", "waypoint", 17.50, 83.70, "", "synthetic"),
    ("WP_COL_OFF", "Off Colombo",       "waypoint",  6.95, 79.50, "", "synthetic"),
    # --- Arabian Sea offshore lane (west of Lakshadweep, via 9-degree channel)
    ("WP_ARAB_N",  "Arabian Sea North",       "waypoint", 17.00, 71.50, "", "synthetic"),
    ("WP_ARAB_W",  "Arabian Sea West",        "waypoint", 13.00, 71.00, "", "synthetic"),
    ("WP_NINE_W",  "Nine Degree Channel W",   "waypoint",  9.20, 71.80, "", "synthetic"),
    ("WP_NINE_E",  "Nine Degree Channel E",   "waypoint",  9.00, 74.50, "", "synthetic"),
    # --- Southern tip and around Sri Lanka ----------------------------------
    ("WP_CAPE_S",  "South of Kanyakumari",    "waypoint",  7.60, 77.50, "", "synthetic"),
    ("WP_SL_SW",   "SW of Sri Lanka",         "waypoint",  5.80, 79.90, "", "synthetic"),
    ("WP_DONDRA",  "South of Dondra Head",    "waypoint",  5.55, 80.60, "", "synthetic"),
    ("WP_SL_SE",   "SE of Sri Lanka",         "waypoint",  6.00, 81.80, "", "synthetic"),
    ("WP_SL_E",    "East of Sri Lanka",       "waypoint",  7.60, 82.30, "", "synthetic"),
    ("WP_TRINCO",  "Off Trincomalee",         "waypoint",  8.80, 81.70, "", "synthetic"),
    # --- Shallow Pamban / Palk Strait passage (small craft only) -----------
    ("WP_PAMBAN",  "South of Pamban Pass",    "waypoint",  9.20, 79.20, "", "synthetic"),
    ("WP_PALK_BAY", "Palk Bay",               "waypoint",  9.60, 79.50, "", "synthetic"),
    ("WP_PALK_STR", "Palk Strait",            "waypoint", 10.05, 79.95, "", "synthetic"),
    # --- Bay of Bengal coastal + offshore lanes -----------------------------
    ("WP_EAST_MID", "Off Pondicherry",        "waypoint", 11.00, 81.00, "", "synthetic"),
    ("WP_KRISHNA", "Off Krishna Delta",       "waypoint", 15.50, 81.30, "", "synthetic"),
    ("WP_GODAVARI", "Off Godavari Delta",     "waypoint", 16.20, 82.60, "", "synthetic"),
    ("WP_BAY_S",   "Bay of Bengal South",     "waypoint", 10.50, 83.00, "", "synthetic"),
    ("WP_BAY_MID", "Bay of Bengal Central",   "waypoint", 13.50, 83.00, "", "synthetic"),
    ("WP_BAY_N",   "Bay of Bengal North",     "waypoint", 16.00, 85.00, "", "synthetic"),
]

# u, v, lane_type, min_depth_m (ILLUSTRATIVE), navigable
_EDGES = [
    # port approaches
    ("INNSA", "WP_MUM_OFF", "approach", 16.0, True),
    ("INMRM", "WP_GOA_OFF", "approach", 16.0, True),
    ("INNML", "WP_MNG_OFF", "approach", 16.0, True),
    ("INCOK", "WP_KOC_OFF", "approach", 16.0, True),
    ("INTUT", "WP_TUT_OFF", "approach", 16.0, True),
    ("INMAA", "WP_CHE_OFF", "approach", 16.0, True),
    ("INVTZ", "WP_VIZ_OFF", "approach", 16.0, True),
    ("LKCMB", "WP_COL_OFF", "approach", 16.0, True),
    # west coast coastal lane
    ("WP_MUM_OFF", "WP_GOA_OFF", "coastal", 200.0, True),
    ("WP_GOA_OFF", "WP_MNG_OFF", "coastal", 200.0, True),
    ("WP_MNG_OFF", "WP_KOC_OFF", "coastal", 200.0, True),
    ("WP_KOC_OFF", "WP_CAPE_S",  "coastal", 200.0, True),
    # Arabian Sea offshore lane
    ("WP_MUM_OFF", "WP_ARAB_N",  "offshore", 1000.0, True),
    ("WP_ARAB_N",  "WP_ARAB_W",  "offshore", 1000.0, True),
    ("WP_GOA_OFF", "WP_ARAB_W",  "offshore", 1000.0, True),
    ("WP_ARAB_W",  "WP_NINE_W",  "offshore", 1000.0, True),
    ("WP_NINE_W",  "WP_NINE_E",  "channel",  1000.0, True),
    ("WP_NINE_E",  "WP_KOC_OFF", "offshore", 1000.0, True),
    ("WP_NINE_E",  "WP_CAPE_S",  "offshore", 1000.0, True),
    # Gulf of Mannar / south
    ("WP_CAPE_S",  "WP_TUT_OFF", "coastal", 200.0, True),
    ("WP_CAPE_S",  "WP_COL_OFF", "offshore", 1000.0, True),
    ("WP_TUT_OFF", "WP_COL_OFF", "offshore", 1000.0, True),
    # around Sri Lanka
    ("WP_COL_OFF", "WP_SL_SW",  "coastal", 200.0, True),
    ("WP_SL_SW",   "WP_DONDRA", "coastal", 200.0, True),
    ("WP_DONDRA",  "WP_SL_SE",  "coastal", 200.0, True),
    ("WP_SL_SE",   "WP_SL_E",   "coastal", 200.0, True),
    ("WP_SL_E",    "WP_TRINCO", "coastal", 200.0, True),
    # shallow Pamban / Palk passage - illustrative depth, small craft only
    ("WP_TUT_OFF",  "WP_PAMBAN",   "shallow_channel", 4.5, True),
    ("WP_PAMBAN",   "WP_PALK_BAY", "shallow_channel", 4.5, True),
    ("WP_PALK_BAY", "WP_PALK_STR", "shallow_channel", 4.5, True),
    ("WP_PALK_STR", "WP_EAST_MID", "shallow_channel", 6.0, True),
    # Bay of Bengal coastal lane
    ("WP_TRINCO",  "WP_EAST_MID", "coastal", 200.0, True),
    ("WP_EAST_MID", "WP_CHE_OFF", "coastal", 200.0, True),
    ("WP_CHE_OFF", "WP_KRISHNA",  "coastal", 200.0, True),
    ("WP_KRISHNA", "WP_GODAVARI", "coastal", 200.0, True),
    ("WP_GODAVARI", "WP_VIZ_OFF", "coastal", 200.0, True),
    # Bay of Bengal offshore lane
    ("WP_TRINCO",  "WP_BAY_S",   "offshore", 1000.0, True),
    ("WP_SL_E",    "WP_BAY_S",   "offshore", 1000.0, True),
    ("WP_BAY_S",   "WP_BAY_MID", "offshore", 1000.0, True),
    ("WP_CHE_OFF", "WP_BAY_MID", "offshore", 1000.0, True),
    ("WP_BAY_MID", "WP_BAY_N",   "offshore", 1000.0, True),
    ("WP_BAY_N",   "WP_VIZ_OFF", "offshore", 1000.0, True),
]


def create_sample_network():
    """Return (waypoints_df, edges_df) for the synthetic demo network."""
    waypoints = pd.DataFrame(_WAYPOINTS, columns=["node_id", "name", "type", "lat", "lon",
                                                  "country", "notes"])
    edges = pd.DataFrame(_EDGES, columns=["u", "v", "lane_type", "min_depth_m", "navigable"])
    waypoints["data_status"] = "synthetic_demo"
    edges["data_status"] = "synthetic_demo"
    edges["bidirectional"] = True
    return waypoints, edges

In [ ]:
USE_CUSTOM_NETWORK = False   # set True to load your own CSV files from data/

WAYPOINTS_CSV = os.path.join(DIRS["data"], "waypoints.csv")
EDGES_CSV = os.path.join(DIRS["data"], "edges.csv")

if USE_CUSTOM_NETWORK:
    waypoints_df = pd.read_csv(WAYPOINTS_CSV)
    edges_df = pd.read_csv(EDGES_CSV)
    NETWORK_NOTICE = "Custom network loaded from data/ - verify its source before use."
else:
    waypoints_df, edges_df = create_sample_network()
    waypoints_df.to_csv(WAYPOINTS_CSV, index=False)   # saved to Google Drive
    edges_df.to_csv(EDGES_CSV, index=False)
    NETWORK_NOTICE = DATA_NOTICE

print("NOTICE:", NETWORK_NOTICE)
print(f"{len(waypoints_df)} nodes ({(waypoints_df.type == 'port').sum()} ports), {len(edges_df)} sea lanes")
display(waypoints_df.head(10))
display(edges_df.head(10))

## 7. Construct the maritime graph
We use a **directed** graph (`nx.DiGraph`). Each sea lane is added in both directions because the ship's heading — and therefore how the waves hit the ship — is different each way. The tables are validated first (missing columns, duplicate IDs, bad coordinates, edges pointing to unknown nodes).

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Validation and graph construction

# ---------------------------------------------------------------------------
# Building the maritime graph (nodes = ports/waypoints, edges = sea lanes)
# ---------------------------------------------------------------------------

REQUIRED_NODE_COLS = {"node_id", "name", "type", "lat", "lon"}
REQUIRED_EDGE_COLS = {"u", "v", "lane_type", "min_depth_m", "navigable"}


def validate_network_tables(waypoints: pd.DataFrame, edges: pd.DataFrame) -> None:
    """Raise ValueError with a clear message if the tables are malformed."""
    missing = REQUIRED_NODE_COLS - set(waypoints.columns)
    if missing:
        raise ValueError(f"waypoints table is missing columns: {sorted(missing)}")
    missing = REQUIRED_EDGE_COLS - set(edges.columns)
    if missing:
        raise ValueError(f"edges table is missing columns: {sorted(missing)}")
    dup = waypoints["node_id"][waypoints["node_id"].duplicated()].tolist()
    if dup:
        raise ValueError(f"duplicate node_id values: {dup}")
    bad = waypoints[(waypoints.lat.abs() > 90) | (waypoints.lon.abs() > 180)]
    if len(bad):
        raise ValueError(f"invalid coordinates for: {bad.node_id.tolist()}")
    ids = set(waypoints["node_id"])
    unknown = sorted((set(edges.u) | set(edges.v)) - ids)
    if unknown:
        raise ValueError(f"edges reference unknown nodes: {unknown}")
    loops = edges[edges.u == edges.v]
    if len(loops):
        raise ValueError(f"self-loop edges found: {loops.u.tolist()}")


def build_graph(waypoints: pd.DataFrame, edges: pd.DataFrame, data_notice: str = "") -> nx.DiGraph:
    """
    Create a DIRECTED graph. Each sea lane is added in both directions
    (unless bidirectional=False) because the ship's heading - and therefore
    the wave direction relative to the ship - is different each way.
    """
    validate_network_tables(waypoints, edges)
    G = nx.DiGraph(data_notice=data_notice)
    for row in waypoints.itertuples(index=False):
        G.add_node(row.node_id, name=row.name, type=row.type,
                   lat=float(row.lat), lon=float(row.lon),
                   data_status=getattr(row, "data_status", "unknown"))
    for row in edges.itertuples(index=False):
        attrs = dict(lane_type=row.lane_type, min_depth_m=float(row.min_depth_m),
                     navigable=bool(row.navigable),
                     data_status=getattr(row, "data_status", "unknown"))
        G.add_edge(row.u, row.v, **attrs)
        if bool(getattr(row, "bidirectional", True)):
            G.add_edge(row.v, row.u, **attrs)
    return G

In [ ]:
G_structure = build_graph(waypoints_df, edges_df, NETWORK_NOTICE)
print("Nodes:", G_structure.number_of_nodes(), "| directed edges:", G_structure.number_of_edges())
print("Weakly connected:", nx.is_weakly_connected(G_structure))
print("Ports:", [n for n, d in G_structure.nodes(data=True) if d["type"] == "port"])

## 8. Calculate geographic distances
Distances use the **haversine (great-circle) formula** on a spherical Earth (radius 6371 km), which is accurate to well under 1 % for these distances. 1 nautical mile = 1.852 km. We also compute each lane's **initial bearing** (the compass heading the ship steers), which we need for wave direction later.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Great-circle helpers and edge geometry

# ---------------------------------------------------------------------------
# Geographic helpers (great-circle maths) and edge geometry
# ---------------------------------------------------------------------------

EARTH_RADIUS_KM = 6371.0088
KM_PER_NM = 1.852


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance between two points in kilometres."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


def km_to_nm(km: float) -> float:
    return km / KM_PER_NM


def initial_bearing_deg(lat1, lon1, lat2, lon2) -> float:
    """Compass direction (0-360, 0 = north) the ship steers at the start of the segment."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    x = math.sin(dl) * math.cos(p2)
    y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


def intermediate_point(lat1, lon1, lat2, lon2, fraction: float) -> Tuple[float, float]:
    """Point at `fraction` (0..1) of the way along the great circle."""
    p1, l1, p2, l2 = map(math.radians, (lat1, lon1, lat2, lon2))
    d = 2 * math.asin(math.sqrt(math.sin((p2 - p1) / 2) ** 2 +
                                math.cos(p1) * math.cos(p2) * math.sin((l2 - l1) / 2) ** 2))
    if d == 0:
        return lat1, lon1
    a = math.sin((1 - fraction) * d) / math.sin(d)
    b = math.sin(fraction * d) / math.sin(d)
    x = a * math.cos(p1) * math.cos(l1) + b * math.cos(p2) * math.cos(l2)
    y = a * math.cos(p1) * math.sin(l1) + b * math.cos(p2) * math.sin(l2)
    z = a * math.sin(p1) + b * math.sin(p2)
    return math.degrees(math.atan2(z, math.hypot(x, y))), math.degrees(math.atan2(y, x))


def add_geometry(G: nx.DiGraph) -> nx.DiGraph:
    """Attach distance (km, nm) and heading to every edge."""
    for u, v, data in G.edges(data=True):
        a, b = G.nodes[u], G.nodes[v]
        km = haversine_km(a["lat"], a["lon"], b["lat"], b["lon"])
        data["distance_km"] = km
        data["distance_nm"] = km_to_nm(km)
        data["bearing_deg"] = initial_bearing_deg(a["lat"], a["lon"], b["lat"], b["lon"])
    return G


def load_network_from_dataframes(waypoints, edges, data_notice="") -> nx.DiGraph:
    return add_geometry(build_graph(waypoints, edges, data_notice))


def load_network_from_csv(waypoints_csv: str, edges_csv: str, data_notice: str = "") -> nx.DiGraph:
    """Load any network that uses the same column names (real data can be plugged in here)."""
    return load_network_from_dataframes(pd.read_csv(waypoints_csv), pd.read_csv(edges_csv), data_notice)

In [ ]:
G = load_network_from_dataframes(waypoints_df, edges_df, NETWORK_NOTICE)

# Sanity check: Mumbai (JNPT) to Chennai straight-line distance should be about 1,020 km
a, b = G.nodes["INNSA"], G.nodes["INMAA"]
check_km = haversine_km(a["lat"], a["lon"], b["lat"], b["lon"])
print(f"Great-circle Mumbai-Chennai: {check_km:.0f} km = {km_to_nm(check_km):.0f} nm")
assert 1000 < check_km < 1040

try:   # optional cross-check with geopy if it is installed
    from geopy.distance import great_circle
    gp = great_circle((a["lat"], a["lon"]), (b["lat"], b["lon"])).km
    print(f"geopy great_circle: {gp:.0f} km (difference {abs(gp - check_km):.2f} km)")
except ImportError:
    print("geopy not installed - skipped optional cross-check")

edge_table = pd.DataFrame([{"from": u, "to": v, "lane_type": d["lane_type"],
                            "distance_km": round(d["distance_km"], 1),
                            "distance_nm": round(d["distance_nm"], 1),
                            "heading_deg": round(d["bearing_deg"]), "depth_m (illustrative)": d["min_depth_m"]}
                           for u, v, d in G.edges(data=True)])
display(edge_table.head(12))

## 9. Integrate the marine weather API
**Provider design.** The routing code only talks to a `WeatherProvider`. Two providers are included:

* `OpenMeteoProvider` – **real forecasts**, no API key:
  * Marine API `https://marine-api.open-meteo.com/v1/marine` → `wave_height` (m), `wave_direction` (°, direction waves come **from**), `wave_period` (s), `swell_wave_height` (m). We pass `cell_selection=sea` so coastal points use the nearest sea grid cell.
  * Forecast API `https://api.open-meteo.com/v1/forecast` → `wind_speed_10m`, `wind_direction_10m`, `wind_gusts_10m` with `wind_speed_unit=kn`.
  * Many points are sent in **one request** (comma-separated coordinates; the API returns a list in the same order).
  * Errors: HTTP 400 returns `{"error": true, "reason": ...}`; 429 (rate limit) and 5xx are retried with back-off. Published free limits: 600 calls/min, 5,000/hour, <10,000/day, non-commercial use only.
  * Results are **cached** in memory and in `results/cache/weather_cache.json` (default 3 hours).
  * Inside the planning window it keeps the **worst** (maximum) value per point – a conservative prototype choice.
* `SyntheticWeatherProvider` – made-up, reproducible weather for tests.

To add another provider later, write a new class with a `fetch_points()` method and select it in `WeatherConfig.provider`.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Weather configuration, record format and provider base class

# ---------------------------------------------------------------------------
# Weather configuration and common helpers
# ---------------------------------------------------------------------------

# The weather fields every provider must return for a point.
# None means "not available" - it is NEVER silently replaced by 0.
WEATHER_FIELDS = [
    "wave_height_m",        # significant height of mean waves (m)
    "wave_direction_deg",   # direction waves come FROM (0 = from north)
    "wave_period_s",        # mean wave period (s)
    "swell_wave_height_m",  # primary swell height (m)
    "wind_speed_kn",        # 10 m wind speed (knots)
    "wind_direction_deg",   # direction wind comes FROM (0 = from north)
    "wind_gusts_kn",        # 10 m wind gusts (knots)
]


@dataclass
class WeatherConfig:
    """All weather-provider settings in one place (easy to save as JSON)."""
    provider: str = "open-meteo"                       # "open-meteo" or "synthetic"
    marine_url: str = "https://marine-api.open-meteo.com/v1/marine"
    forecast_url: str = "https://api.open-meteo.com/v1/forecast"
    marine_hourly: Tuple[str, ...] = ("wave_height", "wave_direction",
                                      "wave_period", "swell_wave_height")
    wind_hourly: Tuple[str, ...] = ("wind_speed_10m", "wind_direction_10m",
                                    "wind_gusts_10m")
    marine_max_forecast_days: int = 7      # Marine API allows up to 8; keep a margin
    wind_max_forecast_days: int = 15       # Forecast API allows up to 16
    batch_size: int = 50                   # coordinates per HTTP request
    timeout_s: float = 20.0
    max_retries: int = 2                   # retries for 429 / 5xx / network errors
    backoff_s: float = 2.0                 # first back-off delay (doubles each retry)
    min_interval_s: float = 0.3            # polite pause between requests
    cache_ttl_hours: float = 3.0           # re-use cached forecasts this long
    cache_file: Optional[str] = None       # JSON file for the disk cache (optional)
    coord_decimals: int = 2                # rounding for cache keys (~1 km)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["marine_hourly"] = list(self.marine_hourly)
        d["wind_hourly"] = list(self.wind_hourly)
        return d


def point_key(lat: float, lon: float, decimals: int = 2) -> str:
    """Stable text key for a coordinate, e.g. '15.40,73.30'."""
    return f"{round(lat, decimals):.{decimals}f},{round(lon, decimals):.{decimals}f}"


def empty_point_weather(source: str, error: Optional[str] = None) -> dict:
    """A weather record where every field is missing."""
    rec = {f: None for f in WEATHER_FIELDS}
    rec.update({"source": source, "time_of_max_wave": None,
                "time_of_max_wind": None, "error": error})
    return rec


def to_utc(dt) -> datetime:
    """Accept datetime or ISO string; return a timezone-aware UTC datetime."""
    if isinstance(dt, str):
        dt = datetime.fromisoformat(dt.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _clean(value):
    """Convert API numbers to float; keep None / NaN as None."""
    if value is None:
        return None
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(v) else v


class WeatherProvider:
    """Base class. Subclasses return one weather record per point."""
    name = "base"

    def fetch_points(self, points: Sequence[Tuple[float, float]],
                     start_time, window_hours: int) -> Dict[str, dict]:
        """
        points       : list of (lat, lon)
        start_time   : departure time (datetime or ISO string, UTC assumed)
        window_hours : planning window; the WORST value inside the window
                       is returned (a conservative choice for a prototype)
        Returns {point_key: record} where record has the WEATHER_FIELDS keys,
        a 'source' tag and an 'error' message (None if all good).
        """
        raise NotImplementedError

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Real data: Open-Meteo provider with batching, retries and caching

# ---------------------------------------------------------------------------
# Real data: Open-Meteo Marine + Forecast APIs
# ---------------------------------------------------------------------------

class OpenMeteoProvider(WeatherProvider):
    """
    Fetches REAL forecasts from Open-Meteo (no API key needed).

    * Batches many coordinates into one request (comma-separated lat/lon;
      the API then returns a JSON list in the same order).
    * Retries on HTTP 429 (rate limit), 5xx and network errors with back-off.
    * Handles the documented error format {"error": true, "reason": "..."}.
    * Caches results in memory and (optionally) in a JSON file.
    * Never crashes the caller: failures become records with None values
      and an 'error' message.
    """
    name = "open-meteo"

    def __init__(self, config: Optional[WeatherConfig] = None,
                 session: Optional[requests.Session] = None):
        self.config = config or WeatherConfig()
        self.session = session or requests.Session()
        self.session.headers.update({"User-Agent": "coastal-platform-ship-routing/1.0 (academic prototype)"})
        self._cache: Dict[str, dict] = {}
        self._last_request = 0.0
        self.stats = {"http_requests": 0, "cache_hits": 0, "failures": 0}
        self._load_disk_cache()

    # ----- caching ---------------------------------------------------------
    def _load_disk_cache(self):
        path = self.config.cache_file
        if path and os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as fh:
                    self._cache = json.load(fh)
                logger.info("Loaded %d cached weather records", len(self._cache))
            except (OSError, json.JSONDecodeError) as exc:
                logger.warning("Weather cache unreadable (%s) - starting empty", exc)
                self._cache = {}

    def save_cache(self):
        path = self.config.cache_file
        if not path:
            return
        try:
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            with open(path, "w", encoding="utf-8") as fh:
                json.dump(self._cache, fh)
        except OSError as exc:
            logger.warning("Could not save weather cache: %s", exc)

    def _cache_get(self, key: str) -> Optional[dict]:
        entry = self._cache.get(key)
        if not entry:
            return None
        age_h = (time.time() - entry.get("fetched_at", 0)) / 3600.0
        if age_h > self.config.cache_ttl_hours:
            return None
        self.stats["cache_hits"] += 1
        return entry["data"]

    def _cache_put(self, key: str, data: dict):
        self._cache[key] = {"fetched_at": time.time(), "data": data}

    # ----- HTTP ------------------------------------------------------------
    def _get_json(self, url: str, params: dict):
        """GET with rate-limit pause, retries and error handling.
        Returns (json_or_None, error_message_or_None)."""
        delay = self.config.backoff_s
        last_error = None
        for attempt in range(self.config.max_retries + 1):
            wait = self.config.min_interval_s - (time.time() - self._last_request)
            if wait > 0:
                time.sleep(wait)
            self._last_request = time.time()
            try:
                self.stats["http_requests"] += 1
                resp = self.session.get(url, params=params, timeout=self.config.timeout_s)
            except requests.RequestException as exc:
                last_error = f"network error ({exc.__class__.__name__})"
                logger.debug("Weather request exception: %s", exc)
            else:
                if resp.status_code == 200:
                    try:
                        data = resp.json()
                    except ValueError:
                        return None, "invalid JSON in response"
                    if isinstance(data, dict) and data.get("error"):
                        return None, f"API error: {data.get('reason')}"
                    return data, None
                if resp.status_code == 400:
                    # Documented format: {"error": true, "reason": "..."}; not retryable
                    try:
                        reason = resp.json().get("reason")
                    except ValueError:
                        reason = resp.text[:200]
                    return None, f"HTTP 400: {reason}"
                last_error = f"HTTP {resp.status_code}"
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    if retry_after and retry_after.isdigit():
                        delay = max(delay, float(retry_after))
                    last_error = "HTTP 429: rate limit reached"
                elif resp.status_code < 500:
                    return None, last_error        # other 4xx: do not retry
            if attempt < self.config.max_retries:
                logger.warning("Weather request failed (%s) - retry %d in %.1fs",
                               last_error, attempt + 1, delay)
                time.sleep(delay)
                delay *= 2
        self.stats["failures"] += 1
        return None, last_error

    @staticmethod
    def _window_max(hourly: dict, value_field: str, dir_field: Optional[str],
                    t0: datetime, t1: datetime):
        """Largest value of `value_field` within [t0, t1] and the matching direction."""
        times = hourly.get("time") or []
        values = hourly.get(value_field) or []
        dirs = (hourly.get(dir_field) or []) if dir_field else []
        best, best_dir, best_time = None, None, None
        for i, ts in enumerate(times):
            try:
                t = datetime.fromisoformat(ts).replace(tzinfo=timezone.utc)
            except (TypeError, ValueError):
                continue
            if t < t0 or t > t1 or i >= len(values):
                continue
            v = _clean(values[i])
            if v is None:
                continue
            if best is None or v > best:
                best, best_time = v, ts
                best_dir = _clean(dirs[i]) if i < len(dirs) else None
        return best, best_dir, best_time

    def _request_batch(self, url: str, batch: List[Tuple[float, float]],
                       hourly_vars: Sequence[str], t0: datetime, t1: datetime,
                       extra: dict):
        params = {
            "latitude": ",".join(f"{p[0]:.4f}" for p in batch),
            "longitude": ",".join(f"{p[1]:.4f}" for p in batch),
            "hourly": ",".join(hourly_vars),
            "timezone": "GMT",
            "start_hour": t0.strftime("%Y-%m-%dT%H:00"),
            "end_hour": t1.strftime("%Y-%m-%dT%H:00"),
            **extra,
        }
        data, err = self._get_json(url, params)
        if err and err.startswith("HTTP 400"):
            # Fallback: ask for a plain forecast range and filter locally
            logger.warning("start_hour/end_hour rejected (%s); retrying with forecast_days", err)
            params.pop("start_hour"); params.pop("end_hour")
            params["forecast_days"] = 7
            data, err = self._get_json(url, params)
        if data is None:
            return None, err
        if isinstance(data, dict):            # single location -> single object
            data = [data]
        if len(data) != len(batch):
            return None, f"unexpected response length {len(data)} for {len(batch)} points"
        return data, None

    def fetch_points(self, points, start_time, window_hours):
        cfg = self.config
        t0 = to_utc(start_time).replace(minute=0, second=0, microsecond=0)
        t1 = t0 + timedelta(hours=max(1, int(window_hours)))
        now = datetime.now(timezone.utc)
        marine_end = now + timedelta(days=cfg.marine_max_forecast_days)
        wind_end = now + timedelta(days=cfg.wind_max_forecast_days)

        results: Dict[str, dict] = {}
        todo: List[Tuple[float, float]] = []
        window_tag = f"{t0:%Y%m%d%H}-{t1:%Y%m%d%H}"
        for lat, lon in points:
            key = point_key(lat, lon, cfg.coord_decimals)
            if key in results:
                continue
            cached = self._cache_get(f"{key}|{window_tag}")
            if cached is not None:
                results[key] = cached
            else:
                results[key] = empty_point_weather(self.name)
                todo.append((lat, lon))

        if todo and t0 > marine_end:
            msg = "departure is beyond the marine forecast horizon"
            for lat, lon in todo:
                results[point_key(lat, lon, cfg.coord_decimals)]["error"] = msg
            return results

        m_t1 = min(t1, marine_end)
        w_t1 = min(t1, wind_end)
        for i in range(0, len(todo), cfg.batch_size):
            batch = todo[i:i + cfg.batch_size]
            marine, m_err = self._request_batch(cfg.marine_url, batch, cfg.marine_hourly,
                                                t0, m_t1, {"cell_selection": "sea"})
            wind, w_err = self._request_batch(cfg.forecast_url, batch, cfg.wind_hourly,
                                              t0, w_t1, {"wind_speed_unit": "kn"})
            if marine is None and wind is None and i + cfg.batch_size < len(todo):
                # Both APIs unreachable: stop early instead of retrying every batch
                logger.error("Weather APIs unavailable (%s / %s) - remaining points marked missing",
                             m_err, w_err)
                for lat, lon in todo[i:]:
                    results[point_key(lat, lon, cfg.coord_decimals)]["error"] = \
                        f"marine: {m_err}; wind: {w_err}"
                break
            for j, (lat, lon) in enumerate(batch):
                key = point_key(lat, lon, cfg.coord_decimals)
                rec = results[key]
                errors = []
                if marine is not None:
                    h = marine[j].get("hourly", {})
                    wh, wd, wt = self._window_max(h, "wave_height", "wave_direction", t0, m_t1)
                    wp, _, _ = self._window_max(h, "wave_period", None, t0, m_t1)
                    sw, _, _ = self._window_max(h, "swell_wave_height", None, t0, m_t1)
                    rec.update({"wave_height_m": wh, "wave_direction_deg": wd,
                                "wave_period_s": wp, "swell_wave_height_m": sw,
                                "time_of_max_wave": wt})
                    if wh is None:
                        errors.append("no wave data for this cell/time")
                else:
                    errors.append(f"marine: {m_err}")
                if wind is not None:
                    h = wind[j].get("hourly", {})
                    ws, wdir, wt = self._window_max(h, "wind_speed_10m", "wind_direction_10m", t0, w_t1)
                    wg, _, _ = self._window_max(h, "wind_gusts_10m", None, t0, w_t1)
                    rec.update({"wind_speed_kn": ws, "wind_direction_deg": wdir,
                                "wind_gusts_kn": wg, "time_of_max_wind": wt})
                    if ws is None:
                        errors.append("no wind data for this cell/time")
                else:
                    errors.append(f"wind: {w_err}")
                rec["error"] = "; ".join(errors) if errors else None
                # Only cache complete successes so failures are retried next time
                if marine is not None and wind is not None:
                    self._cache_put(f"{key}|{window_tag}", rec)
        self.save_cache()
        return results

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Synthetic test provider (NOT real data) and provider factory

# ---------------------------------------------------------------------------
# Synthetic data: reproducible test weather (NOT REAL)
# ---------------------------------------------------------------------------

def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


@dataclass
class StormZone:
    """A circular area of bad weather used ONLY for synthetic test scenarios.
    Full strength inside core_fraction*radius, fading linearly to calm at radius."""
    name: str
    lat: float
    lon: float
    radius_km: float
    peak_wave_m: float
    peak_wind_kn: float
    wave_direction_deg: float = 225.0
    wind_direction_deg: float = 225.0
    core_fraction: float = 0.5


@dataclass
class MissingZone:
    """Area where the synthetic provider pretends no data is available."""
    lat: float
    lon: float
    radius_km: float


class SyntheticWeatherProvider(WeatherProvider):
    """Deterministic made-up weather for reproducible tests. Tagged 'synthetic'."""
    name = "synthetic"

    def __init__(self, scenario_name: str = "calm",
                 base_wave_m: float = 1.0, base_wind_kn: float = 10.0,
                 base_wave_direction_deg: float = 240.0,
                 storms: Optional[List[StormZone]] = None,
                 missing_zones: Optional[List[MissingZone]] = None,
                 noise_fraction: float = 0.05, seed: int = 42):
        self.scenario_name = scenario_name
        self.base_wave_m = base_wave_m
        self.base_wind_kn = base_wind_kn
        self.base_dir = base_wave_direction_deg
        self.storms = storms or []
        self.missing_zones = missing_zones or []
        self.noise_fraction = noise_fraction
        self.seed = seed

    def _noise(self, lat, lon):
        # Deterministic pseudo-random number in [-1, 1] from coordinates + seed
        x = math.sin(lat * 12.9898 + lon * 78.233 + self.seed) * 43758.5453
        return (x - math.floor(x)) * 2 - 1

    def fetch_points(self, points, start_time, window_hours):
        out = {}
        for lat, lon in points:
            key = point_key(lat, lon)
            if any(_haversine_km(lat, lon, z.lat, z.lon) <= z.radius_km for z in self.missing_zones):
                out[key] = empty_point_weather(f"synthetic:{self.scenario_name}",
                                               error="synthetic missing-data zone")
                continue
            wave, wind = self.base_wave_m, self.base_wind_kn
            wave_dir, wind_dir = self.base_dir, self.base_dir
            for s in self.storms:
                d = _haversine_km(lat, lon, s.lat, s.lon)
                if d >= s.radius_km:
                    continue
                core = s.core_fraction * s.radius_km
                f = 1.0 if d <= core else 1.0 - (d - core) / (s.radius_km - core)
                w = self.base_wave_m + (s.peak_wave_m - self.base_wave_m) * f
                v = self.base_wind_kn + (s.peak_wind_kn - self.base_wind_kn) * f
                if w > wave:
                    wave, wave_dir = w, s.wave_direction_deg
                if v > wind:
                    wind, wind_dir = v, s.wind_direction_deg
            n = 1 + self.noise_fraction * self._noise(lat, lon)
            out[key] = {
                "wave_height_m": round(wave * n, 2),
                "wave_direction_deg": wave_dir,
                "wave_period_s": round(4 + 1.6 * wave, 1),
                "swell_wave_height_m": round(0.6 * wave * n, 2),
                "wind_speed_kn": round(wind * n, 1),
                "wind_direction_deg": wind_dir,
                "wind_gusts_kn": round(wind * 1.3 * n, 1),
                "source": f"synthetic:{self.scenario_name}",
                "time_of_max_wave": None, "time_of_max_wind": None, "error": None,
            }
        return out


def create_weather_provider(config: Optional[WeatherConfig] = None, **synthetic_kwargs) -> WeatherProvider:
    """Factory: choose the provider by name so it stays configurable."""
    config = config or WeatherConfig()
    if config.provider == "open-meteo":
        return OpenMeteoProvider(config)
    if config.provider == "synthetic":
        return SyntheticWeatherProvider(**synthetic_kwargs)
    raise ValueError(f"Unknown weather provider '{config.provider}'. "
                     "Use 'open-meteo' or 'synthetic', or add a new WeatherProvider subclass.")

In [ ]:
WEATHER_CONFIG = WeatherConfig(provider="open-meteo",
                               cache_file=os.path.join(DIRS["cache"], "weather_cache.json"))
live_provider = create_weather_provider(WEATHER_CONFIG)

DEPARTURE = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)
print("Planned departure (UTC):", DEPARTURE.isoformat())

# Quick test: one point off Mumbai, next 24 hours
test = live_provider.fetch_points([(18.90, 72.40)], DEPARTURE, window_hours=24)
record = next(iter(test.values()))
print(json.dumps(record, indent=2))
if record["error"]:
    print("\nWeather API problem:", record["error"])
    print("The notebook continues: missing data is handled conservatively.")
else:
    print("\nReal forecast received. Source:", record["source"])

## 10. Fetch weather data for route segments
For every lane we take weather **samples**: both end points plus points every ~100 km along the great circle. Identical points are only requested once. The **planning window** runs from departure to roughly the expected voyage time plus a 12-hour buffer.

> Limitation: this is not *time-dependent* routing. For each point we use the worst forecast during the whole window, which is safe-side but can be pessimistic for long voyages. Open-Meteo marine forecasts reach about 7–8 days ahead.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Sampling points along lanes and fetching their weather

# ---------------------------------------------------------------------------
# Sampling points along segments and fetching their weather
# ---------------------------------------------------------------------------

def sample_points_for_edge(G: nx.DiGraph, u: str, v: str, spacing_km: float = 100.0):
    """Both end points plus evenly spaced points every ~spacing_km (at least the midpoint)."""
    a, b = G.nodes[u], G.nodes[v]
    km = G.edges[u, v]["distance_km"]
    n = max(2, int(math.ceil(km / spacing_km)))
    return [intermediate_point(a["lat"], a["lon"], b["lat"], b["lon"], i / n) for i in range(n + 1)]


def collect_sample_points(G: nx.DiGraph, spacing_km: float = 100.0):
    """Unique sample points for the whole graph + mapping edge -> point keys."""
    unique: Dict[str, Tuple[float, float]] = {}
    edge_points: Dict[Tuple[str, str], List[str]] = {}
    for u, v in G.edges():
        keys = []
        for lat, lon in sample_points_for_edge(G, u, v, spacing_km):
            k = point_key(lat, lon)
            unique.setdefault(k, (round(lat, 2), round(lon, 2)))
            keys.append(k)
        edge_points[(u, v)] = keys
    return unique, edge_points


def auto_window_hours(G, source, target, speed_kn, config: "RoutingConfig") -> int:
    """Hours of forecast to consider: rough voyage time (great circle x1.6) + 12 h buffer."""
    if config.weather_window_hours:
        return int(min(config.weather_window_hours, config.max_weather_window_hours))
    a, b = G.nodes[source], G.nodes[target]
    nm = km_to_nm(haversine_km(a["lat"], a["lon"], b["lat"], b["lon"]))
    hours = int(math.ceil(nm * 1.6 / max(speed_kn, 1.0))) + 12
    return int(min(max(hours, 12), config.max_weather_window_hours))


def fetch_network_weather(G, provider: WeatherProvider, departure, window_hours: int,
                          spacing_km: float = 100.0):
    """Fetch weather for every sample point. Never raises: a provider crash
    becomes 'missing' records so routing can still decide conservatively."""
    unique, edge_points = collect_sample_points(G, spacing_km)
    try:
        weather = provider.fetch_points(list(unique.values()), departure, window_hours)
    except Exception as exc:          # noqa: BLE001 - protect the caller
        logger.error("Weather provider failed: %s", exc)
        weather = {}
        err = f"provider exception: {exc.__class__.__name__}: {exc}"
    else:
        err = "no record returned"
    for k in unique:
        if k not in weather:
            weather[k] = empty_point_weather(getattr(provider, "name", "unknown"), error=err)
    return weather, edge_points

In [ ]:
LIVE_SOURCE, LIVE_DEST = "INNSA", "INMAA"       # Mumbai (JNPT) -> Chennai
live_window = auto_window_hours(G, LIVE_SOURCE, LIVE_DEST, VESSEL_PRESETS["container_ship"].service_speed_kn,
                                ROUTING_CONFIG)
t0 = time.time()
live_weather, live_edge_points = fetch_network_weather(G, live_provider, DEPARTURE, live_window,
                                                       ROUTING_CONFIG.sample_spacing_km)
print(f"Weather window: {live_window} h | sample points: {len(live_weather)} | "
      f"time {time.time() - t0:.1f}s | provider stats: {getattr(live_provider, 'stats', {})}")

live_points_df = pd.DataFrame([{"point": k, **v} for k, v in live_weather.items()])
missing = live_points_df["wave_height_m"].isna() | live_points_df["wind_speed_kn"].isna()
print(f"Points with missing wave or wind data: {missing.sum()} of {len(live_points_df)}")
live_points_df.to_csv(os.path.join(DIRS["tables"], "live_weather_points.csv"), index=False)
display(live_points_df.head(10))

## 11. Calculate weather risk scores
**Per sample point**
1. Relative wave angle = angle between where the waves come from and the ship's heading (0° head sea, 90° beam sea, 180° following sea).
2. Effective wave height = wave height × direction factor (head 1.10, beam 1.20, following 1.00 – prototype heuristic).
3. Effective wave height, wind and gusts are each classified with the thresholds; the point's level is the **worst** of them.
4. Severity index = largest of value ÷ Severe threshold (1.0 means "at the Severe threshold").

**Per lane:** the **worst** sample point decides the risk level. Missing data with the default `"conservative"` policy counts as at least **Moderate** plus an extra penalty — unknown weather is never treated as safe.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Risk classification for points and lanes

# ---------------------------------------------------------------------------
# Weather risk scoring for points and segments
# ---------------------------------------------------------------------------

def classify_value(value: Optional[float], thresholds: Sequence[float]) -> Optional[int]:
    """0=Low, 1=Moderate, 2=High, 3=Severe. None if value missing."""
    if value is None:
        return None
    level = 0
    for t in thresholds:
        if value >= t:
            level += 1
    return level


def relative_wave_angle(wave_from_deg: Optional[float], heading_deg: float) -> Optional[float]:
    """0 = waves from dead ahead, 90 = beam, 180 = from astern."""
    if wave_from_deg is None:
        return None
    diff = abs((wave_from_deg - heading_deg + 180) % 360 - 180)
    return diff


def encounter_type(angle: Optional[float], config: RoutingConfig) -> Tuple[str, float]:
    if angle is None:
        return "unknown", config.beam_sea_factor       # unknown direction -> worst case factor
    if angle <= config.head_sea_max_angle:
        return "head sea", config.head_sea_factor
    if angle >= config.following_sea_min_angle:
        return "following sea", config.following_sea_factor
    return "beam sea", config.beam_sea_factor


def assess_point(rec: dict, heading_deg: float, vessel: VesselProfile, config: RoutingConfig) -> dict:
    """Risk assessment of one weather record for a ship on a given heading."""
    wave_thr = vessel.wave_thresholds_m or config.wave_thresholds_m
    hs = rec.get("wave_height_m")
    angle = relative_wave_angle(rec.get("wave_direction_deg"), heading_deg)
    enc, factor = encounter_type(angle, config)
    eff = hs * factor if hs is not None else None
    wind = rec.get("wind_speed_kn")
    gust = rec.get("wind_gusts_kn")
    levels = {"wave": classify_value(eff, wave_thr),
              "wind": classify_value(wind, config.wind_thresholds_kn),
              "gust": classify_value(gust, config.gust_thresholds_kn)}
    available = [l for l in levels.values() if l is not None]
    complete = hs is not None and wind is not None
    level = max(available) if available else 0
    ratios = []
    if eff is not None:
        ratios.append(eff / wave_thr[2])
    if wind is not None:
        ratios.append(wind / config.wind_thresholds_kn[2])
    if gust is not None:
        ratios.append(gust / config.gust_thresholds_kn[2])
    severity = max(ratios) if ratios else None

    reasons = []
    if hs is not None and hs > vessel.max_wave_height_m:
        reasons.append(f"wave height {hs:.1f} m exceeds vessel limit {vessel.max_wave_height_m} m")
    if wind is not None and wind > vessel.max_wind_speed_kn:
        reasons.append(f"wind {wind:.0f} kn exceeds vessel limit {vessel.max_wind_speed_kn} kn")
    if not complete:
        if config.missing_data_policy == "conservative":
            level = max(level, config.missing_data_min_level)
        elif config.missing_data_policy == "block":
            reasons.append("weather data missing (policy = block)")
    if level == 3 and config.block_severe_segments:
        reasons.append("Severe weather risk (severe segments are blocked)")
    return {"risk_level": level, "levels": levels, "severity_index": severity,
            "effective_wave_height_m": eff, "relative_wave_angle_deg": angle,
            "encounter": enc, "data_complete": complete, "violations": reasons,
            "record": rec}


def assess_segment(point_recs: List[dict], heading_deg: float, km: float,
                   vessel: VesselProfile, config: RoutingConfig) -> dict:
    """Segment risk = WORST sample point on the segment (conservative)."""
    pts = [assess_point(r, heading_deg, vessel, config) for r in point_recs]
    worst = max(pts, key=lambda p: (p["risk_level"], p["severity_index"] or 0))
    complete = all(p["data_complete"] for p in pts)
    # Report the violations of the worst point (avoids repeating similar messages)
    violations = worst["violations"] or sorted({v for p in pts for v in p["violations"]})
    level = max(p["risk_level"] for p in pts)
    penalty = km * config.penalty_multipliers[level]
    if not complete and config.missing_data_policy == "conservative":
        penalty += km * config.missing_data_penalty_multiplier
    rec = worst["record"]
    sources = sorted({str(p["record"].get("source")) for p in pts})
    errors = sorted({p["record"]["error"] for p in pts if p["record"].get("error")})
    return {
        "risk_level": level, "risk_label": RISK_LABELS[level],
        "severity_index": worst["severity_index"],
        "weather_penalty": penalty, "weather_feasible": not violations,
        "violations": violations, "data_complete": complete,
        "missing_points": sum(not p["data_complete"] for p in pts), "points_sampled": len(pts),
        "wave_height_m": rec.get("wave_height_m"), "wave_direction_deg": rec.get("wave_direction_deg"),
        "wave_period_s": rec.get("wave_period_s"), "swell_wave_height_m": rec.get("swell_wave_height_m"),
        "effective_wave_height_m": worst["effective_wave_height_m"],
        "relative_wave_angle_deg": worst["relative_wave_angle_deg"], "encounter": worst["encounter"],
        "wind_speed_kn": rec.get("wind_speed_kn"), "wind_direction_deg": rec.get("wind_direction_deg"),
        "wind_gusts_kn": rec.get("wind_gusts_kn"),
        "max_wave_height_on_segment_m": max((p["record"].get("wave_height_m") or 0) for p in pts) if any(
            p["record"].get("wave_height_m") is not None for p in pts) else None,
        "max_wind_on_segment_kn": max((p["record"].get("wind_speed_kn") or 0) for p in pts) if any(
            p["record"].get("wind_speed_kn") is not None for p in pts) else None,
        "data_sources": sources, "data_errors": errors[:3],
    }


def annotate_graph_with_weather(G: nx.DiGraph, weather: Dict[str, dict], edge_points,
                                vessel: VesselProfile, config: RoutingConfig) -> nx.DiGraph:
    """Store a weather assessment and navigational feasibility on every edge."""
    for u, v, data in G.edges(data=True):
        recs = [weather[k] for k in edge_points[(u, v)]]
        data["weather"] = assess_segment(recs, data["bearing_deg"], data["distance_km"], vessel, config)
        data["nav_feasible"], data["nav_reason"] = navigational_check(data, vessel, config)
    return G


def navigational_check(edge: dict, vessel: VesselProfile, config: RoutingConfig):
    """Depth / navigability check (independent of weather)."""
    if not edge.get("navigable", True):
        return False, "segment marked non-navigable"
    need = vessel.draft_m + config.under_keel_clearance_m
    if edge.get("min_depth_m", 1e9) < need:
        return False, (f"depth {edge['min_depth_m']:.1f} m < draft {vessel.draft_m} m + "
                       f"clearance {config.under_keel_clearance_m} m")
    return True, None

In [ ]:
# Small worked examples (container ship heading north = 0 deg)
vessel_demo = get_vessel_profile("container_ship")
examples = [
    {"label": "calm",               "wave_height_m": 1.2, "wave_direction_deg": 0,   "wind_speed_kn": 10, "wind_gusts_kn": 14},
    {"label": "beam swell",         "wave_height_m": 3.0, "wave_direction_deg": 90,  "wind_speed_kn": 18, "wind_gusts_kn": 24},
    {"label": "following sea",      "wave_height_m": 3.0, "wave_direction_deg": 180, "wind_speed_kn": 18, "wind_gusts_kn": 24},
    {"label": "gale",               "wave_height_m": 4.6, "wave_direction_deg": 10,  "wind_speed_kn": 36, "wind_gusts_kn": 46},
    {"label": "storm",              "wave_height_m": 6.5, "wave_direction_deg": 30,  "wind_speed_kn": 50, "wind_gusts_kn": 65},
    {"label": "missing wave data",  "wave_height_m": None, "wave_direction_deg": None, "wind_speed_kn": 12, "wind_gusts_kn": 16},
]
rows = []
for ex in examples:
    p = assess_point(ex, heading_deg=0, vessel=vessel_demo, config=ROUTING_CONFIG)
    rows.append({"case": ex["label"], "wave_m": ex["wave_height_m"], "encounter": p["encounter"],
                 "effective_wave_m": None if p["effective_wave_height_m"] is None else round(p["effective_wave_height_m"], 2),
                 "wind_kn": ex["wind_speed_kn"], "risk": RISK_LABELS[p["risk_level"]],
                 "severity_index": None if p["severity_index"] is None else round(p["severity_index"], 2),
                 "data_complete": p["data_complete"], "violations": "; ".join(p["violations"])})
display(pd.DataFrame(rows))

# Apply to the whole network with the LIVE weather for a container ship
G_live = G.copy()
annotate_graph_with_weather(G_live, live_weather, live_edge_points, vessel_demo, ROUTING_CONFIG)
lane_risk = pd.DataFrame([{"from": u, "to": v, "risk": d["weather"]["risk_label"],
                           "wave_m": d["weather"]["wave_height_m"], "wind_kn": d["weather"]["wind_speed_kn"],
                           "encounter": d["weather"]["encounter"], "penalty": round(d["weather"]["weather_penalty"], 1),
                           "data_complete": d["weather"]["data_complete"],
                           "available": d["nav_feasible"] and d["weather"]["weather_feasible"]}
                          for u, v, d in G_live.edges(data=True)])
print("Lane risk counts (live weather):", lane_risk["risk"].value_counts().to_dict())
display(lane_risk.head(12))

## 12. Implement Dijkstra's algorithm
Dijkstra's algorithm repeatedly takes the unvisited node with the **smallest known cost** (using a priority queue) and relaxes its outgoing edges. With non-negative costs it is guaranteed to find the cheapest path.

We write it ourselves first (to understand it), then check that it gives exactly the same answer as NetworkX's `dijkstra_path`. A weight function returning `None` means "this edge may not be used".

The **shortest-distance route** uses distance only and ignores weather (it still respects depth). It is the baseline to compare against.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Dijkstra (manual + NetworkX) and the shortest-distance route

# ---------------------------------------------------------------------------
# Dijkstra's algorithm and the shortest-distance route
# ---------------------------------------------------------------------------

def make_distance_weight(G: nx.DiGraph):
    """Weight = distance only. Navigationally impossible edges are hidden (None).
    Weather is IGNORED here on purpose - this is the baseline route."""
    def weight(u, v, d):
        if not d.get("nav_feasible", True):
            return None
        return d["distance_km"]
    return weight


def dijkstra_manual(G: nx.DiGraph, source: str, target: str, weight_fn: Callable):
    """
    Textbook Dijkstra with a priority queue (for learning; NetworkX is used
    in production and the notebook checks both give the same answer).
    Returns (cost, path) or (inf, None) if unreachable.
    """
    dist = {source: 0.0}
    prev: Dict[str, Optional[str]] = {source: None}
    heap = [(0.0, source)]
    visited = set()
    while heap:
        d_u, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        if u == target:
            path = []
            while u is not None:
                path.append(u)
                u = prev[u]
            return d_u, path[::-1]
        for v, data in G[u].items():
            w = weight_fn(u, v, data)
            if w is None or v in visited:      # None = edge not allowed
                continue
            nd = d_u + w
            if nd < dist.get(v, math.inf):
                dist[v], prev[v] = nd, u
                heapq.heappush(heap, (nd, v))
    return math.inf, None


def shortest_distance_route(G: nx.DiGraph, source: str, target: str) -> Optional[List[str]]:
    try:
        return nx.dijkstra_path(G, source, target, weight=make_distance_weight(G))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None

In [ ]:
dist_weight = make_distance_weight(G_live)
manual_cost, manual_path = dijkstra_manual(G_live, LIVE_SOURCE, LIVE_DEST, dist_weight)
nx_path = shortest_distance_route(G_live, LIVE_SOURCE, LIVE_DEST)
print("Manual Dijkstra :", " -> ".join(manual_path), f"({manual_cost:.0f} km)")
print("NetworkX        :", " -> ".join(nx_path))
assert manual_path == nx_path, "manual and NetworkX Dijkstra disagree"
print("Both implementations agree.")

## 13. Implement weather-aware route optimisation
Now the edge weight is **distance + weather penalty**, and any lane that breaks a navigational or weather constraint is hidden. Dijkstra then finds the lowest-cost **feasible** route.

**A\*** is the same idea plus a *heuristic* (straight-line distance to the destination) that steers the search. Because the straight-line distance never overestimates the real cost, A\* still finds the optimum. Both must give the same route.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Weather-aware cost, Dijkstra / A* and route summaries

# ---------------------------------------------------------------------------
# Weather-aware optimisation (Dijkstra or A*) and route summaries
# ---------------------------------------------------------------------------

def make_weather_weight(G: nx.DiGraph, config: RoutingConfig):
    """Edge cost = distance_weight*km + weather_weight*penalty.
    Edges that break navigational OR weather constraints are hidden (None)."""
    def weight(u, v, d):
        if not d.get("nav_feasible", True):
            return None
        w = d.get("weather")
        if w is None:
            return None                      # never assume unknown = safe
        if not w["weather_feasible"]:
            return None
        return config.distance_weight * d["distance_km"] + config.weather_weight * w["weather_penalty"]
    return weight


def weather_aware_route(G: nx.DiGraph, source: str, target: str, config: RoutingConfig):
    weight = make_weather_weight(G, config)
    try:
        if config.algorithm == "astar":
            t = G.nodes[target]
            # Admissible heuristic: straight-line distance never overestimates
            heuristic = lambda n, _t: config.distance_weight * haversine_km(
                G.nodes[n]["lat"], G.nodes[n]["lon"], t["lat"], t["lon"])
            return nx.astar_path(G, source, target, heuristic=heuristic, weight=weight)
        return nx.dijkstra_path(G, source, target, weight=weight)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def format_duration(hours: float) -> str:
    total_min = int(round(hours * 60))
    d, rem = divmod(total_min, 24 * 60)
    h, m = divmod(rem, 60)
    return (f"{d} d " if d else "") + f"{h} h {m:02d} min"


def _r(x, n=2):
    return None if x is None else round(float(x), n)


def summarize_route(G: nx.DiGraph, path: Optional[List[str]], vessel: VesselProfile,
                    config: RoutingConfig, departure: datetime) -> Optional[dict]:
    """Totals, risk counts, travel time and per-segment details for a path."""
    if not path:
        return None
    segs, km, penalty, hours, cost = [], 0.0, 0.0, 0.0, 0.0
    for u, v in zip(path[:-1], path[1:]):
        d = G.edges[u, v]
        w = d["weather"]
        speed = vessel.service_speed_kn * (1 - config.speed_loss_fraction[w["risk_level"]])
        t_h = d["distance_nm"] / speed
        km += d["distance_km"]; penalty += w["weather_penalty"]; hours += t_h
        cost += config.distance_weight * d["distance_km"] + config.weather_weight * w["weather_penalty"]
        segs.append({
            "from": u, "to": v, "from_name": G.nodes[u]["name"], "to_name": G.nodes[v]["name"],
            "lane_type": d["lane_type"], "distance_km": _r(d["distance_km"], 1),
            "distance_nm": _r(d["distance_nm"], 1), "heading_deg": _r(d["bearing_deg"], 0),
            "risk_level": w["risk_level"], "risk_label": w["risk_label"],
            "severity_index": _r(w["severity_index"], 2), "weather_penalty": _r(w["weather_penalty"], 1),
            "wave_height_m": _r(w["wave_height_m"]), "max_wave_height_on_segment_m": _r(w["max_wave_height_on_segment_m"]),
            "wave_direction_deg": _r(w["wave_direction_deg"], 0),
            "relative_wave_angle_deg": _r(w["relative_wave_angle_deg"], 0), "encounter": w["encounter"],
            "effective_wave_height_m": _r(w["effective_wave_height_m"]),
            "wave_period_s": _r(w["wave_period_s"], 1), "swell_wave_height_m": _r(w["swell_wave_height_m"]),
            "wind_speed_kn": _r(w["wind_speed_kn"], 1), "max_wind_on_segment_kn": _r(w["max_wind_on_segment_kn"], 1),
            "wind_gusts_kn": _r(w["wind_gusts_kn"], 1), "wind_direction_deg": _r(w["wind_direction_deg"], 0),
            "data_complete": w["data_complete"], "data_sources": w["data_sources"],
            "data_errors": w["data_errors"], "nav_feasible": d["nav_feasible"],
            "weather_feasible": w["weather_feasible"], "violations": w["violations"],
            "planned_speed_kn": _r(speed, 1), "segment_time_h": _r(t_h, 2),
        })
    worst = max(s["risk_level"] for s in segs)
    return {
        "path": path, "path_names": [G.nodes[n]["name"] for n in path],
        "distance_km": _r(km, 1), "distance_nautical_miles": _r(km_to_nm(km), 1),
        "total_weather_penalty": _r(penalty, 1), "total_cost": _r(cost, 1),
        "weather_risk": RISK_LABELS[worst], "weather_risk_level": worst,
        "low_risk_segments": sum(s["risk_level"] == 0 for s in segs),
        "moderate_risk_segments": sum(s["risk_level"] == 1 for s in segs),
        "high_risk_segments": sum(s["risk_level"] == 2 for s in segs),
        "severe_risk_segments": sum(s["risk_level"] == 3 for s in segs),
        "segments_with_missing_weather": sum(not s["data_complete"] for s in segs),
        "weather_data_complete": all(s["data_complete"] for s in segs),
        "meets_constraints": all(s["nav_feasible"] and s["weather_feasible"] for s in segs),
        "constraint_violations": [f"{s['from']}->{s['to']}: {v}" for s in segs for v in s["violations"]],
        "estimated_travel_time": {"hours": _r(hours, 1), "text": format_duration(hours),
                                  "eta_utc": (departure + timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                                  "basis": f"service speed {vessel.service_speed_kn} kn reduced by "
                                           f"prototype weather speed-loss factors"},
        "segments": segs,
    }

In [ ]:
wa_dijkstra = weather_aware_route(G_live, LIVE_SOURCE, LIVE_DEST, ROUTING_CONFIG)
wa_astar = weather_aware_route(G_live, LIVE_SOURCE, LIVE_DEST, replace(ROUTING_CONFIG, algorithm="astar"))
print("Weather-aware (Dijkstra):", wa_dijkstra)
print("Weather-aware (A*)      :", wa_astar)
if wa_dijkstra:
    s = summarize_route(G_live, wa_dijkstra, vessel_demo, ROUTING_CONFIG, DEPARTURE)
    print(f"Distance {s['distance_km']} km / {s['distance_nautical_miles']} nm | "
          f"risk {s['weather_risk']} | time {s['estimated_travel_time']['text']}")
else:
    print("No feasible weather-aware route under the current constraints.")

## 14. Compare routes
`plan_route()` puts everything together: validate the request → fetch weather → score lanes → shortest-distance route → weather-aware route → comparison and explanation. It **never crashes** on bad input or API failures; it returns a `status`:

| status | meaning |
|---|---|
| `ok` | a feasible weather-aware route was found |
| `no_feasible_route` | ports are connected, but every connection breaks a safety constraint – **no route is recommended** |
| `no_route` | the ports are not connected in the network for this vessel |
| `error` | invalid request (unknown port, bad vessel type, …) |

Below it runs with **real weather** for Mumbai → Chennai.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Comparison table, explanation and plan_route()

# ---------------------------------------------------------------------------
# Route comparison, explanation and the main plan_route() entry point
# ---------------------------------------------------------------------------

DISCLAIMER = ("Prototype decision-support output for an academic project. Risk thresholds and "
              "vessel limits are configurable prototype values, not certified maritime safety "
              "limits. A low route cost does not mean the route is safe. Verify with official "
              "forecasts, nautical charts and professional maritime guidance.")


def resolve_port(G: nx.DiGraph, query) -> Optional[str]:
    """Accept a port id (e.g. 'INNSA') or a name / unique part of a name ('mumbai')."""
    if not isinstance(query, str) or not query.strip():
        return None
    q = query.strip().lower()
    ports = [n for n, d in G.nodes(data=True) if d.get("type") == "port"]
    for n in ports:
        if n.lower() == q or G.nodes[n]["name"].lower() == q:
            return n
    hits = [n for n in ports if q in G.nodes[n]["name"].lower()]
    return hits[0] if len(hits) == 1 else None


def list_ports(G):
    return {n: d["name"] for n, d in G.nodes(data=True) if d.get("type") == "port"}


def comparison_table(shortest: Optional[dict], recommended: Optional[dict]) -> pd.DataFrame:
    rows = [("Path", lambda r: " -> ".join(r["path"])),
            ("Distance (km)", lambda r: r["distance_km"]),
            ("Distance (nautical miles)", lambda r: r["distance_nautical_miles"]),
            ("Estimated travel time", lambda r: r["estimated_travel_time"]["text"]),
            ("ETA (UTC)", lambda r: r["estimated_travel_time"]["eta_utc"]),
            ("Total weather penalty", lambda r: r["total_weather_penalty"]),
            ("Total route cost", lambda r: r["total_cost"]),
            ("Overall weather risk", lambda r: r["weather_risk"]),
            ("High-risk segments", lambda r: r["high_risk_segments"]),
            ("Severe-risk segments", lambda r: r["severe_risk_segments"]),
            ("Segments with missing weather", lambda r: r["segments_with_missing_weather"]),
            ("Meets all constraints", lambda r: r["meets_constraints"])]
    data = {"Metric": [n for n, _ in rows],
            "Shortest-distance route": [f(shortest) if shortest else "no route" for _, f in rows],
            "Weather-aware route": [f(recommended) if recommended else "NO FEASIBLE ROUTE" for _, f in rows]}
    return pd.DataFrame(data).set_index("Metric")


def explain_routes(shortest: Optional[dict], recommended: Optional[dict], G=None) -> str:
    if recommended is None:
        if shortest is None:
            return "No navigable connection exists between these ports in the network."
        v = shortest["constraint_violations"][:4]
        return ("NO FEASIBLE ROUTE FOUND. Every connection between the ports passes through at least "
                "one segment that violates the configured safety constraints, so no route is "
                "recommended. The shortest-distance route is shown only for reference and must NOT be "
                "used. Examples of blocked segments: " + "; ".join(v) +
                ". Options: delay departure, use a vessel with higher limits, or review constraints "
                "with maritime professionals.")
    parts = []
    if shortest and shortest["path"] == recommended["path"]:
        parts.append(f"The shortest-distance route ({recommended['distance_km']} km) is also the "
                     f"weather-aware recommendation: its weather risk is {recommended['weather_risk']}, "
                     "so no detour is justified.")
    elif shortest:
        extra = recommended["distance_km"] - shortest["distance_km"]
        pct = 100 * extra / shortest["distance_km"] if shortest["distance_km"] else 0
        reason = []
        if not shortest["meets_constraints"]:
            reason.append("the shortest route violates safety constraints (" +
                          "; ".join(shortest["constraint_violations"][:3]) + ")")
        if shortest["high_risk_segments"] or shortest["severe_risk_segments"]:
            reason.append(f"the shortest route has {shortest['high_risk_segments']} high-risk and "
                          f"{shortest['severe_risk_segments']} severe-risk segment(s)")
        if not reason:
            reason.append("the shortest route has a higher combined distance + weather cost")
        parts.append(f"The weather-aware route was selected because {' and '.join(reason)}. It is "
                     f"{extra:.0f} km ({pct:.1f}%) longer, but its overall risk is "
                     f"{recommended['weather_risk']} and its weather penalty is "
                     f"{recommended['total_weather_penalty']} versus {shortest['total_weather_penalty']} "
                     f"(total cost {recommended['total_cost']} vs {shortest['total_cost']}).")
    if recommended["high_risk_segments"] or recommended["severe_risk_segments"]:
        parts.append(f"Caution: the recommended route still contains {recommended['high_risk_segments']} "
                     "high-risk segment(s); it was the lowest-cost route that meets the constraints.")
    if not recommended["weather_data_complete"]:
        parts.append(f"WARNING: weather data was missing for {recommended['segments_with_missing_weather']} "
                     "segment(s); those were treated conservatively but their risk is NOT verified.")
    return " ".join(parts)


def plan_route(G_base: nx.DiGraph, request: dict, provider: WeatherProvider,
               config: Optional[RoutingConfig] = None) -> dict:
    """
    Main entry point. Never raises for bad input or weather failures:
    returns {"status": "ok" | "no_feasible_route" | "no_route" | "error", ...}.
    """
    config = config or RoutingConfig()
    base = {"status": "error", "request": request, "disclaimer": DISCLAIMER,
            "data_notice": G_base.graph.get("data_notice", ""), "warnings": []}
    # 1. validate the request
    try:
        src = resolve_port(G_base, request.get("source_port"))
        dst = resolve_port(G_base, request.get("destination_port"))
        if src is None or dst is None:
            bad = request.get("source_port") if src is None else request.get("destination_port")
            raise ValueError(f"Unknown port '{bad}'. Valid ports: {list_ports(G_base)}")
        if src == dst:
            raise ValueError("source_port and destination_port must be different")
        vessel = get_vessel_profile(request.get("vessel_type", "container_ship"),
                                    request.get("vessel_speed"), request.get("vessel_specs"))
        dep = request.get("departure_time")
        departure = to_utc(dep) if dep else datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    except (ValueError, TypeError) as exc:
        base["error"] = str(exc)
        return base

    # 2. weather for every segment (work on a copy so requests never interfere)
    G = G_base.copy()
    window = auto_window_hours(G, src, dst, vessel.service_speed_kn, config)
    weather, edge_points = fetch_network_weather(G, provider, departure, window, config.sample_spacing_km)
    annotate_graph_with_weather(G, weather, edge_points, vessel, config)
    n_missing = sum(1 for r in weather.values() if r.get("wave_height_m") is None or r.get("wind_speed_kn") is None)
    if n_missing:
        base["warnings"].append(f"Weather data missing for {n_missing} of {len(weather)} sample points "
                                f"(policy: {config.missing_data_policy}).")

    # 3. routes
    short_path = shortest_distance_route(G, src, dst)
    rec_path = weather_aware_route(G, src, dst, config) if short_path else None
    shortest = summarize_route(G, short_path, vessel, config, departure)
    recommended = summarize_route(G, rec_path, vessel, config, departure)

    status = "ok" if recommended else ("no_feasible_route" if shortest else "no_route")
    base.update({
        "status": status,
        "source_port": {"id": src, "name": G.nodes[src]["name"]},
        "destination_port": {"id": dst, "name": G.nodes[dst]["name"]},
        "vessel": asdict(vessel),
        "departure_time_utc": departure.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "weather_window_hours": window,
        "weather_source": sorted({str(r.get("source")) for r in weather.values()}),
        "algorithm": config.algorithm,
        "recommended_route": recommended,
        "alternative_route": shortest,
        "alternative_route_role": "shortest-distance route (ignores weather; for comparison only)",
        "distance_km": recommended["distance_km"] if recommended else None,
        "distance_nautical_miles": recommended["distance_nautical_miles"] if recommended else None,
        "estimated_travel_time": recommended["estimated_travel_time"] if recommended else None,
        "weather_risk": recommended["weather_risk"] if recommended else "No feasible route",
        "segment_weather_details": recommended["segments"] if recommended else
                                   (shortest["segments"] if shortest else []),
        "route_explanation": explain_routes(shortest, recommended, G),
    })
    if status == "no_route":
        base["route_explanation"] = ("No navigable connection exists between "
                                     f"{src} and {dst} for this vessel (graph is disconnected or "
                                     "every link fails depth/navigability checks).")
    if any(str(s).startswith("synthetic") for s in base["weather_source"]):
        base["warnings"].append("Weather data is SYNTHETIC test data, not a real forecast.")
    base["_graph"] = G   # annotated graph for visualisation (removed by to_json_safe)
    return base

In [ ]:
LIVE_REQUEST = {
    "source_port": "INNSA", "destination_port": "INMAA",
    "vessel_speed": 15, "vessel_type": "container_ship",
    "vessel_specs": {"draft_m": 12.5},
    "departure_time": DEPARTURE.strftime("%Y-%m-%dT%H:%M:%SZ"),
}
live_result = plan_route(G, LIVE_REQUEST, live_provider, ROUTING_CONFIG)
print("Status:", live_result["status"], "| weather source:", live_result.get("weather_source"))
for w in live_result["warnings"]:
    print("WARNING:", w)
if live_result["status"] != "error":
    live_table = comparison_table(live_result["alternative_route"], live_result["recommended_route"])
    display(live_table)
    print("\nExplanation:\n", live_result["route_explanation"])
    display(pd.DataFrame(live_result["segment_weather_details"])[
        ["from", "to", "distance_km", "risk_label", "wave_height_m", "encounter", "wind_speed_kn",
         "wind_gusts_kn", "weather_penalty", "planned_speed_kn", "data_complete"]])

## 15. Visualise routes
* **Static map (Matplotlib):** lanes coloured by risk, the weather-aware route as a wide blue band, the shortest-distance route as a dashed dark line, source/destination circled. There is no coastline on this schematic.
* **Risk profile:** the severity of each segment along the recommended route.
* **Interactive map (Folium):** real basemap, toggleable layers, a legend and hover tooltips with the weather of each lane.

All maps are labelled as **demonstrations, not navigational charts**.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# Plotting helpers (Matplotlib + Folium)

RISK_COLORS = {0: "#0aac23", 1: "#fbb315", 2: "#f18a59", 3: "#dc3c3d"}
RISK_NAMES = {0: "Low", 1: "Moderate", 2: "High", 3: "Severe"}
MISSING_COLOR = "#9c9c9c"
RECOMMENDED_COLOR = "#257de0"
SHORTEST_COLOR = "#474747"
INK = "#0b0b0b"
GRID = "#e7e7e7"


def _undirected_edges(G):
    """One entry per sea lane with the WORSE of its two directions."""
    seen = {}
    for u, v, d in G.edges(data=True):
        key = tuple(sorted((u, v)))
        w = d.get("weather") or {}
        level = w.get("risk_level", 0)
        missing = not w.get("data_complete", True)
        blocked = not (d.get("nav_feasible", True) and w.get("weather_feasible", True))
        if key not in seen or level > seen[key]["level"]:
            seen[key] = {"u": key[0], "v": key[1], "level": level, "missing": missing,
                         "blocked": blocked, "data": d}
        else:
            seen[key]["missing"] |= missing
            seen[key]["blocked"] |= blocked
    return list(seen.values())


def _title(result):
    s, t = result.get("source_port", {}), result.get("destination_port", {})
    return f"{s.get('name', '?')} to {t.get('name', '?')}  |  status: {result.get('status')}"


def plot_route_map(result: dict, save_path: Optional[str] = None, title: Optional[str] = None, show=True):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    G = result["_graph"]
    fig, ax = plt.subplots(figsize=(10, 10))
    # risk-coloured lanes
    for e in _undirected_edges(G):
        a, b = G.nodes[e["u"]], G.nodes[e["v"]]
        color = MISSING_COLOR if e["missing"] and e["level"] <= 1 else RISK_COLORS[e["level"]]
        ax.plot([a["lon"], b["lon"]], [a["lat"], b["lat"]], color=color, lw=2,
                ls=":" if e["blocked"] else "-", zorder=2, solid_capstyle="round")
    # routes
    rec, alt = result.get("recommended_route"), result.get("alternative_route")
    if rec:
        xs = [G.nodes[n]["lon"] for n in rec["path"]]; ys = [G.nodes[n]["lat"] for n in rec["path"]]
        ax.plot(xs, ys, color=RECOMMENDED_COLOR, lw=9, alpha=0.35, zorder=1, solid_capstyle="round")
    if alt:
        xs = [G.nodes[n]["lon"] for n in alt["path"]]; ys = [G.nodes[n]["lat"] for n in alt["path"]]
        ax.plot(xs, ys, color=SHORTEST_COLOR, lw=1.6, ls=(0, (6, 3)), zorder=3)
    # nodes
    for n, d in G.nodes(data=True):
        if d["type"] == "port":
            ax.scatter(d["lon"], d["lat"], s=70, marker="s", color=INK, zorder=4,
                       edgecolor="white", linewidth=1.5)
            ax.annotate(d["name"], (d["lon"], d["lat"]), xytext=(6, 4), textcoords="offset points",
                        fontsize=8, color=INK)
        else:
            ax.scatter(d["lon"], d["lat"], s=14, color="#9c9c9c", zorder=4)
    src, dst = result.get("source_port", {}).get("id"), result.get("destination_port", {}).get("id")
    for n, lab in ((src, "SOURCE"), (dst, "DESTINATION")):
        if n in G:
            ax.scatter(G.nodes[n]["lon"], G.nodes[n]["lat"], s=220, facecolor="none",
                       edgecolor=RECOMMENDED_COLOR, linewidth=2.5, zorder=5)
            ax.annotate(lab, (G.nodes[n]["lon"], G.nodes[n]["lat"]), xytext=(6, -12),
                        textcoords="offset points", fontsize=8, fontweight="bold", color=INK)
    handles = [Line2D([], [], color=RISK_COLORS[i], lw=2, label=f"Lane risk: {RISK_NAMES[i]}") for i in range(4)]
    handles += [Line2D([], [], color=MISSING_COLOR, lw=2, label="Lane: weather data missing"),
                Line2D([], [], color=INK, lw=2, ls=":", label="Lane blocked (constraints)"),
                Line2D([], [], color=RECOMMENDED_COLOR, lw=8, alpha=0.35, label="Weather-aware route"),
                Line2D([], [], color=SHORTEST_COLOR, lw=1.6, ls=(0, (6, 3)), label="Shortest-distance route"),
                Line2D([], [], marker="s", color=INK, ls="", label="Port"),
                Line2D([], [], marker="o", color="#9c9c9c", ls="", markersize=4, label="Waypoint")]
    ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=True)
    mean_lat = sum(d["lat"] for _, d in G.nodes(data=True)) / G.number_of_nodes()
    ax.set_aspect(1 / math.cos(math.radians(mean_lat)))
    ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel("Longitude (deg E)"); ax.set_ylabel("Latitude (deg N)")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_title((title or _title(result)) + "\nDEMONSTRATION ONLY - schematic network, no coastline, "
                 "not a navigational chart", fontsize=10, color=INK, loc="left")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def plot_risk_profile(result: dict, save_path: Optional[str] = None, show=True):
    """Bar per segment of the recommended route (or the reference route if none)."""
    import matplotlib.pyplot as plt
    route = result.get("recommended_route") or result.get("alternative_route")
    if not route:
        print("No route to plot.")
        return None
    segs = route["segments"]
    labels = [f"{s['from']}\n->{s['to']}" for s in segs]
    vals = [s["severity_index"] if s["severity_index"] is not None else 0 for s in segs]
    colors = [RISK_COLORS[s["risk_level"]] if s["data_complete"] else MISSING_COLOR for s in segs]
    fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(segs)), 4.5))
    ax.bar(range(len(segs)), vals, color=colors, width=0.7, edgecolor="white", linewidth=2)
    ax.axhline(1.0, color=INK, lw=1, ls="--")
    ax.text(len(segs) - 0.5, 1.02, "Severe threshold", ha="right", va="bottom", fontsize=8, color=INK)
    for i, s in enumerate(segs):
        ax.text(i, vals[i] + 0.02, s["risk_label"] if s["data_complete"] else "missing",
                ha="center", va="bottom", fontsize=7, color=INK)
    ax.set_xticks(range(len(segs))); ax.set_xticklabels(labels, fontsize=7, rotation=90)
    ax.set_ylabel("Severity index (1.0 = Severe threshold)")
    which = "recommended" if result.get("recommended_route") else "reference (NOT recommended)"
    ax.set_title(f"Weather severity along the {which} route\n{_title(result)}", fontsize=10, loc="left")
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def _segment_tooltip(G, u, v):
    d = G.edges[u, v]; w = d.get("weather", {})
    fmt = lambda x, unit="": "n/a" if x is None else f"{x:.1f}{unit}"
    return (f"<b>{G.nodes[u]['name']} &rarr; {G.nodes[v]['name']}</b><br>"
            f"Risk: <b>{w.get('risk_label', '?')}</b>{'' if w.get('data_complete', True) else ' (data missing)'}<br>"
            f"Distance: {d['distance_km']:.0f} km / {d['distance_nm']:.0f} nm<br>"
            f"Wave height: {fmt(w.get('wave_height_m'), ' m')} ({w.get('encounter', '?')})<br>"
            f"Wind: {fmt(w.get('wind_speed_kn'), ' kn')}, gusts {fmt(w.get('wind_gusts_kn'), ' kn')}<br>"
            f"Depth (illustrative): {d['min_depth_m']:.1f} m<br>"
            f"Source: {', '.join(w.get('data_sources', []))}")


def make_folium_map(result: dict, save_path: Optional[str] = None):
    """Interactive map. Returns the folium.Map, or None if folium is not installed."""
    try:
        import folium
    except ImportError:
        print("folium is not installed - skipping the interactive map (pip install folium).")
        return None
    G = result["_graph"]
    lats = [d["lat"] for _, d in G.nodes(data=True)]; lons = [d["lon"] for _, d in G.nodes(data=True)]
    m = folium.Map(location=[sum(lats) / len(lats), sum(lons) / len(lons)], zoom_start=5,
                   tiles="CartoDB positron", control_scale=True)
    lanes = folium.FeatureGroup(name="Sea lanes coloured by weather risk", show=True)
    for e in _undirected_edges(G):
        a, b = G.nodes[e["u"]], G.nodes[e["v"]]
        color = MISSING_COLOR if e["missing"] and e["level"] <= 1 else RISK_COLORS[e["level"]]
        folium.PolyLine([(a["lat"], a["lon"]), (b["lat"], b["lon"])], color=color, weight=3,
                        opacity=0.9, dash_array="4 6" if e["blocked"] else None,
                        tooltip=folium.Tooltip(_segment_tooltip(G, e["u"], e["v"]))).add_to(lanes)
    lanes.add_to(m)
    for key, label, color, weight, dash in (("alternative_route", "Shortest-distance route", SHORTEST_COLOR, 3, "10 6"),
                                            ("recommended_route", "Weather-aware recommended route", RECOMMENDED_COLOR, 7, None)):
        route = result.get(key)
        if not route:
            continue
        fg = folium.FeatureGroup(name=label, show=True)
        for s in route["segments"]:
            a, b = G.nodes[s["from"]], G.nodes[s["to"]]
            folium.PolyLine([(a["lat"], a["lon"]), (b["lat"], b["lon"])], color=color, weight=weight,
                            opacity=0.55 if key == "recommended_route" else 0.9, dash_array=dash,
                            tooltip=folium.Tooltip(f"<b>{label}</b><br>" + _segment_tooltip(G, s["from"], s["to"]))
                            ).add_to(fg)
        fg.add_to(m)
    nodes = folium.FeatureGroup(name="Ports and waypoints", show=True)
    src, dst = result.get("source_port", {}).get("id"), result.get("destination_port", {}).get("id")
    for n, d in G.nodes(data=True):
        if n in (src, dst):
            folium.Marker((d["lat"], d["lon"]), tooltip=f"{'SOURCE' if n == src else 'DESTINATION'}: {d['name']}",
                          icon=folium.Icon(color="blue" if n == src else "red",
                                           icon="play" if n == src else "flag")).add_to(nodes)
        elif d["type"] == "port":
            folium.CircleMarker((d["lat"], d["lon"]), radius=6, color=INK, fill=True, fill_opacity=1,
                                tooltip=f"Port: {d['name']} ({n})").add_to(nodes)
        else:
            folium.CircleMarker((d["lat"], d["lon"]), radius=3, color="#7a7a7a", fill=True,
                                tooltip=f"Waypoint (synthetic): {d['name']}").add_to(nodes)
    nodes.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    rows = "".join(f'<div><span style="display:inline-block;width:22px;height:4px;background:{RISK_COLORS[i]};'
                   f'vertical-align:middle;margin-right:6px"></span>{RISK_NAMES[i]} risk lane</div>' for i in range(4))
    legend = f"""
    <div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 12px;
                border:1px solid rgba(11,11,11,0.15);border-radius:6px;font:12px sans-serif;color:{INK}">
      <b>Legend</b>{rows}
      <div><span style="display:inline-block;width:22px;height:4px;background:{MISSING_COLOR};vertical-align:middle;margin-right:6px"></span>Weather data missing</div>
      <div><span style="display:inline-block;width:22px;border-top:3px dashed {INK};vertical-align:middle;margin-right:6px"></span>Blocked lane (constraints)</div>
      <div><span style="display:inline-block;width:22px;height:7px;background:{RECOMMENDED_COLOR};opacity:.55;vertical-align:middle;margin-right:6px"></span>Weather-aware route</div>
      <div><span style="display:inline-block;width:22px;border-top:3px dashed {SHORTEST_COLOR};vertical-align:middle;margin-right:6px"></span>Shortest-distance route</div>
    </div>"""
    banner = f"""
    <div style="position:fixed;top:12px;left:60px;right:220px;z-index:9999;background:#fff8e1;padding:8px 12px;
                border:1px solid #fbb315;border-radius:6px;font:12px sans-serif;color:{INK}">
      <b>DEMONSTRATION MAP - NOT A NAVIGATIONAL CHART.</b> Synthetic waypoint network.
      {_title(result)} | weather source: {', '.join(result.get('weather_source', []))}
      | Weather data by Open-Meteo.com (CC-BY 4.0) when the source is open-meteo.
    </div>"""
    m.get_root().html.add_child(folium.Element(legend + banner))
    if save_path:
        m.save(save_path)
    return m

In [ ]:
if live_result["status"] != "error":
    plot_route_map(live_result, save_path=os.path.join(DIRS["figures"], "live_route_map.png"))
    plot_risk_profile(live_result, save_path=os.path.join(DIRS["figures"], "live_risk_profile.png"))
    live_map = make_folium_map(live_result, save_path=os.path.join(DIRS["maps"], "live_route_map.html"))
    if live_map is not None:
        display(live_map)

## 16. Run test scenarios
All scenarios use **synthetic, reproducible weather** (tagged `synthetic:<name>`) so the results are the same every time.

| Scenario | Weather | Expected behaviour |
|---|---|---|
| 1 – Normal weather | calm everywhere | both routes identical |
| 2 – Bad weather on the shortest route | High-risk storm off Mangalore | detour west of Lakshadweep via the Nine Degree Channel |
| 2b – Moderate weather | Moderate conditions off Mangalore | detour **not** worth it – stay on the short route (shows the balance) |
| 3 – Severe weather | cyclone south of Sri Lanka | container ship: **no feasible route** reported |
| 3b – Severe weather, small vessel | same cyclone | the shallow Pamban/Palk passage is usable by a small craft, so a route exists |

Then the robustness tests: invalid port, same port, unknown vessel type, invalid JSON, missing weather data, weather API failure, disconnected graph, and algorithm consistency.

In [ ]:
SCENARIO_DEPARTURE = "2026-12-01T00:00:00Z"   # fixed for reproducibility
STORM_MANGALORE_HIGH = StormZone("Synthetic storm off Mangalore", 12.9, 74.6, 180, peak_wave_m=4.2, peak_wind_kn=32)
STORM_MANGALORE_MOD = StormZone("Synthetic moderate weather off Mangalore", 12.9, 74.6, 180, peak_wave_m=2.6, peak_wind_kn=20)
CYCLONE_SRI_LANKA = StormZone("Synthetic cyclone south of Sri Lanka", 5.9, 80.6, 300, peak_wave_m=7.5, peak_wind_kn=60)

def req(src, dst, vessel="container_ship"):
    return {"source_port": src, "destination_port": dst, "vessel_type": vessel,
            "departure_time": SCENARIO_DEPARTURE}

SCENARIOS = [
    {"id": "S1_normal_weather", "provider": SyntheticWeatherProvider("normal_weather"),
     "request": req("INNSA", "INMAA"), "expect": "same_route"},
    {"id": "S2_bad_weather_on_shortest", "provider": SyntheticWeatherProvider("storm_mangalore", storms=[STORM_MANGALORE_HIGH]),
     "request": req("INNSA", "INCOK"), "expect": "detour"},
    {"id": "S2b_moderate_weather", "provider": SyntheticWeatherProvider("moderate_mangalore", storms=[STORM_MANGALORE_MOD]),
     "request": req("INNSA", "INCOK"), "expect": "same_route"},
    {"id": "S3_severe_weather", "provider": SyntheticWeatherProvider("cyclone_sri_lanka", storms=[CYCLONE_SRI_LANKA]),
     "request": req("INNSA", "INMAA"), "expect": "no_feasible_route"},
    {"id": "S3b_severe_weather_small_vessel", "provider": SyntheticWeatherProvider("cyclone_sri_lanka", storms=[CYCLONE_SRI_LANKA]),
     "request": req("INNSA", "INMAA", "small_coastal_vessel"), "expect": "ok"},
]

scenario_results, scenario_rows = {}, []
for sc in SCENARIOS:
    res = plan_route(G, sc["request"], sc["provider"], ROUTING_CONFIG)
    scenario_results[sc["id"]] = res
    rec, alt = res["recommended_route"], res["alternative_route"]
    if sc["expect"] == "same_route":
        passed = res["status"] == "ok" and rec["path"] == alt["path"]
    elif sc["expect"] == "detour":
        passed = res["status"] == "ok" and rec["path"] != alt["path"] and rec["distance_km"] > alt["distance_km"]
    elif sc["expect"] == "no_feasible_route":
        passed = res["status"] == "no_feasible_route" and rec is None
    else:
        passed = res["status"] == sc["expect"]
    scenario_rows.append({
        "scenario": sc["id"], "status": res["status"], "expected": sc["expect"], "PASS": passed,
        "shortest_km": alt["distance_km"] if alt else None, "shortest_risk": alt["weather_risk"] if alt else None,
        "recommended_km": rec["distance_km"] if rec else None, "recommended_risk": rec["weather_risk"] if rec else None,
        "high_segments (rec)": rec["high_risk_segments"] if rec else None,
        "severe_segments (rec)": rec["severe_risk_segments"] if rec else None,
        "travel_time (rec)": rec["estimated_travel_time"]["text"] if rec else None,
    })
    print(f"\n=== {sc['id']}  ->  {res['status']}  ({'PASS' if passed else 'FAIL'})")
    print(res["route_explanation"])
    plot_route_map(res, save_path=os.path.join(DIRS["figures"], f"{sc['id']}_map.png"), show=False)
    plot_risk_profile(res, save_path=os.path.join(DIRS["figures"], f"{sc['id']}_risk_profile.png"), show=False)
    make_folium_map(res, save_path=os.path.join(DIRS["maps"], f"{sc['id']}_map.html"))

scenario_summary = pd.DataFrame(scenario_rows)
scenario_summary.to_csv(os.path.join(DIRS["tables"], "scenario_summary.csv"), index=False)
display(scenario_summary)
assert scenario_summary["PASS"].all(), "A scenario did not behave as expected"
print("All 5 scenarios behaved as expected.")

In [ ]:
# Look at Scenario 2 in detail: the comparison table and both maps
s2 = scenario_results["S2_bad_weather_on_shortest"]
display(comparison_table(s2["alternative_route"], s2["recommended_route"]))
plot_route_map(s2)
plot_risk_profile(s2)
m2 = make_folium_map(s2)
if m2 is not None:
    display(m2)

In [ ]:
# ---------- Robustness tests ----------
calm = SyntheticWeatherProvider("normal_weather")
test_rows = []

def record_test(name, passed, detail):
    test_rows.append({"test": name, "PASS": bool(passed), "detail": detail})
    print(("PASS " if passed else "FAIL ") + name + " - " + detail)

# 1. invalid source / destination
r = plan_route(G, req("Atlantis", "INMAA"), calm, ROUTING_CONFIG)
record_test("invalid source port", r["status"] == "error", r.get("error", "")[:90])
r = plan_route(G, req("INNSA", "Nowhere"), calm, ROUTING_CONFIG)
record_test("invalid destination port", r["status"] == "error", r.get("error", "")[:90])
r = plan_route(G, req("INNSA", "Mumbai (JNPT / Nhava Sheva)"), calm, ROUTING_CONFIG)
record_test("same source and destination", r["status"] == "error", r.get("error", ""))
r = plan_route(G, req("INNSA", "INMAA", "submarine"), calm, ROUTING_CONFIG)
record_test("unknown vessel type", r["status"] == "error", r.get("error", "")[:90])

# 2. missing weather data (synthetic hole off Mangalore)
holey = SyntheticWeatherProvider("missing_data", missing_zones=[MissingZone(12.9, 74.3, 150)])
r = plan_route(G, req("INNSA", "INCOK"), holey, ROUTING_CONFIG)
rec = r["recommended_route"]
record_test("missing weather -> conservative",
            r["status"] == "ok" and rec["segments_with_missing_weather"] > 0 and not rec["weather_data_complete"],
            f"{rec['segments_with_missing_weather']} segment(s) flagged, risk {rec['weather_risk']}")
r = plan_route(G, req("INNSA", "INCOK"), holey, replace(ROUTING_CONFIG, missing_data_policy="block"))
record_test("missing weather -> block policy avoids gap",
            r["status"] == "ok" and r["recommended_route"]["weather_data_complete"],
            " -> ".join(r["recommended_route"]["path"]))

# 3. weather API failure (unreachable host, no retries) - must not crash
broken = OpenMeteoProvider(WeatherConfig(marine_url="https://invalid.invalid/v1/marine",
                                         forecast_url="https://invalid.invalid/v1/forecast",
                                         max_retries=0, timeout_s=5, cache_file=None))
r = plan_route(G, req("INNSA", "INCOK"), broken, ROUTING_CONFIG)
record_test("weather API failure handled",
            r["status"] == "ok" and not r["recommended_route"]["weather_data_complete"] and r["warnings"],
            r["warnings"][0] if r["warnings"] else "")

class CrashingProvider(WeatherProvider):
    name = "crashing"
    def fetch_points(self, points, start_time, window_hours):
        raise RuntimeError("simulated provider crash")
r = plan_route(G, req("INNSA", "INCOK"), CrashingProvider(), ROUTING_CONFIG)
record_test("provider exception handled", r["status"] == "ok" and not r["recommended_route"]["weather_data_complete"],
            "route returned with weather marked unverified")

# 3b. Open-Meteo parsing with a MOCKED response in the documented format (no network needed):
#     first call returns HTTP 429 (rate limit) to test the retry, then JSON lists like the real API.
class FakeResponse:
    def __init__(self, status, payload, headers=None):
        self.status_code, self._payload, self.headers = status, payload, headers or {}
        self.text = json.dumps(payload)
    def json(self):
        return self._payload

class FakeSession:
    def __init__(self):
        self.calls, self.headers = 0, {}
    def get(self, url, params=None, timeout=None):
        self.calls += 1
        if self.calls == 1:
            return FakeResponse(429, {"error": True, "reason": "Too many requests"}, {"Retry-After": "0"})
        n = len(params["latitude"].split(","))
        start = datetime.strptime(params["start_hour"], "%Y-%m-%dT%H:%M")
        times = [(start + timedelta(hours=h)).strftime("%Y-%m-%dT%H:%M") for h in range(4)]
        out = []
        for i in range(n):
            if "marine" in url:   # point 2 is a cell with no wave data (all null)
                hourly = {"time": times, "wave_height": [1.0, 2.5, None, 1.8] if i == 0 else [None] * 4,
                          "wave_direction": [200, 210, 220, 230], "wave_period": [6, 7, 7, 6],
                          "swell_wave_height": [0.8, 1.2, 1.1, 1.0]}
            else:
                hourly = {"time": times, "wind_speed_10m": [10, 22, 18, 12],
                          "wind_direction_10m": [240, 250, 260, 270], "wind_gusts_10m": [15, 30, 25, 17]}
            out.append({"latitude": 0.0, "longitude": 0.0, "hourly": hourly})
        return FakeResponse(200, out if n > 1 else out[0])

fake_session = FakeSession()
fake = OpenMeteoProvider(WeatherConfig(backoff_s=0.01, min_interval_s=0, cache_file=None), session=fake_session)
recs = fake.fetch_points([(18.9, 72.4), (9.95, 75.8)], DEPARTURE, window_hours=3)
r0, r1 = recs[point_key(18.9, 72.4)], recs[point_key(9.95, 75.8)]
passed = (r0["wave_height_m"] == 2.5 and r0["wave_direction_deg"] == 210 and r0["wind_speed_kn"] == 22
          and r0["wind_gusts_kn"] == 30 and r1["wave_height_m"] is None and bool(r1["error"])
          and fake_session.calls == 3)
record_test("Open-Meteo parsing + 429 retry (mocked)", passed,
            f"max wave {r0['wave_height_m']} m, max wind {r0['wind_speed_kn']} kn; null cell -> '{r1['error']}'")

# 4. disconnected graph
G_disc = G.copy()
G_disc.add_node("XXISO", name="Isolated Demo Port", type="port", lat=20.0, lon=65.0)
r = plan_route(G_disc, req("INNSA", "XXISO"), calm, ROUTING_CONFIG)
record_test("disconnected graph", r["status"] == "no_route", r["route_explanation"][:90])

# 5. algorithm consistency: manual Dijkstra == NetworkX Dijkstra == A*
ok = True
for sc in SCENARIOS:
    res = scenario_results[sc["id"]]
    Ga = res["_graph"]; s_, t_ = res["source_port"]["id"], res["destination_port"]["id"]
    _, p_manual = dijkstra_manual(Ga, s_, t_, make_weather_weight(Ga, ROUTING_CONFIG))
    p_nx = weather_aware_route(Ga, s_, t_, ROUTING_CONFIG)
    p_astar = weather_aware_route(Ga, s_, t_, replace(ROUTING_CONFIG, algorithm="astar"))
    ok &= (p_manual == p_nx == p_astar)
record_test("manual Dijkstra == NetworkX == A*", ok, f"checked {len(SCENARIOS)} scenarios")

tests_df = pd.DataFrame(test_rows)
tests_df.to_csv(os.path.join(DIRS["tables"], "robustness_tests.csv"), index=False)
display(tests_df)
assert tests_df["PASS"].all(), "Some robustness tests failed"
print("All robustness tests passed.")

## 17. Prepare structured JSON output
`handle_route_request()` is what a FastAPI endpoint will call later: JSON in → JSON out. The response contains `recommended_route`, `alternative_route`, `distance_km`, `distance_nautical_miles`, `estimated_travel_time`, `weather_risk`, `segment_weather_details`, `route_explanation`, plus `status`, `warnings`, `data_notice` and a `disclaimer`.

We save a **sample input and output** (reproducible, Scenario 2 synthetic weather) and the **live** output.

In [ ]:
# ---- Same code as in the module file (see Section 18) ----
# JSON request/response layer

# ---------------------------------------------------------------------------
# JSON request/response layer (what a FastAPI endpoint will call later)
# ---------------------------------------------------------------------------

def to_json_safe(result: dict) -> dict:
    """Drop internal objects (the graph) and make everything JSON-serialisable."""
    clean = {k: v for k, v in result.items() if not k.startswith("_")}
    return json.loads(json.dumps(clean, default=str))


def handle_route_request(request_json, G: nx.DiGraph, provider: WeatherProvider,
                         config: Optional[RoutingConfig] = None) -> str:
    """Accepts a JSON string or dict, returns a JSON string (backend-friendly)."""
    try:
        request = json.loads(request_json) if isinstance(request_json, str) else dict(request_json)
    except (json.JSONDecodeError, TypeError, ValueError) as exc:
        return json.dumps({"status": "error", "error": f"invalid JSON request: {exc}"})
    required = ["source_port", "destination_port"]
    missing = [k for k in required if not request.get(k)]
    if missing:
        return json.dumps({"status": "error", "error": f"missing required fields: {missing}"})
    return json.dumps(to_json_safe(plan_route(G, request, provider, config)), indent=2)

In [ ]:
sample_request = {
    "source_port": "INNSA",
    "destination_port": "INCOK",
    "vessel_speed": 15,
    "vessel_type": "container_ship",
    "vessel_specs": {"draft_m": 12.5, "max_wave_height_m": 6.0, "max_wind_speed_kn": 45},
    "departure_time": SCENARIO_DEPARTURE,
}
sample_response_json = handle_route_request(json.dumps(sample_request), G,
                                            SyntheticWeatherProvider("storm_mangalore", storms=[STORM_MANGALORE_HIGH]),
                                            ROUTING_CONFIG)
with open(os.path.join(DIRS["json"], "sample_request.json"), "w") as fh:
    json.dump(sample_request, fh, indent=2)
with open(os.path.join(DIRS["json"], "sample_response_scenario2.json"), "w") as fh:
    fh.write(sample_response_json)

live_response = to_json_safe(live_result)
with open(os.path.join(DIRS["json"], "live_response.json"), "w") as fh:
    json.dump(live_response, fh, indent=2)
for sc_id, res in scenario_results.items():
    with open(os.path.join(DIRS["json"], f"{sc_id}_response.json"), "w") as fh:
        json.dump(to_json_safe(res), fh, indent=2)

resp = json.loads(sample_response_json)
print("Top-level keys:", list(resp.keys()))
short = {k: resp[k] for k in ["status", "distance_km", "distance_nautical_miles", "estimated_travel_time",
                              "weather_risk", "route_explanation"]}
short["recommended_route.path"] = resp["recommended_route"]["path"]
short["alternative_route.path"] = resp["alternative_route"]["path"]
short["segment_weather_details[0]"] = resp["segment_weather_details"][0]
print(json.dumps(short, indent=2))

# Error handling of the JSON layer
print(handle_route_request("{not valid json", G, calm))
print(handle_route_request({"source_port": "INNSA"}, G, calm))

## 18. Save Python source files
Each cell below writes one file of the module to Google Drive using `%%writefile`. The code is **identical** to the code you ran above, just organised into files so the backend can `import` it later:

* `weather_service.py` – weather providers
* `routing.py` – graph, risk scoring, routing, JSON layer
* `visualization.py` – maps and charts
* `sample_network.py` – the synthetic demo network
* `requirements.txt`, `README.md`, `fastapi_app_example.py`

After writing, we import the saved files and check they give exactly the same result as the notebook.

In [ ]:
%%writefile {PROJECT_DIR}/weather_service.py
"""
weather_service.py
==================
Weather data providers for the Weather-Aware Ship Routing module.

Part of: Supply Chain and Coastal Chain Intelligence Platform
Module : ml/ship_routing

What this file does
-------------------
* Defines a small "provider" interface so the routing code does not care
  WHERE weather data comes from.
* OpenMeteoProvider  -> REAL forecast data from the free Open-Meteo APIs
    - Marine API  : https://marine-api.open-meteo.com/v1/marine
                    hourly: wave_height, wave_direction, wave_period,
                            swell_wave_height
    - Forecast API: https://api.open-meteo.com/v1/forecast
                    hourly: wind_speed_10m, wind_direction_10m, wind_gusts_10m
  Docs: https://open-meteo.com/en/docs/marine-weather-api
        https://open-meteo.com/en/docs
  No API key is needed for non-commercial use. Free-tier limits published by
  Open-Meteo (Nov 2026): 600 calls/minute, 5,000/hour, <10,000/day.
  Data licence: CC-BY 4.0 - attribute "Weather data by Open-Meteo.com".
  Open-Meteo itself warns that marine accuracy in coastal areas is limited
  and that the data is NOT suitable for coastal navigation.
* SyntheticWeatherProvider -> REPRODUCIBLE, MADE-UP weather for testing
  (calm sea + configurable "storm zones"). Never real data.

Every value returned is tagged with a "source" field so real and synthetic
data can never be confused.

To add another provider later (e.g. Stormglass, NOAA, a paid service),
subclass WeatherProvider and implement fetch_points().
"""

from __future__ import annotations

import json
import logging
import math
import os
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Sequence, Tuple

import requests

logger = logging.getLogger("ship_routing.weather")

# >>> NB-CELL: weather_core
# ---------------------------------------------------------------------------
# Weather configuration and common helpers
# ---------------------------------------------------------------------------

# The weather fields every provider must return for a point.
# None means "not available" - it is NEVER silently replaced by 0.
WEATHER_FIELDS = [
    "wave_height_m",        # significant height of mean waves (m)
    "wave_direction_deg",   # direction waves come FROM (0 = from north)
    "wave_period_s",        # mean wave period (s)
    "swell_wave_height_m",  # primary swell height (m)
    "wind_speed_kn",        # 10 m wind speed (knots)
    "wind_direction_deg",   # direction wind comes FROM (0 = from north)
    "wind_gusts_kn",        # 10 m wind gusts (knots)
]


@dataclass
class WeatherConfig:
    """All weather-provider settings in one place (easy to save as JSON)."""
    provider: str = "open-meteo"                       # "open-meteo" or "synthetic"
    marine_url: str = "https://marine-api.open-meteo.com/v1/marine"
    forecast_url: str = "https://api.open-meteo.com/v1/forecast"
    marine_hourly: Tuple[str, ...] = ("wave_height", "wave_direction",
                                      "wave_period", "swell_wave_height")
    wind_hourly: Tuple[str, ...] = ("wind_speed_10m", "wind_direction_10m",
                                    "wind_gusts_10m")
    marine_max_forecast_days: int = 7      # Marine API allows up to 8; keep a margin
    wind_max_forecast_days: int = 15       # Forecast API allows up to 16
    batch_size: int = 50                   # coordinates per HTTP request
    timeout_s: float = 20.0
    max_retries: int = 2                   # retries for 429 / 5xx / network errors
    backoff_s: float = 2.0                 # first back-off delay (doubles each retry)
    min_interval_s: float = 0.3            # polite pause between requests
    cache_ttl_hours: float = 3.0           # re-use cached forecasts this long
    cache_file: Optional[str] = None       # JSON file for the disk cache (optional)
    coord_decimals: int = 2                # rounding for cache keys (~1 km)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["marine_hourly"] = list(self.marine_hourly)
        d["wind_hourly"] = list(self.wind_hourly)
        return d


def point_key(lat: float, lon: float, decimals: int = 2) -> str:
    """Stable text key for a coordinate, e.g. '15.40,73.30'."""
    return f"{round(lat, decimals):.{decimals}f},{round(lon, decimals):.{decimals}f}"


def empty_point_weather(source: str, error: Optional[str] = None) -> dict:
    """A weather record where every field is missing."""
    rec = {f: None for f in WEATHER_FIELDS}
    rec.update({"source": source, "time_of_max_wave": None,
                "time_of_max_wind": None, "error": error})
    return rec


def to_utc(dt) -> datetime:
    """Accept datetime or ISO string; return a timezone-aware UTC datetime."""
    if isinstance(dt, str):
        dt = datetime.fromisoformat(dt.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _clean(value):
    """Convert API numbers to float; keep None / NaN as None."""
    if value is None:
        return None
    try:
        v = float(value)
    except (TypeError, ValueError):
        return None
    return None if math.isnan(v) else v


class WeatherProvider:
    """Base class. Subclasses return one weather record per point."""
    name = "base"

    def fetch_points(self, points: Sequence[Tuple[float, float]],
                     start_time, window_hours: int) -> Dict[str, dict]:
        """
        points       : list of (lat, lon)
        start_time   : departure time (datetime or ISO string, UTC assumed)
        window_hours : planning window; the WORST value inside the window
                       is returned (a conservative choice for a prototype)
        Returns {point_key: record} where record has the WEATHER_FIELDS keys,
        a 'source' tag and an 'error' message (None if all good).
        """
        raise NotImplementedError


# >>> NB-CELL: weather_openmeteo
# ---------------------------------------------------------------------------
# Real data: Open-Meteo Marine + Forecast APIs
# ---------------------------------------------------------------------------

class OpenMeteoProvider(WeatherProvider):
    """
    Fetches REAL forecasts from Open-Meteo (no API key needed).

    * Batches many coordinates into one request (comma-separated lat/lon;
      the API then returns a JSON list in the same order).
    * Retries on HTTP 429 (rate limit), 5xx and network errors with back-off.
    * Handles the documented error format {"error": true, "reason": "..."}.
    * Caches results in memory and (optionally) in a JSON file.
    * Never crashes the caller: failures become records with None values
      and an 'error' message.
    """
    name = "open-meteo"

    def __init__(self, config: Optional[WeatherConfig] = None,
                 session: Optional[requests.Session] = None):
        self.config = config or WeatherConfig()
        self.session = session or requests.Session()
        self.session.headers.update({"User-Agent": "coastal-platform-ship-routing/1.0 (academic prototype)"})
        self._cache: Dict[str, dict] = {}
        self._last_request = 0.0
        self.stats = {"http_requests": 0, "cache_hits": 0, "failures": 0}
        self._load_disk_cache()

    # ----- caching ---------------------------------------------------------
    def _load_disk_cache(self):
        path = self.config.cache_file
        if path and os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as fh:
                    self._cache = json.load(fh)
                logger.info("Loaded %d cached weather records", len(self._cache))
            except (OSError, json.JSONDecodeError) as exc:
                logger.warning("Weather cache unreadable (%s) - starting empty", exc)
                self._cache = {}

    def save_cache(self):
        path = self.config.cache_file
        if not path:
            return
        try:
            os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
            with open(path, "w", encoding="utf-8") as fh:
                json.dump(self._cache, fh)
        except OSError as exc:
            logger.warning("Could not save weather cache: %s", exc)

    def _cache_get(self, key: str) -> Optional[dict]:
        entry = self._cache.get(key)
        if not entry:
            return None
        age_h = (time.time() - entry.get("fetched_at", 0)) / 3600.0
        if age_h > self.config.cache_ttl_hours:
            return None
        self.stats["cache_hits"] += 1
        return entry["data"]

    def _cache_put(self, key: str, data: dict):
        self._cache[key] = {"fetched_at": time.time(), "data": data}

    # ----- HTTP ------------------------------------------------------------
    def _get_json(self, url: str, params: dict):
        """GET with rate-limit pause, retries and error handling.
        Returns (json_or_None, error_message_or_None)."""
        delay = self.config.backoff_s
        last_error = None
        for attempt in range(self.config.max_retries + 1):
            wait = self.config.min_interval_s - (time.time() - self._last_request)
            if wait > 0:
                time.sleep(wait)
            self._last_request = time.time()
            try:
                self.stats["http_requests"] += 1
                resp = self.session.get(url, params=params, timeout=self.config.timeout_s)
            except requests.RequestException as exc:
                last_error = f"network error ({exc.__class__.__name__})"
                logger.debug("Weather request exception: %s", exc)
            else:
                if resp.status_code == 200:
                    try:
                        data = resp.json()
                    except ValueError:
                        return None, "invalid JSON in response"
                    if isinstance(data, dict) and data.get("error"):
                        return None, f"API error: {data.get('reason')}"
                    return data, None
                if resp.status_code == 400:
                    # Documented format: {"error": true, "reason": "..."}; not retryable
                    try:
                        reason = resp.json().get("reason")
                    except ValueError:
                        reason = resp.text[:200]
                    return None, f"HTTP 400: {reason}"
                last_error = f"HTTP {resp.status_code}"
                if resp.status_code == 429:
                    retry_after = resp.headers.get("Retry-After")
                    if retry_after and retry_after.isdigit():
                        delay = max(delay, float(retry_after))
                    last_error = "HTTP 429: rate limit reached"
                elif resp.status_code < 500:
                    return None, last_error        # other 4xx: do not retry
            if attempt < self.config.max_retries:
                logger.warning("Weather request failed (%s) - retry %d in %.1fs",
                               last_error, attempt + 1, delay)
                time.sleep(delay)
                delay *= 2
        self.stats["failures"] += 1
        return None, last_error

    @staticmethod
    def _window_max(hourly: dict, value_field: str, dir_field: Optional[str],
                    t0: datetime, t1: datetime):
        """Largest value of `value_field` within [t0, t1] and the matching direction."""
        times = hourly.get("time") or []
        values = hourly.get(value_field) or []
        dirs = (hourly.get(dir_field) or []) if dir_field else []
        best, best_dir, best_time = None, None, None
        for i, ts in enumerate(times):
            try:
                t = datetime.fromisoformat(ts).replace(tzinfo=timezone.utc)
            except (TypeError, ValueError):
                continue
            if t < t0 or t > t1 or i >= len(values):
                continue
            v = _clean(values[i])
            if v is None:
                continue
            if best is None or v > best:
                best, best_time = v, ts
                best_dir = _clean(dirs[i]) if i < len(dirs) else None
        return best, best_dir, best_time

    def _request_batch(self, url: str, batch: List[Tuple[float, float]],
                       hourly_vars: Sequence[str], t0: datetime, t1: datetime,
                       extra: dict):
        params = {
            "latitude": ",".join(f"{p[0]:.4f}" for p in batch),
            "longitude": ",".join(f"{p[1]:.4f}" for p in batch),
            "hourly": ",".join(hourly_vars),
            "timezone": "GMT",
            "start_hour": t0.strftime("%Y-%m-%dT%H:00"),
            "end_hour": t1.strftime("%Y-%m-%dT%H:00"),
            **extra,
        }
        data, err = self._get_json(url, params)
        if err and err.startswith("HTTP 400"):
            # Fallback: ask for a plain forecast range and filter locally
            logger.warning("start_hour/end_hour rejected (%s); retrying with forecast_days", err)
            params.pop("start_hour"); params.pop("end_hour")
            params["forecast_days"] = 7
            data, err = self._get_json(url, params)
        if data is None:
            return None, err
        if isinstance(data, dict):            # single location -> single object
            data = [data]
        if len(data) != len(batch):
            return None, f"unexpected response length {len(data)} for {len(batch)} points"
        return data, None

    def fetch_points(self, points, start_time, window_hours):
        cfg = self.config
        t0 = to_utc(start_time).replace(minute=0, second=0, microsecond=0)
        t1 = t0 + timedelta(hours=max(1, int(window_hours)))
        now = datetime.now(timezone.utc)
        marine_end = now + timedelta(days=cfg.marine_max_forecast_days)
        wind_end = now + timedelta(days=cfg.wind_max_forecast_days)

        results: Dict[str, dict] = {}
        todo: List[Tuple[float, float]] = []
        window_tag = f"{t0:%Y%m%d%H}-{t1:%Y%m%d%H}"
        for lat, lon in points:
            key = point_key(lat, lon, cfg.coord_decimals)
            if key in results:
                continue
            cached = self._cache_get(f"{key}|{window_tag}")
            if cached is not None:
                results[key] = cached
            else:
                results[key] = empty_point_weather(self.name)
                todo.append((lat, lon))

        if todo and t0 > marine_end:
            msg = "departure is beyond the marine forecast horizon"
            for lat, lon in todo:
                results[point_key(lat, lon, cfg.coord_decimals)]["error"] = msg
            return results

        m_t1 = min(t1, marine_end)
        w_t1 = min(t1, wind_end)
        for i in range(0, len(todo), cfg.batch_size):
            batch = todo[i:i + cfg.batch_size]
            marine, m_err = self._request_batch(cfg.marine_url, batch, cfg.marine_hourly,
                                                t0, m_t1, {"cell_selection": "sea"})
            wind, w_err = self._request_batch(cfg.forecast_url, batch, cfg.wind_hourly,
                                              t0, w_t1, {"wind_speed_unit": "kn"})
            if marine is None and wind is None and i + cfg.batch_size < len(todo):
                # Both APIs unreachable: stop early instead of retrying every batch
                logger.error("Weather APIs unavailable (%s / %s) - remaining points marked missing",
                             m_err, w_err)
                for lat, lon in todo[i:]:
                    results[point_key(lat, lon, cfg.coord_decimals)]["error"] = \
                        f"marine: {m_err}; wind: {w_err}"
                break
            for j, (lat, lon) in enumerate(batch):
                key = point_key(lat, lon, cfg.coord_decimals)
                rec = results[key]
                errors = []
                if marine is not None:
                    h = marine[j].get("hourly", {})
                    wh, wd, wt = self._window_max(h, "wave_height", "wave_direction", t0, m_t1)
                    wp, _, _ = self._window_max(h, "wave_period", None, t0, m_t1)
                    sw, _, _ = self._window_max(h, "swell_wave_height", None, t0, m_t1)
                    rec.update({"wave_height_m": wh, "wave_direction_deg": wd,
                                "wave_period_s": wp, "swell_wave_height_m": sw,
                                "time_of_max_wave": wt})
                    if wh is None:
                        errors.append("no wave data for this cell/time")
                else:
                    errors.append(f"marine: {m_err}")
                if wind is not None:
                    h = wind[j].get("hourly", {})
                    ws, wdir, wt = self._window_max(h, "wind_speed_10m", "wind_direction_10m", t0, w_t1)
                    wg, _, _ = self._window_max(h, "wind_gusts_10m", None, t0, w_t1)
                    rec.update({"wind_speed_kn": ws, "wind_direction_deg": wdir,
                                "wind_gusts_kn": wg, "time_of_max_wind": wt})
                    if ws is None:
                        errors.append("no wind data for this cell/time")
                else:
                    errors.append(f"wind: {w_err}")
                rec["error"] = "; ".join(errors) if errors else None
                # Only cache complete successes so failures are retried next time
                if marine is not None and wind is not None:
                    self._cache_put(f"{key}|{window_tag}", rec)
        self.save_cache()
        return results


# >>> NB-CELL: weather_synthetic
# ---------------------------------------------------------------------------
# Synthetic data: reproducible test weather (NOT REAL)
# ---------------------------------------------------------------------------

def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


@dataclass
class StormZone:
    """A circular area of bad weather used ONLY for synthetic test scenarios.
    Full strength inside core_fraction*radius, fading linearly to calm at radius."""
    name: str
    lat: float
    lon: float
    radius_km: float
    peak_wave_m: float
    peak_wind_kn: float
    wave_direction_deg: float = 225.0
    wind_direction_deg: float = 225.0
    core_fraction: float = 0.5


@dataclass
class MissingZone:
    """Area where the synthetic provider pretends no data is available."""
    lat: float
    lon: float
    radius_km: float


class SyntheticWeatherProvider(WeatherProvider):
    """Deterministic made-up weather for reproducible tests. Tagged 'synthetic'."""
    name = "synthetic"

    def __init__(self, scenario_name: str = "calm",
                 base_wave_m: float = 1.0, base_wind_kn: float = 10.0,
                 base_wave_direction_deg: float = 240.0,
                 storms: Optional[List[StormZone]] = None,
                 missing_zones: Optional[List[MissingZone]] = None,
                 noise_fraction: float = 0.05, seed: int = 42):
        self.scenario_name = scenario_name
        self.base_wave_m = base_wave_m
        self.base_wind_kn = base_wind_kn
        self.base_dir = base_wave_direction_deg
        self.storms = storms or []
        self.missing_zones = missing_zones or []
        self.noise_fraction = noise_fraction
        self.seed = seed

    def _noise(self, lat, lon):
        # Deterministic pseudo-random number in [-1, 1] from coordinates + seed
        x = math.sin(lat * 12.9898 + lon * 78.233 + self.seed) * 43758.5453
        return (x - math.floor(x)) * 2 - 1

    def fetch_points(self, points, start_time, window_hours):
        out = {}
        for lat, lon in points:
            key = point_key(lat, lon)
            if any(_haversine_km(lat, lon, z.lat, z.lon) <= z.radius_km for z in self.missing_zones):
                out[key] = empty_point_weather(f"synthetic:{self.scenario_name}",
                                               error="synthetic missing-data zone")
                continue
            wave, wind = self.base_wave_m, self.base_wind_kn
            wave_dir, wind_dir = self.base_dir, self.base_dir
            for s in self.storms:
                d = _haversine_km(lat, lon, s.lat, s.lon)
                if d >= s.radius_km:
                    continue
                core = s.core_fraction * s.radius_km
                f = 1.0 if d <= core else 1.0 - (d - core) / (s.radius_km - core)
                w = self.base_wave_m + (s.peak_wave_m - self.base_wave_m) * f
                v = self.base_wind_kn + (s.peak_wind_kn - self.base_wind_kn) * f
                if w > wave:
                    wave, wave_dir = w, s.wave_direction_deg
                if v > wind:
                    wind, wind_dir = v, s.wind_direction_deg
            n = 1 + self.noise_fraction * self._noise(lat, lon)
            out[key] = {
                "wave_height_m": round(wave * n, 2),
                "wave_direction_deg": wave_dir,
                "wave_period_s": round(4 + 1.6 * wave, 1),
                "swell_wave_height_m": round(0.6 * wave * n, 2),
                "wind_speed_kn": round(wind * n, 1),
                "wind_direction_deg": wind_dir,
                "wind_gusts_kn": round(wind * 1.3 * n, 1),
                "source": f"synthetic:{self.scenario_name}",
                "time_of_max_wave": None, "time_of_max_wind": None, "error": None,
            }
        return out


def create_weather_provider(config: Optional[WeatherConfig] = None, **synthetic_kwargs) -> WeatherProvider:
    """Factory: choose the provider by name so it stays configurable."""
    config = config or WeatherConfig()
    if config.provider == "open-meteo":
        return OpenMeteoProvider(config)
    if config.provider == "synthetic":
        return SyntheticWeatherProvider(**synthetic_kwargs)
    raise ValueError(f"Unknown weather provider '{config.provider}'. "
                     "Use 'open-meteo' or 'synthetic', or add a new WeatherProvider subclass.")

In [ ]:
%%writefile {PROJECT_DIR}/routing.py
"""
routing.py
==========
Weather-aware ship routing on a maritime waypoint graph.

Part of: Supply Chain and Coastal Chain Intelligence Platform
Module : ml/ship_routing   (independent of ml/port_congestion)

Main entry point
----------------
    plan_route(G, request, provider, config) -> dict  (JSON-serialisable)

The request is the same structure the future backend will send:
    {"source_port": "INNSA", "destination_port": "INMAA",
     "vessel_speed": 15, "vessel_type": "container_ship",
     "vessel_specs": {"draft_m": 12.5}, "departure_time": "2026-11-28T06:00:00Z"}

IMPORTANT SAFETY NOTE
---------------------
All thresholds, vessel limits and penalties in this file are PROTOTYPE
values for an academic project. They are NOT certified maritime safety
limits. Real use needs vessel-specific data (stability booklet, class
rules, master's judgement) and professional maritime guidance. A low route
cost does NOT mean a route is safe.
"""

from __future__ import annotations

import copy
import heapq
import json
import logging
import math
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime, timedelta, timezone
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import networkx as nx
import pandas as pd

from weather_service import (WeatherProvider, empty_point_weather, point_key, to_utc)

logger = logging.getLogger("ship_routing.routing")

# >>> NB-CELL: routing_config
# ---------------------------------------------------------------------------
# Configuration: risk thresholds, penalties, weights and vessel profiles
# (ALL VALUES ARE PROTOTYPE VALUES - NOT CERTIFIED SAFETY LIMITS)
# ---------------------------------------------------------------------------

RISK_LABELS = ["Low", "Moderate", "High", "Severe"]   # index = risk level 0..3


@dataclass
class RoutingConfig:
    # Thresholds are the lower bounds of Moderate, High and Severe.
    # Wave height (m): Low < 2.0 <= Moderate < 3.5 <= High < 5.0 <= Severe
    wave_thresholds_m: Tuple[float, float, float] = (2.0, 3.5, 5.0)
    # Sustained wind (knots), roughly Beaufort 5 / 7 / 9 boundaries
    wind_thresholds_kn: Tuple[float, float, float] = (17.0, 28.0, 41.0)
    # Gusts (knots)
    gust_thresholds_kn: Tuple[float, float, float] = (25.0, 40.0, 55.0)

    # Wave direction relative to the ship's heading (0 deg = waves from dead
    # ahead, 90 = on the beam, 180 = from astern). The wave height is
    # multiplied by this factor BEFORE classification (prototype heuristic:
    # beam seas cause heavy rolling, head seas cause slamming).
    head_sea_max_angle: float = 45.0
    following_sea_min_angle: float = 135.0
    head_sea_factor: float = 1.10
    beam_sea_factor: float = 1.20
    following_sea_factor: float = 1.00

    # Edge cost = distance_weight * km + weather_weight * weather_penalty
    # weather_penalty = km * penalty_multiplier[risk level]
    distance_weight: float = 1.0
    weather_weight: float = 1.0
    penalty_multipliers: Tuple[float, float, float, float] = (0.0, 0.3, 1.5, 4.0)

    # Hard constraints
    block_severe_segments: bool = True     # Severe segments become unavailable
    under_keel_clearance_m: float = 1.5    # draft + clearance must be <= depth

    # Missing weather data: "conservative" (assume at least Moderate + extra
    # penalty), "block" (segment unavailable) or "ignore" (use what exists)
    missing_data_policy: str = "conservative"
    missing_data_min_level: int = 1
    missing_data_penalty_multiplier: float = 0.2

    # Speed loss in bad weather (fraction of service speed) per risk level
    speed_loss_fraction: Tuple[float, float, float, float] = (0.0, 0.05, 0.15, 0.30)

    # Weather sampling along each segment
    sample_spacing_km: float = 100.0
    # Weather planning window (hours from departure). None = automatic
    weather_window_hours: Optional[int] = None
    max_weather_window_hours: int = 168

    algorithm: str = "dijkstra"            # "dijkstra" or "astar"

    def to_dict(self) -> dict:
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(self).items()}

    @classmethod
    def from_dict(cls, d: dict) -> "RoutingConfig":
        fields_ = {f for f in cls.__dataclass_fields__}
        clean = {k: (tuple(v) if isinstance(v, list) else v) for k, v in d.items() if k in fields_}
        return cls(**clean)


@dataclass
class VesselProfile:
    vessel_type: str
    service_speed_kn: float
    draft_m: float
    max_wave_height_m: float        # hard limit on significant wave height
    max_wind_speed_kn: float        # hard limit on sustained wind
    wave_thresholds_m: Optional[Tuple[float, float, float]] = None  # overrides config
    notes: str = "prototype values - replace with real vessel data"


# Illustrative presets only. Replace with data from the Shipping Company module.
VESSEL_PRESETS: Dict[str, VesselProfile] = {
    "container_ship": VesselProfile("container_ship", 16.0, 12.5, 6.0, 45.0),
    "bulk_carrier":   VesselProfile("bulk_carrier",   13.0, 11.5, 5.5, 40.0),
    "tanker":         VesselProfile("tanker",         14.0, 13.0, 6.0, 45.0),
    "general_cargo":  VesselProfile("general_cargo",  12.0,  8.0, 4.5, 38.0),
    "small_coastal_vessel": VesselProfile("small_coastal_vessel", 10.0, 2.5, 3.5, 34.0,
                                          wave_thresholds_m=(1.25, 2.0, 3.0)),
}


def get_vessel_profile(vessel_type: str = "container_ship",
                       vessel_speed: Optional[float] = None,
                       vessel_specs: Optional[dict] = None) -> VesselProfile:
    """Start from a preset and apply speed/spec overrides from the request."""
    key = (vessel_type or "container_ship").strip().lower()
    if key not in VESSEL_PRESETS:
        raise ValueError(f"Unknown vessel_type '{vessel_type}'. "
                         f"Known types: {sorted(VESSEL_PRESETS)}")
    vessel = replace(VESSEL_PRESETS[key])
    if vessel_speed is not None:
        vessel.service_speed_kn = float(vessel_speed)
    allowed = {"draft_m", "max_wave_height_m", "max_wind_speed_kn"}
    for k, v in (vessel_specs or {}).items():
        if k in allowed and v is not None:
            setattr(vessel, k, float(v))
        elif k == "wave_thresholds_m" and v:
            vessel.wave_thresholds_m = tuple(float(x) for x in v)
    if not (0 < vessel.service_speed_kn <= 40):
        raise ValueError("vessel_speed must be between 0 and 40 knots")
    if vessel.draft_m <= 0:
        raise ValueError("draft_m must be positive")
    return vessel


# >>> NB-CELL: graph_build
# ---------------------------------------------------------------------------
# Building the maritime graph (nodes = ports/waypoints, edges = sea lanes)
# ---------------------------------------------------------------------------

REQUIRED_NODE_COLS = {"node_id", "name", "type", "lat", "lon"}
REQUIRED_EDGE_COLS = {"u", "v", "lane_type", "min_depth_m", "navigable"}


def validate_network_tables(waypoints: pd.DataFrame, edges: pd.DataFrame) -> None:
    """Raise ValueError with a clear message if the tables are malformed."""
    missing = REQUIRED_NODE_COLS - set(waypoints.columns)
    if missing:
        raise ValueError(f"waypoints table is missing columns: {sorted(missing)}")
    missing = REQUIRED_EDGE_COLS - set(edges.columns)
    if missing:
        raise ValueError(f"edges table is missing columns: {sorted(missing)}")
    dup = waypoints["node_id"][waypoints["node_id"].duplicated()].tolist()
    if dup:
        raise ValueError(f"duplicate node_id values: {dup}")
    bad = waypoints[(waypoints.lat.abs() > 90) | (waypoints.lon.abs() > 180)]
    if len(bad):
        raise ValueError(f"invalid coordinates for: {bad.node_id.tolist()}")
    ids = set(waypoints["node_id"])
    unknown = sorted((set(edges.u) | set(edges.v)) - ids)
    if unknown:
        raise ValueError(f"edges reference unknown nodes: {unknown}")
    loops = edges[edges.u == edges.v]
    if len(loops):
        raise ValueError(f"self-loop edges found: {loops.u.tolist()}")


def build_graph(waypoints: pd.DataFrame, edges: pd.DataFrame, data_notice: str = "") -> nx.DiGraph:
    """
    Create a DIRECTED graph. Each sea lane is added in both directions
    (unless bidirectional=False) because the ship's heading - and therefore
    the wave direction relative to the ship - is different each way.
    """
    validate_network_tables(waypoints, edges)
    G = nx.DiGraph(data_notice=data_notice)
    for row in waypoints.itertuples(index=False):
        G.add_node(row.node_id, name=row.name, type=row.type,
                   lat=float(row.lat), lon=float(row.lon),
                   data_status=getattr(row, "data_status", "unknown"))
    for row in edges.itertuples(index=False):
        attrs = dict(lane_type=row.lane_type, min_depth_m=float(row.min_depth_m),
                     navigable=bool(row.navigable),
                     data_status=getattr(row, "data_status", "unknown"))
        G.add_edge(row.u, row.v, **attrs)
        if bool(getattr(row, "bidirectional", True)):
            G.add_edge(row.v, row.u, **attrs)
    return G


# >>> NB-CELL: geo
# ---------------------------------------------------------------------------
# Geographic helpers (great-circle maths) and edge geometry
# ---------------------------------------------------------------------------

EARTH_RADIUS_KM = 6371.0088
KM_PER_NM = 1.852


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance between two points in kilometres."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_RADIUS_KM * math.asin(math.sqrt(a))


def km_to_nm(km: float) -> float:
    return km / KM_PER_NM


def initial_bearing_deg(lat1, lon1, lat2, lon2) -> float:
    """Compass direction (0-360, 0 = north) the ship steers at the start of the segment."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dl = math.radians(lon2 - lon1)
    x = math.sin(dl) * math.cos(p2)
    y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360


def intermediate_point(lat1, lon1, lat2, lon2, fraction: float) -> Tuple[float, float]:
    """Point at `fraction` (0..1) of the way along the great circle."""
    p1, l1, p2, l2 = map(math.radians, (lat1, lon1, lat2, lon2))
    d = 2 * math.asin(math.sqrt(math.sin((p2 - p1) / 2) ** 2 +
                                math.cos(p1) * math.cos(p2) * math.sin((l2 - l1) / 2) ** 2))
    if d == 0:
        return lat1, lon1
    a = math.sin((1 - fraction) * d) / math.sin(d)
    b = math.sin(fraction * d) / math.sin(d)
    x = a * math.cos(p1) * math.cos(l1) + b * math.cos(p2) * math.cos(l2)
    y = a * math.cos(p1) * math.sin(l1) + b * math.cos(p2) * math.sin(l2)
    z = a * math.sin(p1) + b * math.sin(p2)
    return math.degrees(math.atan2(z, math.hypot(x, y))), math.degrees(math.atan2(y, x))


def add_geometry(G: nx.DiGraph) -> nx.DiGraph:
    """Attach distance (km, nm) and heading to every edge."""
    for u, v, data in G.edges(data=True):
        a, b = G.nodes[u], G.nodes[v]
        km = haversine_km(a["lat"], a["lon"], b["lat"], b["lon"])
        data["distance_km"] = km
        data["distance_nm"] = km_to_nm(km)
        data["bearing_deg"] = initial_bearing_deg(a["lat"], a["lon"], b["lat"], b["lon"])
    return G


def load_network_from_dataframes(waypoints, edges, data_notice="") -> nx.DiGraph:
    return add_geometry(build_graph(waypoints, edges, data_notice))


def load_network_from_csv(waypoints_csv: str, edges_csv: str, data_notice: str = "") -> nx.DiGraph:
    """Load any network that uses the same column names (real data can be plugged in here)."""
    return load_network_from_dataframes(pd.read_csv(waypoints_csv), pd.read_csv(edges_csv), data_notice)


# >>> NB-CELL: segment_weather
# ---------------------------------------------------------------------------
# Sampling points along segments and fetching their weather
# ---------------------------------------------------------------------------

def sample_points_for_edge(G: nx.DiGraph, u: str, v: str, spacing_km: float = 100.0):
    """Both end points plus evenly spaced points every ~spacing_km (at least the midpoint)."""
    a, b = G.nodes[u], G.nodes[v]
    km = G.edges[u, v]["distance_km"]
    n = max(2, int(math.ceil(km / spacing_km)))
    return [intermediate_point(a["lat"], a["lon"], b["lat"], b["lon"], i / n) for i in range(n + 1)]


def collect_sample_points(G: nx.DiGraph, spacing_km: float = 100.0):
    """Unique sample points for the whole graph + mapping edge -> point keys."""
    unique: Dict[str, Tuple[float, float]] = {}
    edge_points: Dict[Tuple[str, str], List[str]] = {}
    for u, v in G.edges():
        keys = []
        for lat, lon in sample_points_for_edge(G, u, v, spacing_km):
            k = point_key(lat, lon)
            unique.setdefault(k, (round(lat, 2), round(lon, 2)))
            keys.append(k)
        edge_points[(u, v)] = keys
    return unique, edge_points


def auto_window_hours(G, source, target, speed_kn, config: "RoutingConfig") -> int:
    """Hours of forecast to consider: rough voyage time (great circle x1.6) + 12 h buffer."""
    if config.weather_window_hours:
        return int(min(config.weather_window_hours, config.max_weather_window_hours))
    a, b = G.nodes[source], G.nodes[target]
    nm = km_to_nm(haversine_km(a["lat"], a["lon"], b["lat"], b["lon"]))
    hours = int(math.ceil(nm * 1.6 / max(speed_kn, 1.0))) + 12
    return int(min(max(hours, 12), config.max_weather_window_hours))


def fetch_network_weather(G, provider: WeatherProvider, departure, window_hours: int,
                          spacing_km: float = 100.0):
    """Fetch weather for every sample point. Never raises: a provider crash
    becomes 'missing' records so routing can still decide conservatively."""
    unique, edge_points = collect_sample_points(G, spacing_km)
    try:
        weather = provider.fetch_points(list(unique.values()), departure, window_hours)
    except Exception as exc:          # noqa: BLE001 - protect the caller
        logger.error("Weather provider failed: %s", exc)
        weather = {}
        err = f"provider exception: {exc.__class__.__name__}: {exc}"
    else:
        err = "no record returned"
    for k in unique:
        if k not in weather:
            weather[k] = empty_point_weather(getattr(provider, "name", "unknown"), error=err)
    return weather, edge_points


# >>> NB-CELL: risk
# ---------------------------------------------------------------------------
# Weather risk scoring for points and segments
# ---------------------------------------------------------------------------

def classify_value(value: Optional[float], thresholds: Sequence[float]) -> Optional[int]:
    """0=Low, 1=Moderate, 2=High, 3=Severe. None if value missing."""
    if value is None:
        return None
    level = 0
    for t in thresholds:
        if value >= t:
            level += 1
    return level


def relative_wave_angle(wave_from_deg: Optional[float], heading_deg: float) -> Optional[float]:
    """0 = waves from dead ahead, 90 = beam, 180 = from astern."""
    if wave_from_deg is None:
        return None
    diff = abs((wave_from_deg - heading_deg + 180) % 360 - 180)
    return diff


def encounter_type(angle: Optional[float], config: RoutingConfig) -> Tuple[str, float]:
    if angle is None:
        return "unknown", config.beam_sea_factor       # unknown direction -> worst case factor
    if angle <= config.head_sea_max_angle:
        return "head sea", config.head_sea_factor
    if angle >= config.following_sea_min_angle:
        return "following sea", config.following_sea_factor
    return "beam sea", config.beam_sea_factor


def assess_point(rec: dict, heading_deg: float, vessel: VesselProfile, config: RoutingConfig) -> dict:
    """Risk assessment of one weather record for a ship on a given heading."""
    wave_thr = vessel.wave_thresholds_m or config.wave_thresholds_m
    hs = rec.get("wave_height_m")
    angle = relative_wave_angle(rec.get("wave_direction_deg"), heading_deg)
    enc, factor = encounter_type(angle, config)
    eff = hs * factor if hs is not None else None
    wind = rec.get("wind_speed_kn")
    gust = rec.get("wind_gusts_kn")
    levels = {"wave": classify_value(eff, wave_thr),
              "wind": classify_value(wind, config.wind_thresholds_kn),
              "gust": classify_value(gust, config.gust_thresholds_kn)}
    available = [l for l in levels.values() if l is not None]
    complete = hs is not None and wind is not None
    level = max(available) if available else 0
    ratios = []
    if eff is not None:
        ratios.append(eff / wave_thr[2])
    if wind is not None:
        ratios.append(wind / config.wind_thresholds_kn[2])
    if gust is not None:
        ratios.append(gust / config.gust_thresholds_kn[2])
    severity = max(ratios) if ratios else None

    reasons = []
    if hs is not None and hs > vessel.max_wave_height_m:
        reasons.append(f"wave height {hs:.1f} m exceeds vessel limit {vessel.max_wave_height_m} m")
    if wind is not None and wind > vessel.max_wind_speed_kn:
        reasons.append(f"wind {wind:.0f} kn exceeds vessel limit {vessel.max_wind_speed_kn} kn")
    if not complete:
        if config.missing_data_policy == "conservative":
            level = max(level, config.missing_data_min_level)
        elif config.missing_data_policy == "block":
            reasons.append("weather data missing (policy = block)")
    if level == 3 and config.block_severe_segments:
        reasons.append("Severe weather risk (severe segments are blocked)")
    return {"risk_level": level, "levels": levels, "severity_index": severity,
            "effective_wave_height_m": eff, "relative_wave_angle_deg": angle,
            "encounter": enc, "data_complete": complete, "violations": reasons,
            "record": rec}


def assess_segment(point_recs: List[dict], heading_deg: float, km: float,
                   vessel: VesselProfile, config: RoutingConfig) -> dict:
    """Segment risk = WORST sample point on the segment (conservative)."""
    pts = [assess_point(r, heading_deg, vessel, config) for r in point_recs]
    worst = max(pts, key=lambda p: (p["risk_level"], p["severity_index"] or 0))
    complete = all(p["data_complete"] for p in pts)
    # Report the violations of the worst point (avoids repeating similar messages)
    violations = worst["violations"] or sorted({v for p in pts for v in p["violations"]})
    level = max(p["risk_level"] for p in pts)
    penalty = km * config.penalty_multipliers[level]
    if not complete and config.missing_data_policy == "conservative":
        penalty += km * config.missing_data_penalty_multiplier
    rec = worst["record"]
    sources = sorted({str(p["record"].get("source")) for p in pts})
    errors = sorted({p["record"]["error"] for p in pts if p["record"].get("error")})
    return {
        "risk_level": level, "risk_label": RISK_LABELS[level],
        "severity_index": worst["severity_index"],
        "weather_penalty": penalty, "weather_feasible": not violations,
        "violations": violations, "data_complete": complete,
        "missing_points": sum(not p["data_complete"] for p in pts), "points_sampled": len(pts),
        "wave_height_m": rec.get("wave_height_m"), "wave_direction_deg": rec.get("wave_direction_deg"),
        "wave_period_s": rec.get("wave_period_s"), "swell_wave_height_m": rec.get("swell_wave_height_m"),
        "effective_wave_height_m": worst["effective_wave_height_m"],
        "relative_wave_angle_deg": worst["relative_wave_angle_deg"], "encounter": worst["encounter"],
        "wind_speed_kn": rec.get("wind_speed_kn"), "wind_direction_deg": rec.get("wind_direction_deg"),
        "wind_gusts_kn": rec.get("wind_gusts_kn"),
        "max_wave_height_on_segment_m": max((p["record"].get("wave_height_m") or 0) for p in pts) if any(
            p["record"].get("wave_height_m") is not None for p in pts) else None,
        "max_wind_on_segment_kn": max((p["record"].get("wind_speed_kn") or 0) for p in pts) if any(
            p["record"].get("wind_speed_kn") is not None for p in pts) else None,
        "data_sources": sources, "data_errors": errors[:3],
    }


def annotate_graph_with_weather(G: nx.DiGraph, weather: Dict[str, dict], edge_points,
                                vessel: VesselProfile, config: RoutingConfig) -> nx.DiGraph:
    """Store a weather assessment and navigational feasibility on every edge."""
    for u, v, data in G.edges(data=True):
        recs = [weather[k] for k in edge_points[(u, v)]]
        data["weather"] = assess_segment(recs, data["bearing_deg"], data["distance_km"], vessel, config)
        data["nav_feasible"], data["nav_reason"] = navigational_check(data, vessel, config)
    return G


def navigational_check(edge: dict, vessel: VesselProfile, config: RoutingConfig):
    """Depth / navigability check (independent of weather)."""
    if not edge.get("navigable", True):
        return False, "segment marked non-navigable"
    need = vessel.draft_m + config.under_keel_clearance_m
    if edge.get("min_depth_m", 1e9) < need:
        return False, (f"depth {edge['min_depth_m']:.1f} m < draft {vessel.draft_m} m + "
                       f"clearance {config.under_keel_clearance_m} m")
    return True, None


# >>> NB-CELL: dijkstra
# ---------------------------------------------------------------------------
# Dijkstra's algorithm and the shortest-distance route
# ---------------------------------------------------------------------------

def make_distance_weight(G: nx.DiGraph):
    """Weight = distance only. Navigationally impossible edges are hidden (None).
    Weather is IGNORED here on purpose - this is the baseline route."""
    def weight(u, v, d):
        if not d.get("nav_feasible", True):
            return None
        return d["distance_km"]
    return weight


def dijkstra_manual(G: nx.DiGraph, source: str, target: str, weight_fn: Callable):
    """
    Textbook Dijkstra with a priority queue (for learning; NetworkX is used
    in production and the notebook checks both give the same answer).
    Returns (cost, path) or (inf, None) if unreachable.
    """
    dist = {source: 0.0}
    prev: Dict[str, Optional[str]] = {source: None}
    heap = [(0.0, source)]
    visited = set()
    while heap:
        d_u, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        if u == target:
            path = []
            while u is not None:
                path.append(u)
                u = prev[u]
            return d_u, path[::-1]
        for v, data in G[u].items():
            w = weight_fn(u, v, data)
            if w is None or v in visited:      # None = edge not allowed
                continue
            nd = d_u + w
            if nd < dist.get(v, math.inf):
                dist[v], prev[v] = nd, u
                heapq.heappush(heap, (nd, v))
    return math.inf, None


def shortest_distance_route(G: nx.DiGraph, source: str, target: str) -> Optional[List[str]]:
    try:
        return nx.dijkstra_path(G, source, target, weight=make_distance_weight(G))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


# >>> NB-CELL: weather_aware
# ---------------------------------------------------------------------------
# Weather-aware optimisation (Dijkstra or A*) and route summaries
# ---------------------------------------------------------------------------

def make_weather_weight(G: nx.DiGraph, config: RoutingConfig):
    """Edge cost = distance_weight*km + weather_weight*penalty.
    Edges that break navigational OR weather constraints are hidden (None)."""
    def weight(u, v, d):
        if not d.get("nav_feasible", True):
            return None
        w = d.get("weather")
        if w is None:
            return None                      # never assume unknown = safe
        if not w["weather_feasible"]:
            return None
        return config.distance_weight * d["distance_km"] + config.weather_weight * w["weather_penalty"]
    return weight


def weather_aware_route(G: nx.DiGraph, source: str, target: str, config: RoutingConfig):
    weight = make_weather_weight(G, config)
    try:
        if config.algorithm == "astar":
            t = G.nodes[target]
            # Admissible heuristic: straight-line distance never overestimates
            heuristic = lambda n, _t: config.distance_weight * haversine_km(
                G.nodes[n]["lat"], G.nodes[n]["lon"], t["lat"], t["lon"])
            return nx.astar_path(G, source, target, heuristic=heuristic, weight=weight)
        return nx.dijkstra_path(G, source, target, weight=weight)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def format_duration(hours: float) -> str:
    total_min = int(round(hours * 60))
    d, rem = divmod(total_min, 24 * 60)
    h, m = divmod(rem, 60)
    return (f"{d} d " if d else "") + f"{h} h {m:02d} min"


def _r(x, n=2):
    return None if x is None else round(float(x), n)


def summarize_route(G: nx.DiGraph, path: Optional[List[str]], vessel: VesselProfile,
                    config: RoutingConfig, departure: datetime) -> Optional[dict]:
    """Totals, risk counts, travel time and per-segment details for a path."""
    if not path:
        return None
    segs, km, penalty, hours, cost = [], 0.0, 0.0, 0.0, 0.0
    for u, v in zip(path[:-1], path[1:]):
        d = G.edges[u, v]
        w = d["weather"]
        speed = vessel.service_speed_kn * (1 - config.speed_loss_fraction[w["risk_level"]])
        t_h = d["distance_nm"] / speed
        km += d["distance_km"]; penalty += w["weather_penalty"]; hours += t_h
        cost += config.distance_weight * d["distance_km"] + config.weather_weight * w["weather_penalty"]
        segs.append({
            "from": u, "to": v, "from_name": G.nodes[u]["name"], "to_name": G.nodes[v]["name"],
            "lane_type": d["lane_type"], "distance_km": _r(d["distance_km"], 1),
            "distance_nm": _r(d["distance_nm"], 1), "heading_deg": _r(d["bearing_deg"], 0),
            "risk_level": w["risk_level"], "risk_label": w["risk_label"],
            "severity_index": _r(w["severity_index"], 2), "weather_penalty": _r(w["weather_penalty"], 1),
            "wave_height_m": _r(w["wave_height_m"]), "max_wave_height_on_segment_m": _r(w["max_wave_height_on_segment_m"]),
            "wave_direction_deg": _r(w["wave_direction_deg"], 0),
            "relative_wave_angle_deg": _r(w["relative_wave_angle_deg"], 0), "encounter": w["encounter"],
            "effective_wave_height_m": _r(w["effective_wave_height_m"]),
            "wave_period_s": _r(w["wave_period_s"], 1), "swell_wave_height_m": _r(w["swell_wave_height_m"]),
            "wind_speed_kn": _r(w["wind_speed_kn"], 1), "max_wind_on_segment_kn": _r(w["max_wind_on_segment_kn"], 1),
            "wind_gusts_kn": _r(w["wind_gusts_kn"], 1), "wind_direction_deg": _r(w["wind_direction_deg"], 0),
            "data_complete": w["data_complete"], "data_sources": w["data_sources"],
            "data_errors": w["data_errors"], "nav_feasible": d["nav_feasible"],
            "weather_feasible": w["weather_feasible"], "violations": w["violations"],
            "planned_speed_kn": _r(speed, 1), "segment_time_h": _r(t_h, 2),
        })
    worst = max(s["risk_level"] for s in segs)
    return {
        "path": path, "path_names": [G.nodes[n]["name"] for n in path],
        "distance_km": _r(km, 1), "distance_nautical_miles": _r(km_to_nm(km), 1),
        "total_weather_penalty": _r(penalty, 1), "total_cost": _r(cost, 1),
        "weather_risk": RISK_LABELS[worst], "weather_risk_level": worst,
        "low_risk_segments": sum(s["risk_level"] == 0 for s in segs),
        "moderate_risk_segments": sum(s["risk_level"] == 1 for s in segs),
        "high_risk_segments": sum(s["risk_level"] == 2 for s in segs),
        "severe_risk_segments": sum(s["risk_level"] == 3 for s in segs),
        "segments_with_missing_weather": sum(not s["data_complete"] for s in segs),
        "weather_data_complete": all(s["data_complete"] for s in segs),
        "meets_constraints": all(s["nav_feasible"] and s["weather_feasible"] for s in segs),
        "constraint_violations": [f"{s['from']}->{s['to']}: {v}" for s in segs for v in s["violations"]],
        "estimated_travel_time": {"hours": _r(hours, 1), "text": format_duration(hours),
                                  "eta_utc": (departure + timedelta(hours=hours)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                                  "basis": f"service speed {vessel.service_speed_kn} kn reduced by "
                                           f"prototype weather speed-loss factors"},
        "segments": segs,
    }


# >>> NB-CELL: compare
# ---------------------------------------------------------------------------
# Route comparison, explanation and the main plan_route() entry point
# ---------------------------------------------------------------------------

DISCLAIMER = ("Prototype decision-support output for an academic project. Risk thresholds and "
              "vessel limits are configurable prototype values, not certified maritime safety "
              "limits. A low route cost does not mean the route is safe. Verify with official "
              "forecasts, nautical charts and professional maritime guidance.")


def resolve_port(G: nx.DiGraph, query) -> Optional[str]:
    """Accept a port id (e.g. 'INNSA') or a name / unique part of a name ('mumbai')."""
    if not isinstance(query, str) or not query.strip():
        return None
    q = query.strip().lower()
    ports = [n for n, d in G.nodes(data=True) if d.get("type") == "port"]
    for n in ports:
        if n.lower() == q or G.nodes[n]["name"].lower() == q:
            return n
    hits = [n for n in ports if q in G.nodes[n]["name"].lower()]
    return hits[0] if len(hits) == 1 else None


def list_ports(G):
    return {n: d["name"] for n, d in G.nodes(data=True) if d.get("type") == "port"}


def comparison_table(shortest: Optional[dict], recommended: Optional[dict]) -> pd.DataFrame:
    rows = [("Path", lambda r: " -> ".join(r["path"])),
            ("Distance (km)", lambda r: r["distance_km"]),
            ("Distance (nautical miles)", lambda r: r["distance_nautical_miles"]),
            ("Estimated travel time", lambda r: r["estimated_travel_time"]["text"]),
            ("ETA (UTC)", lambda r: r["estimated_travel_time"]["eta_utc"]),
            ("Total weather penalty", lambda r: r["total_weather_penalty"]),
            ("Total route cost", lambda r: r["total_cost"]),
            ("Overall weather risk", lambda r: r["weather_risk"]),
            ("High-risk segments", lambda r: r["high_risk_segments"]),
            ("Severe-risk segments", lambda r: r["severe_risk_segments"]),
            ("Segments with missing weather", lambda r: r["segments_with_missing_weather"]),
            ("Meets all constraints", lambda r: r["meets_constraints"])]
    data = {"Metric": [n for n, _ in rows],
            "Shortest-distance route": [f(shortest) if shortest else "no route" for _, f in rows],
            "Weather-aware route": [f(recommended) if recommended else "NO FEASIBLE ROUTE" for _, f in rows]}
    return pd.DataFrame(data).set_index("Metric")


def explain_routes(shortest: Optional[dict], recommended: Optional[dict], G=None) -> str:
    if recommended is None:
        if shortest is None:
            return "No navigable connection exists between these ports in the network."
        v = shortest["constraint_violations"][:4]
        return ("NO FEASIBLE ROUTE FOUND. Every connection between the ports passes through at least "
                "one segment that violates the configured safety constraints, so no route is "
                "recommended. The shortest-distance route is shown only for reference and must NOT be "
                "used. Examples of blocked segments: " + "; ".join(v) +
                ". Options: delay departure, use a vessel with higher limits, or review constraints "
                "with maritime professionals.")
    parts = []
    if shortest and shortest["path"] == recommended["path"]:
        parts.append(f"The shortest-distance route ({recommended['distance_km']} km) is also the "
                     f"weather-aware recommendation: its weather risk is {recommended['weather_risk']}, "
                     "so no detour is justified.")
    elif shortest:
        extra = recommended["distance_km"] - shortest["distance_km"]
        pct = 100 * extra / shortest["distance_km"] if shortest["distance_km"] else 0
        reason = []
        if not shortest["meets_constraints"]:
            reason.append("the shortest route violates safety constraints (" +
                          "; ".join(shortest["constraint_violations"][:3]) + ")")
        if shortest["high_risk_segments"] or shortest["severe_risk_segments"]:
            reason.append(f"the shortest route has {shortest['high_risk_segments']} high-risk and "
                          f"{shortest['severe_risk_segments']} severe-risk segment(s)")
        if not reason:
            reason.append("the shortest route has a higher combined distance + weather cost")
        parts.append(f"The weather-aware route was selected because {' and '.join(reason)}. It is "
                     f"{extra:.0f} km ({pct:.1f}%) longer, but its overall risk is "
                     f"{recommended['weather_risk']} and its weather penalty is "
                     f"{recommended['total_weather_penalty']} versus {shortest['total_weather_penalty']} "
                     f"(total cost {recommended['total_cost']} vs {shortest['total_cost']}).")
    if recommended["high_risk_segments"] or recommended["severe_risk_segments"]:
        parts.append(f"Caution: the recommended route still contains {recommended['high_risk_segments']} "
                     "high-risk segment(s); it was the lowest-cost route that meets the constraints.")
    if not recommended["weather_data_complete"]:
        parts.append(f"WARNING: weather data was missing for {recommended['segments_with_missing_weather']} "
                     "segment(s); those were treated conservatively but their risk is NOT verified.")
    return " ".join(parts)


def plan_route(G_base: nx.DiGraph, request: dict, provider: WeatherProvider,
               config: Optional[RoutingConfig] = None) -> dict:
    """
    Main entry point. Never raises for bad input or weather failures:
    returns {"status": "ok" | "no_feasible_route" | "no_route" | "error", ...}.
    """
    config = config or RoutingConfig()
    base = {"status": "error", "request": request, "disclaimer": DISCLAIMER,
            "data_notice": G_base.graph.get("data_notice", ""), "warnings": []}
    # 1. validate the request
    try:
        src = resolve_port(G_base, request.get("source_port"))
        dst = resolve_port(G_base, request.get("destination_port"))
        if src is None or dst is None:
            bad = request.get("source_port") if src is None else request.get("destination_port")
            raise ValueError(f"Unknown port '{bad}'. Valid ports: {list_ports(G_base)}")
        if src == dst:
            raise ValueError("source_port and destination_port must be different")
        vessel = get_vessel_profile(request.get("vessel_type", "container_ship"),
                                    request.get("vessel_speed"), request.get("vessel_specs"))
        dep = request.get("departure_time")
        departure = to_utc(dep) if dep else datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
    except (ValueError, TypeError) as exc:
        base["error"] = str(exc)
        return base

    # 2. weather for every segment (work on a copy so requests never interfere)
    G = G_base.copy()
    window = auto_window_hours(G, src, dst, vessel.service_speed_kn, config)
    weather, edge_points = fetch_network_weather(G, provider, departure, window, config.sample_spacing_km)
    annotate_graph_with_weather(G, weather, edge_points, vessel, config)
    n_missing = sum(1 for r in weather.values() if r.get("wave_height_m") is None or r.get("wind_speed_kn") is None)
    if n_missing:
        base["warnings"].append(f"Weather data missing for {n_missing} of {len(weather)} sample points "
                                f"(policy: {config.missing_data_policy}).")

    # 3. routes
    short_path = shortest_distance_route(G, src, dst)
    rec_path = weather_aware_route(G, src, dst, config) if short_path else None
    shortest = summarize_route(G, short_path, vessel, config, departure)
    recommended = summarize_route(G, rec_path, vessel, config, departure)

    status = "ok" if recommended else ("no_feasible_route" if shortest else "no_route")
    base.update({
        "status": status,
        "source_port": {"id": src, "name": G.nodes[src]["name"]},
        "destination_port": {"id": dst, "name": G.nodes[dst]["name"]},
        "vessel": asdict(vessel),
        "departure_time_utc": departure.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "weather_window_hours": window,
        "weather_source": sorted({str(r.get("source")) for r in weather.values()}),
        "algorithm": config.algorithm,
        "recommended_route": recommended,
        "alternative_route": shortest,
        "alternative_route_role": "shortest-distance route (ignores weather; for comparison only)",
        "distance_km": recommended["distance_km"] if recommended else None,
        "distance_nautical_miles": recommended["distance_nautical_miles"] if recommended else None,
        "estimated_travel_time": recommended["estimated_travel_time"] if recommended else None,
        "weather_risk": recommended["weather_risk"] if recommended else "No feasible route",
        "segment_weather_details": recommended["segments"] if recommended else
                                   (shortest["segments"] if shortest else []),
        "route_explanation": explain_routes(shortest, recommended, G),
    })
    if status == "no_route":
        base["route_explanation"] = ("No navigable connection exists between "
                                     f"{src} and {dst} for this vessel (graph is disconnected or "
                                     "every link fails depth/navigability checks).")
    if any(str(s).startswith("synthetic") for s in base["weather_source"]):
        base["warnings"].append("Weather data is SYNTHETIC test data, not a real forecast.")
    base["_graph"] = G   # annotated graph for visualisation (removed by to_json_safe)
    return base


# >>> NB-CELL: json_api
# ---------------------------------------------------------------------------
# JSON request/response layer (what a FastAPI endpoint will call later)
# ---------------------------------------------------------------------------

def to_json_safe(result: dict) -> dict:
    """Drop internal objects (the graph) and make everything JSON-serialisable."""
    clean = {k: v for k, v in result.items() if not k.startswith("_")}
    return json.loads(json.dumps(clean, default=str))


def handle_route_request(request_json, G: nx.DiGraph, provider: WeatherProvider,
                         config: Optional[RoutingConfig] = None) -> str:
    """Accepts a JSON string or dict, returns a JSON string (backend-friendly)."""
    try:
        request = json.loads(request_json) if isinstance(request_json, str) else dict(request_json)
    except (json.JSONDecodeError, TypeError, ValueError) as exc:
        return json.dumps({"status": "error", "error": f"invalid JSON request: {exc}"})
    required = ["source_port", "destination_port"]
    missing = [k for k in required if not request.get(k)]
    if missing:
        return json.dumps({"status": "error", "error": f"missing required fields: {missing}"})
    return json.dumps(to_json_safe(plan_route(G, request, provider, config)), indent=2)

In [ ]:
%%writefile {PROJECT_DIR}/visualization.py
"""
visualization.py
================
Maps and charts for the weather-aware routing results.

* plot_route_map()       - static Matplotlib schematic (lat/lon axes, no coastline)
* plot_risk_profile()    - severity of each segment along the recommended route
* make_folium_map()      - interactive Leaflet map (needs `folium`)

All functions take the dict returned by routing.plan_route() (which carries the
annotated graph under the key "_graph").

Colours: risk levels use a fixed status palette (always shown with a text
label in the legend/tooltips, never colour alone).
"""

from __future__ import annotations

import math
from typing import Optional

# >>> NB-CELL: visualization
RISK_COLORS = {0: "#0aac23", 1: "#fbb315", 2: "#f18a59", 3: "#dc3c3d"}
RISK_NAMES = {0: "Low", 1: "Moderate", 2: "High", 3: "Severe"}
MISSING_COLOR = "#9c9c9c"
RECOMMENDED_COLOR = "#257de0"
SHORTEST_COLOR = "#474747"
INK = "#0b0b0b"
GRID = "#e7e7e7"


def _undirected_edges(G):
    """One entry per sea lane with the WORSE of its two directions."""
    seen = {}
    for u, v, d in G.edges(data=True):
        key = tuple(sorted((u, v)))
        w = d.get("weather") or {}
        level = w.get("risk_level", 0)
        missing = not w.get("data_complete", True)
        blocked = not (d.get("nav_feasible", True) and w.get("weather_feasible", True))
        if key not in seen or level > seen[key]["level"]:
            seen[key] = {"u": key[0], "v": key[1], "level": level, "missing": missing,
                         "blocked": blocked, "data": d}
        else:
            seen[key]["missing"] |= missing
            seen[key]["blocked"] |= blocked
    return list(seen.values())


def _title(result):
    s, t = result.get("source_port", {}), result.get("destination_port", {})
    return f"{s.get('name', '?')} to {t.get('name', '?')}  |  status: {result.get('status')}"


def plot_route_map(result: dict, save_path: Optional[str] = None, title: Optional[str] = None, show=True):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    G = result["_graph"]
    fig, ax = plt.subplots(figsize=(10, 10))
    # risk-coloured lanes
    for e in _undirected_edges(G):
        a, b = G.nodes[e["u"]], G.nodes[e["v"]]
        color = MISSING_COLOR if e["missing"] and e["level"] <= 1 else RISK_COLORS[e["level"]]
        ax.plot([a["lon"], b["lon"]], [a["lat"], b["lat"]], color=color, lw=2,
                ls=":" if e["blocked"] else "-", zorder=2, solid_capstyle="round")
    # routes
    rec, alt = result.get("recommended_route"), result.get("alternative_route")
    if rec:
        xs = [G.nodes[n]["lon"] for n in rec["path"]]; ys = [G.nodes[n]["lat"] for n in rec["path"]]
        ax.plot(xs, ys, color=RECOMMENDED_COLOR, lw=9, alpha=0.35, zorder=1, solid_capstyle="round")
    if alt:
        xs = [G.nodes[n]["lon"] for n in alt["path"]]; ys = [G.nodes[n]["lat"] for n in alt["path"]]
        ax.plot(xs, ys, color=SHORTEST_COLOR, lw=1.6, ls=(0, (6, 3)), zorder=3)
    # nodes
    for n, d in G.nodes(data=True):
        if d["type"] == "port":
            ax.scatter(d["lon"], d["lat"], s=70, marker="s", color=INK, zorder=4,
                       edgecolor="white", linewidth=1.5)
            ax.annotate(d["name"], (d["lon"], d["lat"]), xytext=(6, 4), textcoords="offset points",
                        fontsize=8, color=INK)
        else:
            ax.scatter(d["lon"], d["lat"], s=14, color="#9c9c9c", zorder=4)
    src, dst = result.get("source_port", {}).get("id"), result.get("destination_port", {}).get("id")
    for n, lab in ((src, "SOURCE"), (dst, "DESTINATION")):
        if n in G:
            ax.scatter(G.nodes[n]["lon"], G.nodes[n]["lat"], s=220, facecolor="none",
                       edgecolor=RECOMMENDED_COLOR, linewidth=2.5, zorder=5)
            ax.annotate(lab, (G.nodes[n]["lon"], G.nodes[n]["lat"]), xytext=(6, -12),
                        textcoords="offset points", fontsize=8, fontweight="bold", color=INK)
    handles = [Line2D([], [], color=RISK_COLORS[i], lw=2, label=f"Lane risk: {RISK_NAMES[i]}") for i in range(4)]
    handles += [Line2D([], [], color=MISSING_COLOR, lw=2, label="Lane: weather data missing"),
                Line2D([], [], color=INK, lw=2, ls=":", label="Lane blocked (constraints)"),
                Line2D([], [], color=RECOMMENDED_COLOR, lw=8, alpha=0.35, label="Weather-aware route"),
                Line2D([], [], color=SHORTEST_COLOR, lw=1.6, ls=(0, (6, 3)), label="Shortest-distance route"),
                Line2D([], [], marker="s", color=INK, ls="", label="Port"),
                Line2D([], [], marker="o", color="#9c9c9c", ls="", markersize=4, label="Waypoint")]
    ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=True)
    mean_lat = sum(d["lat"] for _, d in G.nodes(data=True)) / G.number_of_nodes()
    ax.set_aspect(1 / math.cos(math.radians(mean_lat)))
    ax.grid(color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel("Longitude (deg E)"); ax.set_ylabel("Latitude (deg N)")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_title((title or _title(result)) + "\nDEMONSTRATION ONLY - schematic network, no coastline, "
                 "not a navigational chart", fontsize=10, color=INK, loc="left")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def plot_risk_profile(result: dict, save_path: Optional[str] = None, show=True):
    """Bar per segment of the recommended route (or the reference route if none)."""
    import matplotlib.pyplot as plt
    route = result.get("recommended_route") or result.get("alternative_route")
    if not route:
        print("No route to plot.")
        return None
    segs = route["segments"]
    labels = [f"{s['from']}\n->{s['to']}" for s in segs]
    vals = [s["severity_index"] if s["severity_index"] is not None else 0 for s in segs]
    colors = [RISK_COLORS[s["risk_level"]] if s["data_complete"] else MISSING_COLOR for s in segs]
    fig, ax = plt.subplots(figsize=(max(8, 0.7 * len(segs)), 4.5))
    ax.bar(range(len(segs)), vals, color=colors, width=0.7, edgecolor="white", linewidth=2)
    ax.axhline(1.0, color=INK, lw=1, ls="--")
    ax.text(len(segs) - 0.5, 1.02, "Severe threshold", ha="right", va="bottom", fontsize=8, color=INK)
    for i, s in enumerate(segs):
        ax.text(i, vals[i] + 0.02, s["risk_label"] if s["data_complete"] else "missing",
                ha="center", va="bottom", fontsize=7, color=INK)
    ax.set_xticks(range(len(segs))); ax.set_xticklabels(labels, fontsize=7, rotation=90)
    ax.set_ylabel("Severity index (1.0 = Severe threshold)")
    which = "recommended" if result.get("recommended_route") else "reference (NOT recommended)"
    ax.set_title(f"Weather severity along the {which} route\n{_title(result)}", fontsize=10, loc="left")
    ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def _segment_tooltip(G, u, v):
    d = G.edges[u, v]; w = d.get("weather", {})
    fmt = lambda x, unit="": "n/a" if x is None else f"{x:.1f}{unit}"
    return (f"<b>{G.nodes[u]['name']} &rarr; {G.nodes[v]['name']}</b><br>"
            f"Risk: <b>{w.get('risk_label', '?')}</b>{'' if w.get('data_complete', True) else ' (data missing)'}<br>"
            f"Distance: {d['distance_km']:.0f} km / {d['distance_nm']:.0f} nm<br>"
            f"Wave height: {fmt(w.get('wave_height_m'), ' m')} ({w.get('encounter', '?')})<br>"
            f"Wind: {fmt(w.get('wind_speed_kn'), ' kn')}, gusts {fmt(w.get('wind_gusts_kn'), ' kn')}<br>"
            f"Depth (illustrative): {d['min_depth_m']:.1f} m<br>"
            f"Source: {', '.join(w.get('data_sources', []))}")


def make_folium_map(result: dict, save_path: Optional[str] = None):
    """Interactive map. Returns the folium.Map, or None if folium is not installed."""
    try:
        import folium
    except ImportError:
        print("folium is not installed - skipping the interactive map (pip install folium).")
        return None
    G = result["_graph"]
    lats = [d["lat"] for _, d in G.nodes(data=True)]; lons = [d["lon"] for _, d in G.nodes(data=True)]
    m = folium.Map(location=[sum(lats) / len(lats), sum(lons) / len(lons)], zoom_start=5,
                   tiles="CartoDB positron", control_scale=True)
    lanes = folium.FeatureGroup(name="Sea lanes coloured by weather risk", show=True)
    for e in _undirected_edges(G):
        a, b = G.nodes[e["u"]], G.nodes[e["v"]]
        color = MISSING_COLOR if e["missing"] and e["level"] <= 1 else RISK_COLORS[e["level"]]
        folium.PolyLine([(a["lat"], a["lon"]), (b["lat"], b["lon"])], color=color, weight=3,
                        opacity=0.9, dash_array="4 6" if e["blocked"] else None,
                        tooltip=folium.Tooltip(_segment_tooltip(G, e["u"], e["v"]))).add_to(lanes)
    lanes.add_to(m)
    for key, label, color, weight, dash in (("alternative_route", "Shortest-distance route", SHORTEST_COLOR, 3, "10 6"),
                                            ("recommended_route", "Weather-aware recommended route", RECOMMENDED_COLOR, 7, None)):
        route = result.get(key)
        if not route:
            continue
        fg = folium.FeatureGroup(name=label, show=True)
        for s in route["segments"]:
            a, b = G.nodes[s["from"]], G.nodes[s["to"]]
            folium.PolyLine([(a["lat"], a["lon"]), (b["lat"], b["lon"])], color=color, weight=weight,
                            opacity=0.55 if key == "recommended_route" else 0.9, dash_array=dash,
                            tooltip=folium.Tooltip(f"<b>{label}</b><br>" + _segment_tooltip(G, s["from"], s["to"]))
                            ).add_to(fg)
        fg.add_to(m)
    nodes = folium.FeatureGroup(name="Ports and waypoints", show=True)
    src, dst = result.get("source_port", {}).get("id"), result.get("destination_port", {}).get("id")
    for n, d in G.nodes(data=True):
        if n in (src, dst):
            folium.Marker((d["lat"], d["lon"]), tooltip=f"{'SOURCE' if n == src else 'DESTINATION'}: {d['name']}",
                          icon=folium.Icon(color="blue" if n == src else "red",
                                           icon="play" if n == src else "flag")).add_to(nodes)
        elif d["type"] == "port":
            folium.CircleMarker((d["lat"], d["lon"]), radius=6, color=INK, fill=True, fill_opacity=1,
                                tooltip=f"Port: {d['name']} ({n})").add_to(nodes)
        else:
            folium.CircleMarker((d["lat"], d["lon"]), radius=3, color="#7a7a7a", fill=True,
                                tooltip=f"Waypoint (synthetic): {d['name']}").add_to(nodes)
    nodes.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    rows = "".join(f'<div><span style="display:inline-block;width:22px;height:4px;background:{RISK_COLORS[i]};'
                   f'vertical-align:middle;margin-right:6px"></span>{RISK_NAMES[i]} risk lane</div>' for i in range(4))
    legend = f"""
    <div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 12px;
                border:1px solid rgba(11,11,11,0.15);border-radius:6px;font:12px sans-serif;color:{INK}">
      <b>Legend</b>{rows}
      <div><span style="display:inline-block;width:22px;height:4px;background:{MISSING_COLOR};vertical-align:middle;margin-right:6px"></span>Weather data missing</div>
      <div><span style="display:inline-block;width:22px;border-top:3px dashed {INK};vertical-align:middle;margin-right:6px"></span>Blocked lane (constraints)</div>
      <div><span style="display:inline-block;width:22px;height:7px;background:{RECOMMENDED_COLOR};opacity:.55;vertical-align:middle;margin-right:6px"></span>Weather-aware route</div>
      <div><span style="display:inline-block;width:22px;border-top:3px dashed {SHORTEST_COLOR};vertical-align:middle;margin-right:6px"></span>Shortest-distance route</div>
    </div>"""
    banner = f"""
    <div style="position:fixed;top:12px;left:60px;right:220px;z-index:9999;background:#fff8e1;padding:8px 12px;
                border:1px solid #fbb315;border-radius:6px;font:12px sans-serif;color:{INK}">
      <b>DEMONSTRATION MAP - NOT A NAVIGATIONAL CHART.</b> Synthetic waypoint network.
      {_title(result)} | weather source: {', '.join(result.get('weather_source', []))}
      | Weather data by Open-Meteo.com (CC-BY 4.0) when the source is open-meteo.
    </div>"""
    m.get_root().html.add_child(folium.Element(legend + banner))
    if save_path:
        m.save(save_path)
    return m

In [ ]:
%%writefile {PROJECT_DIR}/sample_network.py
"""
sample_network.py
=================
A SMALL, SYNTHETIC maritime waypoint network around the Indian peninsula and
Sri Lanka, for DEMONSTRATION ONLY.

    !!!  NOT A NAVIGATIONAL CHART. NOT VERIFIED REAL-WORLD ROUTE DATA.  !!!

* Port coordinates are approximate public locations of the port areas.
* Offshore waypoints were placed by hand in open water and each straight
  segment was checked by hand to stay at sea (to keep it away from land and
  the Lakshadweep islands). They are NOT official traffic lanes.
* Depths ("min_depth_m") are ROUGH ILLUSTRATIVE VALUES, not survey data.
  The Pamban / Palk Strait link is deliberately given a very shallow depth
  because that passage is known to be too shallow for large ships; the exact
  number here is illustrative.

To use real data later, replace these tables with a real maritime network
(for example, a lane network derived from AIS traffic density, or a
published shipping-lane dataset) using the SAME column names, and load it
with routing.load_network_from_csv().
"""

from __future__ import annotations

import pandas as pd

# >>> NB-CELL: sample_network
DATA_NOTICE = ("SYNTHETIC DEMONSTRATION NETWORK - hand-placed waypoints and "
               "illustrative depths. Not a navigational chart and not verified "
               "real-world route data. Do not use for actual navigation.")

# node_id, name, type, lat, lon, country, notes
_WAYPOINTS = [
    # --- Ports (approximate locations; IDs follow UN/LOCODE style) ---------
    ("INNSA", "Mumbai (JNPT / Nhava Sheva)", "port", 18.95, 72.95, "IN", "port"),
    ("INMRM", "Mormugao (Goa)",              "port", 15.41, 73.80, "IN", "port"),
    ("INNML", "New Mangalore",               "port", 12.92, 74.80, "IN", "port"),
    ("INCOK", "Kochi (Cochin)",              "port",  9.97, 76.24, "IN", "port"),
    ("INTUT", "Tuticorin (V.O. Chidambaranar)", "port", 8.76, 78.20, "IN", "port"),
    ("INMAA", "Chennai",                     "port", 13.10, 80.30, "IN", "port"),
    ("INVTZ", "Visakhapatnam",               "port", 17.69, 83.30, "IN", "port"),
    ("LKCMB", "Colombo",                     "port",  6.95, 79.84, "LK", "port"),
    # --- Offshore approach waypoints (synthetic) ---------------------------
    ("WP_MUM_OFF", "Off Mumbai",        "waypoint", 18.90, 72.40, "", "synthetic"),
    ("WP_GOA_OFF", "Off Goa",           "waypoint", 15.40, 73.30, "", "synthetic"),
    ("WP_MNG_OFF", "Off Mangalore",     "waypoint", 12.90, 74.30, "", "synthetic"),
    ("WP_KOC_OFF", "Off Kochi",         "waypoint",  9.95, 75.80, "", "synthetic"),
    ("WP_TUT_OFF", "Off Tuticorin",     "waypoint",  8.70, 78.60, "", "synthetic"),
    ("WP_CHE_OFF", "Off Chennai",       "waypoint", 13.10, 80.70, "", "synthetic"),
    ("WP_VIZ_OFF", "Off Visakhapatnam", "waypoint", 17.50, 83.70, "", "synthetic"),
    ("WP_COL_OFF", "Off Colombo",       "waypoint",  6.95, 79.50, "", "synthetic"),
    # --- Arabian Sea offshore lane (west of Lakshadweep, via 9-degree channel)
    ("WP_ARAB_N",  "Arabian Sea North",       "waypoint", 17.00, 71.50, "", "synthetic"),
    ("WP_ARAB_W",  "Arabian Sea West",        "waypoint", 13.00, 71.00, "", "synthetic"),
    ("WP_NINE_W",  "Nine Degree Channel W",   "waypoint",  9.20, 71.80, "", "synthetic"),
    ("WP_NINE_E",  "Nine Degree Channel E",   "waypoint",  9.00, 74.50, "", "synthetic"),
    # --- Southern tip and around Sri Lanka ----------------------------------
    ("WP_CAPE_S",  "South of Kanyakumari",    "waypoint",  7.60, 77.50, "", "synthetic"),
    ("WP_SL_SW",   "SW of Sri Lanka",         "waypoint",  5.80, 79.90, "", "synthetic"),
    ("WP_DONDRA",  "South of Dondra Head",    "waypoint",  5.55, 80.60, "", "synthetic"),
    ("WP_SL_SE",   "SE of Sri Lanka",         "waypoint",  6.00, 81.80, "", "synthetic"),
    ("WP_SL_E",    "East of Sri Lanka",       "waypoint",  7.60, 82.30, "", "synthetic"),
    ("WP_TRINCO",  "Off Trincomalee",         "waypoint",  8.80, 81.70, "", "synthetic"),
    # --- Shallow Pamban / Palk Strait passage (small craft only) -----------
    ("WP_PAMBAN",  "South of Pamban Pass",    "waypoint",  9.20, 79.20, "", "synthetic"),
    ("WP_PALK_BAY", "Palk Bay",               "waypoint",  9.60, 79.50, "", "synthetic"),
    ("WP_PALK_STR", "Palk Strait",            "waypoint", 10.05, 79.95, "", "synthetic"),
    # --- Bay of Bengal coastal + offshore lanes -----------------------------
    ("WP_EAST_MID", "Off Pondicherry",        "waypoint", 11.00, 81.00, "", "synthetic"),
    ("WP_KRISHNA", "Off Krishna Delta",       "waypoint", 15.50, 81.30, "", "synthetic"),
    ("WP_GODAVARI", "Off Godavari Delta",     "waypoint", 16.20, 82.60, "", "synthetic"),
    ("WP_BAY_S",   "Bay of Bengal South",     "waypoint", 10.50, 83.00, "", "synthetic"),
    ("WP_BAY_MID", "Bay of Bengal Central",   "waypoint", 13.50, 83.00, "", "synthetic"),
    ("WP_BAY_N",   "Bay of Bengal North",     "waypoint", 16.00, 85.00, "", "synthetic"),
]

# u, v, lane_type, min_depth_m (ILLUSTRATIVE), navigable
_EDGES = [
    # port approaches
    ("INNSA", "WP_MUM_OFF", "approach", 16.0, True),
    ("INMRM", "WP_GOA_OFF", "approach", 16.0, True),
    ("INNML", "WP_MNG_OFF", "approach", 16.0, True),
    ("INCOK", "WP_KOC_OFF", "approach", 16.0, True),
    ("INTUT", "WP_TUT_OFF", "approach", 16.0, True),
    ("INMAA", "WP_CHE_OFF", "approach", 16.0, True),
    ("INVTZ", "WP_VIZ_OFF", "approach", 16.0, True),
    ("LKCMB", "WP_COL_OFF", "approach", 16.0, True),
    # west coast coastal lane
    ("WP_MUM_OFF", "WP_GOA_OFF", "coastal", 200.0, True),
    ("WP_GOA_OFF", "WP_MNG_OFF", "coastal", 200.0, True),
    ("WP_MNG_OFF", "WP_KOC_OFF", "coastal", 200.0, True),
    ("WP_KOC_OFF", "WP_CAPE_S",  "coastal", 200.0, True),
    # Arabian Sea offshore lane
    ("WP_MUM_OFF", "WP_ARAB_N",  "offshore", 1000.0, True),
    ("WP_ARAB_N",  "WP_ARAB_W",  "offshore", 1000.0, True),
    ("WP_GOA_OFF", "WP_ARAB_W",  "offshore", 1000.0, True),
    ("WP_ARAB_W",  "WP_NINE_W",  "offshore", 1000.0, True),
    ("WP_NINE_W",  "WP_NINE_E",  "channel",  1000.0, True),
    ("WP_NINE_E",  "WP_KOC_OFF", "offshore", 1000.0, True),
    ("WP_NINE_E",  "WP_CAPE_S",  "offshore", 1000.0, True),
    # Gulf of Mannar / south
    ("WP_CAPE_S",  "WP_TUT_OFF", "coastal", 200.0, True),
    ("WP_CAPE_S",  "WP_COL_OFF", "offshore", 1000.0, True),
    ("WP_TUT_OFF", "WP_COL_OFF", "offshore", 1000.0, True),
    # around Sri Lanka
    ("WP_COL_OFF", "WP_SL_SW",  "coastal", 200.0, True),
    ("WP_SL_SW",   "WP_DONDRA", "coastal", 200.0, True),
    ("WP_DONDRA",  "WP_SL_SE",  "coastal", 200.0, True),
    ("WP_SL_SE",   "WP_SL_E",   "coastal", 200.0, True),
    ("WP_SL_E",    "WP_TRINCO", "coastal", 200.0, True),
    # shallow Pamban / Palk passage - illustrative depth, small craft only
    ("WP_TUT_OFF",  "WP_PAMBAN",   "shallow_channel", 4.5, True),
    ("WP_PAMBAN",   "WP_PALK_BAY", "shallow_channel", 4.5, True),
    ("WP_PALK_BAY", "WP_PALK_STR", "shallow_channel", 4.5, True),
    ("WP_PALK_STR", "WP_EAST_MID", "shallow_channel", 6.0, True),
    # Bay of Bengal coastal lane
    ("WP_TRINCO",  "WP_EAST_MID", "coastal", 200.0, True),
    ("WP_EAST_MID", "WP_CHE_OFF", "coastal", 200.0, True),
    ("WP_CHE_OFF", "WP_KRISHNA",  "coastal", 200.0, True),
    ("WP_KRISHNA", "WP_GODAVARI", "coastal", 200.0, True),
    ("WP_GODAVARI", "WP_VIZ_OFF", "coastal", 200.0, True),
    # Bay of Bengal offshore lane
    ("WP_TRINCO",  "WP_BAY_S",   "offshore", 1000.0, True),
    ("WP_SL_E",    "WP_BAY_S",   "offshore", 1000.0, True),
    ("WP_BAY_S",   "WP_BAY_MID", "offshore", 1000.0, True),
    ("WP_CHE_OFF", "WP_BAY_MID", "offshore", 1000.0, True),
    ("WP_BAY_MID", "WP_BAY_N",   "offshore", 1000.0, True),
    ("WP_BAY_N",   "WP_VIZ_OFF", "offshore", 1000.0, True),
]


def create_sample_network():
    """Return (waypoints_df, edges_df) for the synthetic demo network."""
    waypoints = pd.DataFrame(_WAYPOINTS, columns=["node_id", "name", "type", "lat", "lon",
                                                  "country", "notes"])
    edges = pd.DataFrame(_EDGES, columns=["u", "v", "lane_type", "min_depth_m", "navigable"])
    waypoints["data_status"] = "synthetic_demo"
    edges["data_status"] = "synthetic_demo"
    edges["bidirectional"] = True
    return waypoints, edges

In [ ]:
%%writefile {PROJECT_DIR}/requirements.txt
# Weather-Aware Ship Routing module (ml/ship_routing)
# Core routing + weather service
networkx>=3.0
numpy>=1.24
pandas>=2.0
requests>=2.31

# Visualisation (notebook and reports)
matplotlib>=3.7
folium>=0.14

# Future API service - uncomment when you build the FastAPI layer
# fastapi>=0.110
# uvicorn[standard]>=0.29

In [ ]:
%%writefile {PROJECT_DIR}/fastapi_app_example.py
"""
fastapi_app_example.py
======================
EXAMPLE ONLY - shows how the routing module can be exposed as a FastAPI
service later. It is not used by the notebook.

Run (from the ml/ship_routing folder, after `pip install fastapi uvicorn`):
    uvicorn fastapi_app_example:app --reload --port 8001

Then POST to http://localhost:8001/route with a JSON body such as:
    {"source_port": "INNSA", "destination_port": "INMAA",
     "vessel_speed": 15, "vessel_type": "container_ship",
     "vessel_specs": {"draft_m": 12.5},
     "departure_time": "2026-12-01T06:00:00Z"}
"""

import os
from typing import Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from routing import RoutingConfig, list_ports, load_network_from_csv, plan_route, to_json_safe
from sample_network import DATA_NOTICE
from weather_service import OpenMeteoProvider, WeatherConfig

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")

# Load the network and weather provider ONCE when the service starts
GRAPH = load_network_from_csv(os.path.join(DATA_DIR, "waypoints.csv"),
                              os.path.join(DATA_DIR, "edges.csv"), DATA_NOTICE)
PROVIDER = OpenMeteoProvider(WeatherConfig(cache_file=os.path.join(BASE_DIR, "results", "cache",
                                                                   "weather_cache.json")))
CONFIG = RoutingConfig()

app = FastAPI(title="Weather-Aware Ship Routing (prototype)", version="0.1.0")


class VesselSpecs(BaseModel):
    draft_m: Optional[float] = Field(None, gt=0)
    max_wave_height_m: Optional[float] = Field(None, gt=0)
    max_wind_speed_kn: Optional[float] = Field(None, gt=0)


class RouteRequest(BaseModel):
    source_port: str
    destination_port: str
    vessel_speed: Optional[float] = Field(None, gt=0, le=40)
    vessel_type: str = "container_ship"
    vessel_specs: Optional[VesselSpecs] = None
    departure_time: Optional[str] = None      # ISO 8601, UTC recommended


@app.get("/health")
def health():
    return {"status": "ok", "nodes": GRAPH.number_of_nodes(), "edges": GRAPH.number_of_edges()}


@app.get("/ports")
def ports():
    return list_ports(GRAPH)


@app.post("/route")
def route(req: RouteRequest):
    # A plain `def` endpoint runs in a worker thread, so the blocking HTTP
    # calls to the weather API do not freeze the server.
    result = to_json_safe(plan_route(GRAPH, req.model_dump(exclude_none=True), PROVIDER, CONFIG))
    if result["status"] == "error":
        raise HTTPException(status_code=422, detail=result.get("error"))
    return result

In [ ]:
%%writefile {PROJECT_DIR}/README.md
# Weather-Aware Ship Routing (`ml/ship_routing`)

Part of the **Supply Chain and Coastal Chain Intelligence Platform**. This module supports the
*Shipping Company* role: given a source port, destination port, vessel and departure time it
returns a recommended sea route that balances distance against weather risk, plus the
shortest-distance route for comparison.

It is independent of `ml/port_congestion/` and does not modify it.

> **Prototype for an academic project.** The waypoint network is synthetic, and the risk
> thresholds and vessel limits are prototype values, not certified maritime safety limits.
> Do not use for real navigation.

## Folder structure

```
ml/ship_routing/
├── notebooks/weather_aware_ship_routing.ipynb   # Colab notebook (sections 1-21)
├── routing.py              # graph, risk scoring, Dijkstra/A*, plan_route(), JSON layer
├── weather_service.py      # Open-Meteo provider (real data) + synthetic test provider
├── visualization.py        # Matplotlib schematic, risk profile, Folium map
├── sample_network.py       # SYNTHETIC demo waypoint network (India + Sri Lanka)
├── fastapi_app_example.py  # example API wrapper (not used by the notebook)
├── requirements.txt
├── data/                   # waypoints.csv, edges.csv, *_config.json
└── results/                # json/, tables/, figures/, maps/, cache/ (cache not needed in Git)
```

## How it works

1. **Graph:** ports and offshore waypoints are nodes; sea lanes are directed edges with
   great-circle distance (haversine), heading and an illustrative depth.
2. **Weather:** points every ~100 km along each lane are sent in batches to Open-Meteo:
   * Marine API `https://marine-api.open-meteo.com/v1/marine`: `wave_height`,
     `wave_direction`, `wave_period`, `swell_wave_height` (with `cell_selection=sea`)
   * Forecast API `https://api.open-meteo.com/v1/forecast`: `wind_speed_10m`,
     `wind_direction_10m`, `wind_gusts_10m` (with `wind_speed_unit=kn`)

   No API key is needed (non-commercial use). The module retries on HTTP 429/5xx, caches
   responses, and never crashes on failures: missing data is handled by a policy
   (`conservative` by default). Weather data by Open-Meteo.com, CC-BY 4.0.
3. **Risk:** each sample point is classified Low / Moderate / High / Severe from the effective
   wave height (wave height × head/beam/following-sea factor), wind and gusts. The worst point
   sets the lane's level.
4. **Routing:** `cost = distance_weight·km + weather_weight·km·penalty[level]`. Lanes that break
   vessel limits (wave, wind, depth) or are Severe are removed, then Dijkstra (or A*) finds the
   cheapest feasible route. If there is none, the status is `no_feasible_route` and no route is
   recommended.

Default prototype thresholds (all configurable in `RoutingConfig`):

| Factor | Moderate from | High from | Severe from |
|---|---|---|---|
| Wave height (m) | 2.0 | 3.5 | 5.0 |
| Wind (kn) | 17 | 28 | 41 |
| Gusts (kn) | 25 | 40 | 55 |

## Running the notebook

1. Upload `notebooks/weather_aware_ship_routing.ipynb` to Google Colab (File → Upload notebook).
2. Runtime → Run all. Allow Google Drive access when asked.
3. Everything is saved under `MyDrive/coastal_platform/ml/ship_routing/`; the ZIP package
   goes to `MyDrive/coastal_platform/exports/` and is downloaded automatically.

## Using the module from Python

```python
import sys; sys.path.insert(0, "ml/ship_routing")
from routing import load_network_from_csv, plan_route, to_json_safe, RoutingConfig
from weather_service import OpenMeteoProvider, WeatherConfig
from sample_network import DATA_NOTICE

G = load_network_from_csv("ml/ship_routing/data/waypoints.csv", "ml/ship_routing/data/edges.csv", DATA_NOTICE)
provider = OpenMeteoProvider(WeatherConfig(cache_file="ml/ship_routing/results/cache/weather_cache.json"))
result = to_json_safe(plan_route(G, {"source_port": "INNSA", "destination_port": "INMAA",
                                     "vessel_speed": 15, "vessel_type": "container_ship"},
                                 provider, RoutingConfig()))
print(result["status"], result["distance_km"], result["weather_risk"])
```

## Sample input and output

Input (`results/json/sample_request.json`):

```json
{
  "source_port": "INNSA",
  "destination_port": "INCOK",
  "vessel_speed": 15,
  "vessel_type": "container_ship",
  "vessel_specs": {
    "draft_m": 12.5,
    "max_wave_height_m": 6.0,
    "max_wind_speed_kn": 45
  },
  "departure_time": "2026-12-01T00:00:00Z"
}
```

Output excerpt (`results/json/sample_response_scenario2.json`, **synthetic** storm off Mangalore):

```json
{
  "status": "ok",
  "source_port": {
    "id": "INNSA",
    "name": "Mumbai (JNPT / Nhava Sheva)"
  },
  "destination_port": {
    "id": "INCOK",
    "name": "Kochi (Cochin)"
  },
  "recommended_route": {
    "path": [
      "INNSA",
      "WP_MUM_OFF",
      "WP_ARAB_N",
      "WP_ARAB_W",
      "WP_NINE_W",
      "WP_NINE_E",
      "WP_KOC_OFF",
      "INCOK"
    ],
    "distance_km": 1692.3,
    "distance_nautical_miles": 913.8,
    "weather_risk": "Low",
    "total_weather_penalty": 0.0,
    "high_risk_segments": 0,
    "severe_risk_segments": 0
  },
  "alternative_route": {
    "path": [
      "INNSA",
      "WP_MUM_OFF",
      "WP_GOA_OFF",
      "WP_MNG_OFF",
      "WP_KOC_OFF",
      "INCOK"
    ],
    "distance_km": 1171.8,
    "weather_risk": "High",
    "total_weather_penalty": 997.0,
    "high_risk_segments": 2
  },
  "distance_km": 1692.3,
  "distance_nautical_miles": 913.8,
  "estimated_travel_time": {
    "hours": 60.9,
    "text": "2 d 12 h 55 min",
    "eta_utc": "2026-12-03T12:55:02Z",
    "basis": "service speed 15.0 kn reduced by prototype weather speed-loss factors"
  },
  "weather_risk": "Low",
  "segment_weather_details": [
    {
      "from": "WP_MUM_OFF",
      "to": "WP_ARAB_N",
      "distance_km": 231.7,
      "risk_label": "Low",
      "wave_height_m": 1.03,
      "encounter": "head sea",
      "effective_wave_height_m": 1.13,
      "wind_speed_kn": 10.3,
      "weather_penalty": 0.0,
      "data_sources": [
        "synthetic:storm_mangalore"
      ],
      "weather_feasible": true
    },
    "... one entry per segment ..."
  ],
  "route_explanation": "The weather-aware route was selected because the shortest route has 2 high-risk and 0 severe-risk segment(s). It is 520 km (44.4%) longer, but its overall risk is Low and its weather penalty is 0.0 versus 997.0 (total cost 1692.3 vs 2168.8).",
  "warnings": [
    "Weather data is SYNTHETIC test data, not a real forecast."
  ]
}
```

Response statuses: `ok`, `no_feasible_route` (connected but every route breaks a constraint),
`no_route` (not connected for this vessel), `error` (invalid request).

## FastAPI integration (later)

`fastapi_app_example.py` shows the pattern:

1. `pip install fastapi uvicorn`
2. Load the graph and the weather provider **once** at start-up.
3. Validate the request with a Pydantic model (`source_port`, `destination_port`,
   `vessel_speed`, `vessel_type`, `vessel_specs`, `departure_time`).
4. Call `plan_route(...)`, pass the result through `to_json_safe(...)` and return it.
5. Run: `uvicorn fastapi_app_example:app --port 8001` (from this folder), then the Node/Java/
   other backend calls `POST /route`.

Notes for production: the in-memory weather cache is per process; with several workers use a
shared cache (e.g. Redis). Keep calls to Open-Meteo within its published limits, and get a
commercial plan if the platform is used commercially.

## Replacing the synthetic network

Provide `data/waypoints.csv` (`node_id, name, type, lat, lon`) and `data/edges.csv`
(`u, v, lane_type, min_depth_m, navigable`, optional `bidirectional`) from a real source, for
example a lane network derived from AIS traffic density or a published shipping-lane dataset,
and load them with `load_network_from_csv()`. Validate depths against official charts.

## Limitations

* Synthetic network with hand-placed waypoints and illustrative depths.
* Not time-dependent: the worst forecast within the voyage window is used for every point.
* Direction factors, thresholds, speed-loss factors and vessel presets are prototype heuristics.
* Open-Meteo states its marine data has limited accuracy near coasts and is not suitable
  for coastal navigation.

In [ ]:
# Import the SAVED files and confirm they reproduce Scenario 2 exactly
import importlib
for mod in ("weather_service", "sample_network", "routing", "visualization"):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
import weather_service as ws_mod, sample_network as sn_mod, routing as routing_mod
importlib.reload(ws_mod); importlib.reload(sn_mod); importlib.reload(routing_mod)

wp_m, ed_m = sn_mod.create_sample_network()
G_mod = routing_mod.load_network_from_dataframes(wp_m, ed_m, sn_mod.DATA_NOTICE)
prov_mod = ws_mod.SyntheticWeatherProvider("storm_mangalore", storms=[ws_mod.StormZone(
    "Synthetic storm off Mangalore", 12.9, 74.6, 180, peak_wave_m=4.2, peak_wind_kn=32)])
res_mod = routing_mod.plan_route(G_mod, req("INNSA", "INCOK"), prov_mod, routing_mod.RoutingConfig())
res_nb = scenario_results["S2_bad_weather_on_shortest"]
same = (res_mod["recommended_route"]["path"] == res_nb["recommended_route"]["path"]
        and res_mod["recommended_route"]["total_cost"] == res_nb["recommended_route"]["total_cost"])
print("Saved modules reproduce the notebook result:", same)
assert same

## 19. Save results, visualisations and configuration to Google Drive
Most results were already saved as they were produced. This section saves the configuration files and the comparison tables, and tries to save a copy of **this notebook** into `notebooks/`.

> Saving the running notebook uses an internal Colab helper. If it does not work, use **File → Save a copy in Drive**, then move the copy to `coastal_platform/ml/ship_routing/notebooks/` and name it `weather_aware_ship_routing.ipynb`.

In [ ]:
# Configuration files
with open(os.path.join(DIRS["data"], "routing_config.json"), "w") as fh:
    json.dump(ROUTING_CONFIG.to_dict(), fh, indent=2)
with open(os.path.join(DIRS["data"], "weather_config.json"), "w") as fh:
    json.dump(WEATHER_CONFIG.to_dict(), fh, indent=2)
with open(os.path.join(DIRS["data"], "vessel_presets.json"), "w") as fh:
    json.dump({k: asdict(v) for k, v in VESSEL_PRESETS.items()}, fh, indent=2)

# Comparison tables
if live_result["status"] != "error":
    comparison_table(live_result["alternative_route"], live_result["recommended_route"]).to_csv(
        os.path.join(DIRS["tables"], "live_route_comparison.csv"))
    pd.DataFrame(live_result["segment_weather_details"]).to_csv(
        os.path.join(DIRS["tables"], "live_segment_weather.csv"), index=False)
for sc_id, res in scenario_results.items():
    comparison_table(res["alternative_route"], res["recommended_route"]).to_csv(
        os.path.join(DIRS["tables"], f"{sc_id}_comparison.csv"))
if hasattr(live_provider, "save_cache"):
    live_provider.save_cache()

# Copy of this notebook (Colab only)
NOTEBOOK_PATH = os.path.join(DIRS["notebooks"], "weather_aware_ship_routing.ipynb")
if IN_COLAB:
    try:
        from google.colab import _message
        nb_json = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
        with open(NOTEBOOK_PATH, "w", encoding="utf-8") as fh:
            json.dump(nb_json, fh)
        print("Notebook copy saved to", NOTEBOOK_PATH)
    except Exception as exc:
        print("Could not save the notebook automatically:", exc)
        print("Use File -> Save a copy in Drive and move it to", DIRS["notebooks"])
else:
    print("Not in Colab - save the notebook manually to", DIRS["notebooks"])
print("Configuration and tables saved.")

## 20. Verify saved files
Checks that every important file exists on Google Drive and is not empty.

In [ ]:
IMPORTANT_FILES = [
    "routing.py", "weather_service.py", "visualization.py", "sample_network.py",
    "requirements.txt", "README.md", "fastapi_app_example.py",
    "data/waypoints.csv", "data/edges.csv", "data/routing_config.json",
    "data/weather_config.json", "data/vessel_presets.json",
    "results/json/sample_request.json", "results/json/sample_response_scenario2.json",
    "results/json/live_response.json", "results/tables/scenario_summary.csv",
    "results/tables/robustness_tests.csv", "results/figures/S2_bad_weather_on_shortest_map.png",
    "results/figures/live_route_map.png",
]
OPTIONAL_FILES = ["notebooks/weather_aware_ship_routing.ipynb", "results/maps/live_route_map.html",
                  "results/maps/S2_bad_weather_on_shortest_map.html"]

def check(paths, required):
    rows = []
    for rel in paths:
        full = os.path.join(PROJECT_DIR, rel)
        size = os.path.getsize(full) if os.path.exists(full) else 0
        rows.append({"file": rel, "required": required, "exists": os.path.exists(full),
                     "size_kb": round(size / 1024, 1), "OK": size > 0})
    return rows

verify_df = pd.DataFrame(check(IMPORTANT_FILES, True) + check(OPTIONAL_FILES, False))
display(verify_df)
missing_required = verify_df[(verify_df.required) & (~verify_df.OK)]
if len(missing_required):
    print("MISSING REQUIRED FILES:\n", missing_required.file.tolist())
else:
    print(f"All {len(IMPORTANT_FILES)} required files are saved in", PROJECT_DIR)
missing_optional = verify_df[(~verify_df.required) & (~verify_df.OK)].file.tolist()
if missing_optional:
    print("Optional files not present (see notes above):", missing_optional)

## 21. Package and download the complete routing module for GitHub
Creates `ship_routing_<timestamp>.zip` containing the whole `ship_routing/` folder (the weather cache and `__pycache__` are left out), saves it in `coastal_platform/exports/` on Google Drive, checks its contents, and downloads it to your computer.

No passwords or tokens are used anywhere in this notebook.

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
ZIP_PATH = os.path.join(DIRS["exports"], f"ship_routing_{stamp}.zip")
EXCLUDE_DIRS = {"__pycache__", ".ipynb_checkpoints", "cache"}

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(PROJECT_DIR):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
        for f in files:
            full = os.path.join(root, f)
            arc = os.path.join("ship_routing", os.path.relpath(full, PROJECT_DIR))
            zf.write(full, arc)

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = set(zf.namelist())
    bad = zf.testzip()
required_in_zip = [os.path.join("ship_routing", f) for f in IMPORTANT_FILES]
missing_in_zip = [f for f in required_in_zip if f not in names]
print(f"ZIP saved to Drive: {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1024:.0f} KB, {len(names)} files)")
print("Corrupt entries:", bad)
print("Required files missing from ZIP:", missing_in_zip or "none")
assert bad is None and not missing_in_zip
print("Top of ZIP:", sorted(n for n in names if n.count("/") == 1))

if IN_COLAB:
    from google.colab import files
    files.download(ZIP_PATH)

### Push the module to GitHub (`ml-work` branch) with Git Bash
1. Unzip the downloaded file. You get a folder called `ship_routing/`.
2. In Git Bash:
```bash
cd /path/to/Supply-Chain-And-Coastal-Chain-Intelligence
git checkout ml-work
git pull origin ml-work                 # get the latest version first
ls ml/                                  # you should see port_congestion/ (do not touch it)
```
3. Copy the unzipped `ship_routing/` folder into `ml/` so you have `ml/ship_routing/routing.py` etc. It is a **new folder**, so nothing in `ml/port_congestion/` is overwritten. Check with:
```bash
git status                               # should list only ml/ship_routing/... as new files
```
4. (Optional) skip the weather cache in future commits:
```bash
echo "ml/ship_routing/results/cache/" >> .gitignore
```
5. Commit and push:
```bash
git add ml/ship_routing .gitignore
git commit -m "Add weather-aware ship routing module"
git push origin ml-work
```
When Git asks for credentials, use your normal login / credential manager. Never paste a token into the notebook.

### Saving the notebook itself to GitHub
* **Option A (recommended):** the ZIP already contains `notebooks/weather_aware_ship_routing.ipynb` if Section 19 saved it. Otherwise, in Colab use **File → Download → Download .ipynb** and put it in `ml/ship_routing/notebooks/` before `git add`.
* **Option B:** in Colab, **File → Save a copy in GitHub**. Authorise Colab in the browser pop-up (no token in the notebook), choose the repository, branch `ml-work`, and path `ml/ship_routing/notebooks/weather_aware_ship_routing.ipynb`. Afterwards run `git pull origin ml-work` locally.